# Sionna RT — OSM Scene Builder
## Physics-based radio scene construction from OpenStreetMap + EA LiDAR

This notebook builds a complete 3-D radio scene for Sionna 2.0 and Sionna 0.19 ray tracing from:
- **OpenStreetMap** — buildings, roads, water, vegetation, trees, railways, barriers
- **EA LiDAR** — 1 m resolution DTM (bare earth) + DSM (surface with clutter)
- **nDSM = DSM − DTM** — measured height of every building, tree and structure

---

## Run Order — Scene Build + Differentiable RT Test

### PHASE 1 — Build Scene (this notebook)

Run once. Re-run CELL 4 onwards only if you add new OSM features or change geometry.

| Step | Cell | Purpose | Skip when |
|------|------|---------|-----------|
| 1 | **CELL 0** | Edit config — bbox, frequency, feature knobs | Never |
| 2 | **CELL 1** | Imports | Once per session |
| 3 | **CELL 2b** | Download EA LiDAR DTM tiles | `dtm.tif` already exists |
| 4 | **CELL 2e** | Merge LiDAR tiles → single GeoTIFF | Merged files valid |
| 5 | **CELL 2d** | Compute nDSM = DSM − DTM → `ndsm.tif` | `ndsm.tif` exists |
| 6 | **CELL 3** | Build terrain PLY from DTM | `FLAT_TERRAIN=True` |
| 7 | **CELL 4** | Download OSM → PLY files (buildings, roads, water, vegetation, trees, railways, barriers) | Never |
| 8 | **CELL B3** | Assemble PLYs → `scene_with_full.xml` (Sionna 2 format) | Never |
| 9 | **CELL B1** | Convert `scene_with_full.xml` → `scene_with_full_019.xml` (Sionna 0.19 format) | Never |

**CELL 4 outputs** (scene/meshes_full/):
```
bld_itu_brick.ply          buildings — residential/churches
bld_itu_concrete.ply       buildings — industrial/civic
bld_itu_glass.ply          buildings — offices/retail
road_itu_asphalt.ply       all roads
water_itu_water.ply        rivers, lakes, canals
veg_itu_vegetation.ply     forests, parks, grass patches
trees_itu_vegetation.ply   individual street trees  ← new
rail_itu_metal.ply         railway/tram tracks      ← new
barriers_itu_concrete.ply  walls, noise barriers    ← new
barriers_itu_metal.ply     fences, guard rails      ← new
bld_itu_concrete_bridges.ply   bridges
bld_itu_concrete_embankments.ply  railway embankments
terrain.ply                bare-earth terrain mesh
```

---

### PHASE 2 — Calibrate (`sionna019_differentiable_rt_fixed.ipynb`)

Run after Phase 1. Scene files must exist before starting.

| Step | Cell | Purpose | Output file |
|------|------|---------|-------------|
| 1 | CELL 0–3 | Config + imports + utilities | — |
| 2 | CELL 4 | Load scene | — |
| 3 | CELL 4A | Assign ITU-R materials (auto-loads calibrated JSON if present) | — |
| 4 | CELL 6–7 | Load TX + 1200 RX | — |
| 5 | CELL 8b | Build calibration targets from Ofcom CSV | `calib_rssi_meas` |
| 6 | **CELL 10b** | Scalar offset calibration (~5 min) | `scalar_offset_915mhz.json` |
| 7 | **CELL 11b** | Material calibration (~2–3 h, 300 steps) | `calibrated_materials_915mhz.json` |

---

### PHASE 3 — Validate (`sionna2_915mhz_dem_simulation.ipynb`)

Run after Phase 2. JSON files auto-loaded by Cell 4A.

| Step | Cell | Purpose |
|------|------|---------|
| 1 | CELL 1 | Config |
| 2 | CELL 3 | Load scene (same PLYs as 0.19) |
| 3 | **CELL 4A** | Load materials + scalar offset from JSON files |
| 4 | CELL 5 | Parse Ofcom CSV → 1200 receivers |
| 5 | CELL 7 | Place TX + RX |
| 6 | **CELL 8** | Solve path loss — SCALAR_OFFSET_DB applied automatically |

---

### Quick Test (no 3-hour wait)

To verify the scene loads and scalar offset works without running Cell 11b:

```
Scene builder : CELL 0 → CELL 1 → CELL 4 → CELL B3 → CELL B1
Diff RT       : CELL 0 → CELL 1 → CELL 2 → CELL 3 → CELL 4 → CELL 4A
                → CELL 6 → CELL 7 → CELL 8b → CELL 10b  (5 min)
Sionna 2 DEM  : CELL 1 → CELL 3 → CELL 4A  ← check: "Scalar offset loaded"
                → CELL 8 first band only (0–300 m)
```

---

## Key Design Decisions

### Building heights — nDSM first
`USE_NDSM_HEIGHTS=True` samples LiDAR nDSM at each building centroid.
Falls back to OSM `height=` tag → `building:levels × 3.5 m` → `DEFAULT_HEIGHT_M`.

### Vegetation — flat patches (recommended)
`VEG_3D_GEOMETRY=False` writes flat ground patches — no hard canopy shadow.
At 915 MHz extruded canopy over-attenuates by 10–15 dB. ITU-R P.833
Weissberger post-processing in the DEM notebook adds correct excess attenuation.

### New features — frequency-agnostic geometry
Railways, barriers, individual trees and building age are geometry-only additions.
Their EM impact scales automatically with frequency via the assigned material
(itu_metal σ=1e7, itu_concrete ε_r=5.31) — no re-run needed when FREQUENCY_HZ changes.

### LiDAR provider
`NDSM_PROVIDER`: `'ea'` (England, free 1 m), `'usgs'` (USA, free 1 m), `'opentopo'` (global 30 m).

---

## Section A — OSM Scene Builder
Run cells 0–B3 top-to-bottom for a new scene.

## Section B — Blender Scene Conversion
Standalone converters (CELL B1/B2/B3). Do not depend on Section A.


## CELL 0 — Configuration

**Edit this cell for every new campaign.** All other cells read from these variables.

### Knob reference

| Knob | Default | Effect |
|------|---------|--------|
| `SCENARIO_NAME` | `'nottingham_...'` | Folder name under `~/sionna_rt/` |
| `SCENE_WEST/EAST/SOUTH/NORTH` | Nottingham bbox | Scene extent in WGS84 |
| `UTM_EPSG` | `32630` | UTM zone (30N = UK; change per country) |
| `FLAT_TERRAIN` | `False` | `True` = flat z=0 plane (skip LiDAR cells) |
| `TERRAIN_GRID_N` | `500` | Terrain PLY resolution (500×500 recommended) |
| `USE_NDSM_HEIGHTS` | `True` | Use LiDAR nDSM for building heights |
| `NDSM_BUILDING_MIN_M` | `2.0` | Ignore nDSM < this (noise / ground return) |
| `NDSM_PROVIDER` | `'ea'` | LiDAR source: `'ea'` / `'usgs'` / `'opentopo'` |
| `LIDAR_DIR` | `/home/.../lidar` | Folder with pre-downloaded LiDAR tiles |
| `VEG_3D_GEOMETRY` | `False` | `True` = extrude canopy (legacy, over-blocks at <1 GHz) |
| `VEG_MIN_AREA_M2` | `50.0` | Skip vegetation polygons smaller than this |
| `MIN_BUILDING_AREA_M2` | `30.0` | Skip building footprints smaller than this |
| `DEFAULT_HEIGHT_M` | `8.0` | Fallback building height when no tag or nDSM |
| `CITY_MAX_HEIGHT_M` | `40.0` | Cap building height (avoids LiDAR artefacts) |
| `ROOF_PITCH_DEG` | `30.0` | Pitch angle for untagged pitched roofs |
| `FREQUENCY_HZ` | `915.95 MHz` | Used for documentation / cross-reference |

> **To port to a new city:** change `SCENARIO_NAME`, the four bbox values,
> `UTM_EPSG`, and `LIDAR_DIR`. Nothing else needs editing.


In [ ]:
# ============================================================
# BUILD SEQUENCE — SIONNA 2.0 DEM SCENE (with roads)
# ============================================================
# Step 1 : Run CELL 0   — project config
# Step 2 : Run CELL 1   — imports
# Step 3 : Run CELL 2   — terrain PLY  (skip if terrain.ply exists)
# Step 4 : Run CELL 2b  — nDSM heights (skip if already done)
# Step 5 : Run CELL 4   — OSM buildings + ALL roads → PLYs
# Step 6 : Run CELL B3  — write scene_with_roads.xml  (Sionna 2.0)
# ── Scene is now ready for sionna2_915mhz_dem_simulation.ipynb ──
#
# TO ALSO PREPARE SIONNA 0.19 (differentiable RT):
# Step 7 : Run CELL B1  — convert scene_with_roads.xml
#                          → scene_with_roads_019.xml
# ── Scene is now ready for sionna019_differentiable_rt_fixed.ipynb ──
# ============================================================

# ============================================================
# CELL 0 — CONFIG  (edit this block only)
# ============================================================
import os

# ╔══════════════════════════════════════════════════════════╗
# ║                  SCENARIO CONFIGURATION                  ║
# ║           Edit this block for each new campaign          ║
# ╚══════════════════════════════════════════════════════════╝

SCENARIO_NAME   = 'nottingham_ofcom2018_v3_enhanced'

# ── Scene bbox (WGS84) ──────────────────────────────────
SCENE_WEST   = -1.267685
SCENE_EAST   = -1.119832
SCENE_SOUTH  =  52.943165
SCENE_NORTH  =  53.003037

# ── Coordinate system ──────────────────────────────
UTM_EPSG  = 32630   # UTM zone 30N (UK)

# ── Terrain ────────────────────────────────────────
FLAT_TERRAIN   = False
TERRAIN_SOURCE = 'ea_lidar'   # 'ea_lidar' (England) | 'aws_dem' (global)
TERRAIN_GRID_N = 500
TERRAIN_PAD_M  = 3000          # 3km beyond furthest receiver (9342m)
TILE_ZOOM      = 14

# ── LiDAR clutter heights (nDSM = DSM - DTM) ──────────────
# USE_NDSM_HEIGHTS = True  -> building heights from LiDAR nDSM (portable,
#                             fills OSM gaps; requires CELL 2d to run first)
# USE_NDSM_HEIGHTS = False -> use OSM height= tags only (original behaviour)
USE_NDSM_HEIGHTS    = True
NDSM_BUILDING_MIN_M = 2.0    # ignore nDSM below this (noise / ground return)
NDSM_PROVIDER       = 'ea'   # 'ea' (England) | 'usgs' (USA) | 'opentopo' (global)

# ── Building parameters ───────────────────────────────
MIN_BUILDING_AREA_M2  =  0.0
CITY_MIN_HEIGHT_M     =  0.0
CITY_MAX_HEIGHT_M     = 40.0
HEIGHT_PER_LEVEL_M    =  3.5
DEFAULT_HEIGHT_M      =  8.0

# ── Roof shape ───────────────────────────────────────
ROOF_PITCH_DEG        = 30.0
ROOF_MAX_RIDGE_M      =  6.0

# ── OSM extra features ────────────────────────────────
INCLUDE_ROADS       = True
INCLUDE_WATER       = True    # River Trent + lakes → itu_water PLY (specular reflection)
INCLUDE_VEGETATION  = True    # OSM forests/parks/scrub → itu_vegetation PLY + P.833 post-processing
INCLUDE_BRIDGES     = True    # road/rail bridges → itu_concrete PLY (hard diffraction edges)
INCLUDE_EMBANKMENTS      = True    # railway embankments → itu_concrete PLY
INCLUDE_ROAD_EMBANKMENTS = True    # highway embankments (embankment=yes) → itu_concrete PLY
INCLUDE_ROAD_CUTTINGS    = True    # highway cuttings (cutting=yes) → itu_concrete PLY
INCLUDE_HWY_BRIDGES      = True    # highway bridge decks (bridge=yes) → itu_concrete PLY
VEG_3D_GEOMETRY     = False  # False -> flat ground patches + ITU-R P.833 post-hoc (recommended) patches only (material + scattering)
VEG_MIN_AREA_M2     = 50.0
INCLUDE_TREES       = True    # individual OSM trees (natural=tree) → itu_vegetation disks at canopy height (ITU-R P.833-10)
TREE_DEFAULT_HT_M   = 8.0    # canopy height when OSM height= tag and nDSM both absent
TREE_DISK_RADIUS_M  = 3.0    # disk radius for individual tree canopy representation
USE_BUILDING_AGE    = True    # OSM start_date= tag → pre-1940=brick, 1940-1980=concrete, post-1980=type-based (Jansen et al. 2019)
INCLUDE_RAILWAYS    = True    # railway=rail/tram tracks → itu_metal PLY (steel rail specular reflector)
INCLUDE_BARRIERS    = True    # barrier=wall/fence/noise_barrier → itu_concrete/itu_metal PLY
BARRIER_MIN_LEN_M   = 10.0   # skip barrier segments shorter than this (removes noise)
INCLUDE_PYLONS       = True   # power=tower/pole → itu_metal PLY (metal diffraction edges, [DeE04])
INCLUDE_MASTS        = True   # man_made=mast/communications_tower → itu_metal PLY
INCLUDE_CHIMNEYS     = True   # man_made=chimney → itu_concrete PLY (industrial stacks)
INCLUDE_WATER_TOWERS = True   # man_made=water_tower → itu_metal PLY
INCLUDE_STORAGE_TANKS= True   # man_made=storage_tank → itu_metal PLY (curved backscatterers [DeE04])
INCLUDE_STADIUMS     = True   # leisure=stadium → itu_metal PLY (large roof structures [Xia24])
INCLUDE_SUBSTATIONS  = True   # power=substation → itu_metal PLY (fenced transformer enclosures)
INCLUDE_CAR_PARKS    = True   # building=parking / parking=multi-storey → itu_concrete PLY
INCLUDE_COOLING_TOWERS = True # man_made=cooling_tower → itu_concrete PLY (large cylindrical reflectors)
EXCLUDE_TUNNELS       = True    # skip road/rail ways tagged tunnel=yes (underground — not surface obstacles)
INCLUDE_FUEL_CANOPIES = True    # amenity=fuel → itu_metal flat canopy roof at +5m (strong specular reflector)
INCLUDE_BUS_STATIONS  = True    # amenity=bus_station → itu_metal extruded 6m (large covered urban structure)
INCLUDE_SURFACE_PARKS = True    # amenity=parking (surface) → itu_asphalt flat patch (ground reflection at 3.6 GHz)
INCLUDE_GREENHOUSES   = True    # building=greenhouse/glasshouse → itu_glass extruded (RF-transparent vs brick)

# ── V3 / V4 / V5 high-frequency feature flags ──────────────────────────
# V3 (2.8-6 GHz) : INCLUDE_ROOFTOP_EQUIPMENT
# V4 (6-28 GHz)  : V3 + INCLUDE_PARKED_VEHICLES + INCLUDE_STREET_LAMPS
# V5 (28-60 GHz) : V4 + INCLUDE_BODY_PHANTOMS + INCLUDE_SHELTERS
INCLUDE_ROOFTOP_EQUIPMENT = False  # HVAC/plant-room +2.5m on commercial/industrial roofs
                                    # ITU-R P.2040-2: itu_metal (er=1, sigma=1e7)
ROOFTOP_EQUIP_HEIGHT_M    = 2.5    # height of HVAC box above roof level
ROOFTOP_EQUIP_FRACTION    = 0.25   # fraction of roof area covered (footprint scale)
ROOFTOP_BLDG_TYPES        = {'commercial','retail','industrial','office','supermarket',
                              'warehouse','factory','hospital','university','school'}

INCLUDE_PARKED_VEHICLES   = False  # box primitives along OSM parking lanes
                                    # ITU-R P.2040-2: itu_metal (steel body proxy)
VEHICLE_LENGTH_M          = 4.5
VEHICLE_WIDTH_M           = 1.8
VEHICLE_HEIGHT_M          = 1.4
VEHICLE_SPACING_M         = 6.0
VEHICLE_OFFSET_M          = 2.5

INCLUDE_STREET_LAMPS      = False  # cylinder lamp posts on highway edges
                                    # ITU-R P.2040-2: itu_metal (steel pole)
LAMP_HEIGHT_M             = 6.0
LAMP_RADIUS_M             = 0.05
LAMP_SPACING_M            = 35.0
LAMP_ROAD_OFFSET_M        = 2.5
LAMP_ROAD_TYPES           = {'primary','secondary','tertiary','residential','unclassified'}

INCLUDE_BODY_PHANTOMS     = False  # cylinder human-body proxy at each RX position
                                    # ITU-R P.1238 / P.2040-2: itu_concrete proxy
                                    # 15-20 dB blockage at 60 GHz (V5 only)
PHANTOM_HEIGHT_M          = 1.7
PHANTOM_RADIUS_M          = 0.2

INCLUDE_SHELTERS          = False  # bus shelters / kiosks
                                    # ITU-R P.2040-2: itu_metal roof + itu_glass sides
SHELTER_HEIGHT_M          = 2.5
SHELTER_DEPTH_M           = 1.5
SHELTER_WIDTH_M           = 3.0

# ── Exclude trivial building types ──────────────────────
EXCLUDE_BUILDING_TYPES = set()

# ── TX / RX parameters ──────────────────────────────
FREQUENCY_HZ         = 915.95e6
TX_AGL_M             = 17.0
RX_AGL_M             =  1.5
TX_CONDUCTED_DBM     = 49.0
TX_ANTENNA_GAIN_DBI  =  1.3
RX_EXTRA_GAIN_DB     = -7.8
SITE_CORRECTION_DB   =  0.0
ANTENNA_PATTERN      = 'donut'

# ╔══════════════════════════════════════════════════════════╗
# ║                    PATH CONFIGURATION                    ║
# ╚══════════════════════════════════════════════════════════╝

_ROOT       = os.environ.get('RT_BASE_DIR', os.path.join(os.path.expanduser('~'), 'sionna_rt', 'nottingham_ofcom2018_915mhz_dem'))
BASE_DIR    = _ROOT
# ── Scene version folders ─────────────────────────────────
# scene_v2_infra/    : baseline (buildings + roads + infra)  — 915 MHz
# scene_v3_enhanced/ : V3 (+ rooftop equipment, nDSM)        — 2.8-6 GHz
# scene_v4_enhanced/ : V4 (V3 + parked vehicles + lamps)     — 6-28 GHz
# scene_v5_enhanced/ : V5 (V4 + body phantoms + shelters)    — 28-60 GHz
SCENE_DIR   = os.path.join(_ROOT, 'scene_v3_enhanced')
MESH_DIR    = os.path.join(_ROOT, 'scene_v3_enhanced', 'meshes')
MERGED_DIR  = os.path.join(_ROOT, 'scene', 'meshes_roads')
# ── LiDAR file paths ─────────────────────────────────────
# Not used for flat terrain (FLAT_TERRAIN   = False)
LIDAR_DIR   = _ROOT
EA_DTM_TIFF = os.path.join(LIDAR_DIR, 'dem.tif')
EA_DSM_TIFF = os.path.join(LIDAR_DIR, 'lidar_dsm.tif')
NDSM_TIFF   = os.path.join(_ROOT, 'ndsm.tif')

os.makedirs(MERGED_DIR, exist_ok=True)
os.makedirs(MESH_DIR,   exist_ok=True)

print('Config loaded.')
print(f'  Scenario    : {SCENARIO_NAME}')
print(f'  Scene bbox  : lon [{SCENE_WEST}, {SCENE_EAST}]')
print(f'                lat [{SCENE_SOUTH}, {SCENE_NORTH}]')
print(f'  Output      : {SCENE_DIR}')
print(f'  nDSM heights: {USE_NDSM_HEIGHTS}  (provider={NDSM_PROVIDER})')
print(f'  Veg 3D geom : {VEG_3D_GEOMETRY}')
if not FLAT_TERRAIN:
    print(f'  Terrain src : {TERRAIN_SOURCE}  ({TERRAIN_GRID_N}x{TERRAIN_GRID_N} grid)')


In [ ]:
# ============================================================
# CELL 1 — IMPORTS & DEPENDENCIES
# ============================================================
import os, math, time, json, struct, warnings
import numpy as np
import requests
from io import BytesIO
from concurrent.futures import ThreadPoolExecutor, as_completed
from pyproj import Transformer
import shapely.geometry as sg
import shapely.ops as so
from shapely.geometry import Polygon, MultiPolygon, box

# Optional: rasterio for GeoTIFF reading
try:
    import rasterio
    from rasterio.transform import from_bounds
    _HAS_RASTERIO = True
except ImportError:
    _HAS_RASTERIO = False
    print('⚠  rasterio not found — falling back to PIL for GeoTIFF tiles')

# PIL for fallback
try:
    from PIL import Image
    _HAS_PIL = True
except ImportError:
    _HAS_PIL = False

# osmnx for OSM data
try:
    import osmnx as ox
    ox.settings.use_cache = True
    ox.settings.log_console = False
    _HAS_OSMNX = True
except ImportError:
    _HAS_OSMNX = False
    print('⚠  osmnx not found — install with: pip install osmnx')

# trimesh for PLY export
try:
    import trimesh
    _HAS_TRIMESH = True
except ImportError:
    _HAS_TRIMESH = False
    print('⚠  trimesh not found — install with: pip install trimesh')

# Coordinate transformers
to_utm   = Transformer.from_crs('EPSG:4326', f'EPSG:{UTM_EPSG}', always_xy=True)
to_wgs84 = Transformer.from_crs(f'EPSG:{UTM_EPSG}', 'EPSG:4326', always_xy=True)

# Scene bbox in UTM
sw_utm = to_utm.transform(SCENE_WEST,  SCENE_SOUTH)
ne_utm = to_utm.transform(SCENE_EAST,  SCENE_NORTH)
center_utm = ((sw_utm[0]+ne_utm[0])/2, (sw_utm[1]+ne_utm[1])/2)
center_lon, center_lat = to_wgs84.transform(*center_utm)

print(f'UTM SW : {sw_utm[0]:.1f}, {sw_utm[1]:.1f}')
print(f'UTM NE : {ne_utm[0]:.1f}, {ne_utm[1]:.1f}')
print(f'Center : ({center_lon:.5f}, {center_lat:.5f})')
print(f'Size   : {(ne_utm[0]-sw_utm[0])/1000:.2f} km × {(ne_utm[1]-sw_utm[1])/1000:.2f} km')

In [ ]:
# ============================================================
# CELL 2 — AWS ELEVATION TILES → HEIGHTMAP
# ============================================================
# Only needed when TERRAIN_SOURCE = 'aws_dem'
# Skip this cell if TERRAIN_SOURCE = 'ea_lidar' (use CELL 2b–2e instead)
# ============================================================
if globals().get('TERRAIN_SOURCE', 'aws_dem') != 'aws_dem':
    print(f"TERRAIN_SOURCE='{TERRAIN_SOURCE}' — skipping AWS terrain download.")
    print("For EA LiDAR terrain run CELL 2b → 2c → 2d → 2e → CELL 3.")
else:
 pass  # guard end — rest of cell runs only for aws_dem
# ============================================================
# Downloads GeoTIFF tiles from the public AWS elevation-tiles-prod bucket
# (same source used by Mapzen/Terrarium, Cesium, sionna-large-radio-maps).
# No credentials needed — anonymous public access.
# URL: s3://elevation-tiles-prod/geotiff/{z}/{x}/{y}.tif
# Values are direct metres ASL (float32 GeoTIFF, no conversion formula).

AWS_BASE = 'https://s3.amazonaws.com/elevation-tiles-prod/geotiff'

def _lon2tile(lon, z):
    return int(math.floor((lon + 180) / 360 * 2**z))

def _lat2tile(lat, z):
    lat_r = math.radians(lat)
    return int(math.floor((1 - math.log(math.tan(lat_r) + 1/math.cos(lat_r)) / math.pi) / 2 * 2**z))

def _tile2lon(x, z):
    return x / 2**z * 360 - 180

def _tile2lat(y, z):
    n = math.pi - 2 * math.pi * y / 2**z
    return math.degrees(math.atan(math.sinh(n)))

def _download_tile(z, x, y, retries=3):
    url = f'{AWS_BASE}/{z}/{x}/{y}.tif'
    for attempt in range(retries):
        try:
            r = requests.get(url, timeout=30)
            r.raise_for_status()
            buf = BytesIO(r.content)
            if _HAS_RASTERIO:
                with rasterio.open(buf) as ds:
                    data = ds.read(1).astype(np.float32)
            elif _HAS_PIL:
                # PIL may not read float GeoTIFF correctly — warn
                img = Image.open(buf)
                data = np.array(img, dtype=np.float32)
            else:
                raise RuntimeError('Need rasterio or PIL to read GeoTIFF tiles')
            # Resample to 512×512 if needed
            if data.shape != (512, 512):
                from scipy.ndimage import zoom as nd_zoom
                fx = 512 / data.shape[0]; fy = 512 / data.shape[1]
                data = nd_zoom(data, (fx, fy), order=1).astype(np.float32)
            return data
        except Exception as e:
            if attempt == retries - 1:
                print(f'  ⚠ tile {z}/{x}/{y} failed: {e} — using zeros')
                return np.zeros((512, 512), dtype=np.float32)
            time.sleep(2 ** attempt)

# Compute tile range for scene bbox
x0 = _lon2tile(SCENE_WEST,  TILE_ZOOM)
x1 = _lon2tile(SCENE_EAST,  TILE_ZOOM)
y0 = _lat2tile(SCENE_NORTH, TILE_ZOOM)  # north = smaller y in tile coords
y1 = _lat2tile(SCENE_SOUTH, TILE_ZOOM)
n_cols = x1 - x0 + 1
n_rows = y1 - y0 + 1
print(f'Tile range : x=[{x0},{x1}] y=[{y0},{y1}]  →  {n_cols}×{n_rows} = {n_cols*n_rows} tiles')

# Download in parallel
tile_mosaic = np.zeros((n_rows * 512, n_cols * 512), dtype=np.float32)
jobs = [(TILE_ZOOM, x0+col, y0+row, col, row)
        for row in range(n_rows) for col in range(n_cols)]

print(f'Downloading {len(jobs)} tiles ...')
t0 = time.time()
with ThreadPoolExecutor(max_workers=8) as ex:
    futures = {ex.submit(_download_tile, z, x, y): (col, row)
               for z, x, y, col, row in jobs}
    for i, fut in enumerate(as_completed(futures)):
        col, row = futures[fut]
        tile_mosaic[row*512:(row+1)*512, col*512:(col+1)*512] = fut.result()
        if (i+1) % max(1, len(jobs)//4) == 0:
            print(f'  {i+1}/{len(jobs)} tiles done')

print(f'Download done in {time.time()-t0:.1f}s')

# Extent of the mosaic in WGS84
_map_min_lon = _tile2lon(x0,   TILE_ZOOM)
_map_max_lon = _tile2lon(x1+1, TILE_ZOOM)
_map_max_lat = _tile2lat(y0,   TILE_ZOOM)   # y0 is north
_map_min_lat = _tile2lat(y1+1, TILE_ZOOM)   # y1+1 is south
_mosaic_h, _mosaic_w = tile_mosaic.shape

print(f'Mosaic size : {_mosaic_w}×{_mosaic_h} px')
print(f'Mosaic lon  : [{_map_min_lon:.5f}, {_map_max_lon:.5f}]')
print(f'Mosaic lat  : [{_map_min_lat:.5f}, {_map_max_lat:.5f}]')
print(f'Elevation   : [{tile_mosaic.min():.1f}, {tile_mosaic.max():.1f}] m ASL')

def height_from_wgs84(lon, lat):
    """Bilinear interpolation from mosaic. Returns elevation in metres ASL."""
    u = (lon - _map_min_lon) / (_map_max_lon - _map_min_lon) * (_mosaic_w - 1)
    v = (1 - (lat - _map_min_lat) / (_map_max_lat - _map_min_lat)) * (_mosaic_h - 1)
    u = np.clip(u, 0, _mosaic_w - 1)
    v = np.clip(v, 0, _mosaic_h - 1)
    x0i, y0i = int(np.floor(u)), int(np.floor(v))
    x1i, y1i = min(x0i+1, _mosaic_w-1), min(y0i+1, _mosaic_h-1)
    fx, fy = u - x0i, v - y0i
    h = (tile_mosaic[y0i, x0i] * (1-fx) * (1-fy)
       + tile_mosaic[y0i, x1i] *    fx  * (1-fy)
       + tile_mosaic[y1i, x0i] * (1-fx) *    fy
       + tile_mosaic[y1i, x1i] *    fx  *    fy)
    return float(h)

def height_from_utm(easting, northing):
    """Height at UTM coordinates."""
    lon, lat = to_wgs84.transform(easting, northing)
    return height_from_wgs84(lon, lat)

# Scene centre elevation = local z=0 reference
origin_elev_asl = height_from_wgs84(center_lon, center_lat)
print(f'\nScene centre elevation : {origin_elev_asl:.2f} m ASL  (= local z=0)')

def local_z(lon, lat):
    """Returns local z in metres relative to scene centre elevation."""
    return height_from_wgs84(lon, lat) - origin_elev_asl

## CELL 2b — Download EA LiDAR DTM

Downloads the Environment Agency 1 m Composite DTM (bare-earth terrain) for the
scene bbox from the EA WCS service (free, no credentials, England only).

- Skips automatically if `FLAT_TERRAIN=True` or `dem.tif` already exists
- If you have pre-downloaded tiles, run **CELL 2e** to merge them first
- Coverage check: run **CELL 2c** after this cell


In [ ]:
# ============================================================
# CELL 2b — EA LiDAR DTM Auto-Download  (skip if FLAT_TERRAIN=True)
# ============================================================
# Downloads Environment Agency 1m Composite DTM for scene bbox.
# Source: EA WCS service (free, no credentials, England only).
# CRS: EPSG:27700 (British National Grid) for request, output GeoTIFF.
# Skip this cell entirely when FLAT_TERRAIN=True.
# ============================================================

if globals().get('FLAT_TERRAIN', True):
    print('FLAT_TERRAIN=True — skipping EA LiDAR download.')
    print('Set FLAT_TERRAIN=False in CELL 0 and re-run this cell for real terrain.')
else:
    import requests, os
    from pyproj import Transformer

    print('=' * 60)
    print('CELL 2b — EA LiDAR DTM Download (1m resolution)')
    print('=' * 60)

    if os.path.exists(EA_DTM_TIFF):
        print(f'Already downloaded: {EA_DTM_TIFF}')
        print('Delete the file and re-run to force re-download.')
    else:
        # Convert scene bbox WGS84 → BNG (EPSG:27700) for EA WCS request
        _wgs_to_bng = Transformer.from_crs('EPSG:4326', 'EPSG:27700', always_xy=True)
        _e_min, _n_min = _wgs_to_bng.transform(SCENE_WEST,  SCENE_SOUTH)
        _e_max, _n_max = _wgs_to_bng.transform(SCENE_EAST,  SCENE_NORTH)

        # Add 200m buffer so terrain edge doesn't clip buildings
        _buf = 200
        _e_min -= _buf; _n_min -= _buf
        _e_max += _buf; _n_max += _buf

        print(f'BNG bbox: E[{_e_min:.0f}, {_e_max:.0f}]  N[{_n_min:.0f}, {_n_max:.0f}]')
        print(f'Area    : {(_e_max-_e_min)/1000:.1f} km × {(_n_max-_n_min)/1000:.1f} km')

        # EA WCS endpoint — Composite DTM 1m
        _WCS_URL = (
            'https://environment.data.gov.uk/spatialdata/lidar-composite-dtm-1m/wcs'
            '?SERVICE=WCS&VERSION=2.0.1&REQUEST=GetCoverage'
            '&COVERAGEID=LIDAR_Composite_DTM_1m'
            f'&SUBSET=E,http://www.opengis.net/def/crs/EPSG/0/27700({_e_min:.0f},{_e_max:.0f})'
            f'&SUBSET=N,http://www.opengis.net/def/crs/EPSG/0/27700({_n_min:.0f},{_n_max:.0f})'
            '&FORMAT=image/tiff'
        )

        print(f'Downloading EA LiDAR DTM ...')
        print(f'URL: {_WCS_URL[:100]}...')

        try:
            _r = requests.get(_WCS_URL, timeout=120, stream=True)
            _r.raise_for_status()
            os.makedirs(os.path.dirname(EA_DTM_TIFF), exist_ok=True)
            _total = 0
            with open(EA_DTM_TIFF, 'wb') as _f:
                for _chunk in _r.iter_content(chunk_size=1024*1024):
                    _f.write(_chunk)
                    _total += len(_chunk)
                    print(f'  {_total//1024//1024} MB downloaded ...', end='\r')
            print(f'\nSaved: {EA_DTM_TIFF}  ({os.path.getsize(EA_DTM_TIFF)//1024//1024} MB)')

            # Quick verify with rasterio
            if _HAS_RASTERIO:
                import rasterio as _rio
                with _rio.open(EA_DTM_TIFF) as _ds:
                    print(f'CRS     : {_ds.crs}')
                    print(f'Shape   : {_ds.height} × {_ds.width} px')
                    print(f'Res     : {_ds.res[0]:.1f} m/px')
                    _data = _ds.read(1)
                    print(f'Z range : {float(_data.min()):.1f} – {float(_data.max()):.1f} m ASL')
            print('\n✓ EA LiDAR DTM ready. Now run CELL 3 to build terrain mesh.')

        except requests.exceptions.HTTPError as _e:
            print(f'HTTP error: {_e}')
            print('The EA WCS may be temporarily unavailable. Try again later.')
            print('Alternative: download manually from:')
            print('  https://environment.data.gov.uk/DefraDataDownload/?Mode=survey')
            print('  Select: LIDAR Composite DTM → 1m → your area → GeoTIFF')
            print(f'  Save to: {EA_DTM_TIFF}')
        except Exception as _e:
            print(f'Download failed: {_e}')
            print(f'Manual download URL:')
            print('  https://environment.data.gov.uk/DefraDataDownload/?Mode=survey')


## CELL 2c — DTM Coverage Check

Verifies the downloaded DTM fully encloses the scene bbox before expensive
mesh generation. Reports elevation range and NoData fraction.
Skips when `FLAT_TERRAIN=True`.


In [ ]:
# ============================================================
# CELL 2c — DTM COVERAGE CHECK  (verify dem.tif encloses the AOI)
# ============================================================
# Confirms the local EA/AWS GeoTIFF (EA_DTM_TIFF) actually covers the
# scene bbox BEFORE CELL 3 samples it into terrain.ply. Reprojects the
# raster bounds to WGS84 for an apples-to-apples comparison and reports
# elevation range + NoData fraction. Skip when FLAT_TERRAIN=True.
# ============================================================
if globals().get('FLAT_TERRAIN', True):
    print("FLAT_TERRAIN=True - no DTM needed, skipping coverage check.")
elif not os.path.exists(globals().get('EA_DTM_TIFF', '')):
    print(f"No DTM found at: {globals().get('EA_DTM_TIFF','')}")
    print("Copy your GeoTIFF there, e.g.:")
    print(f"  cp /path/to/your_terrain.tif {globals().get('EA_DTM_TIFF','dem.tif')}")
else:
    import rasterio
    from rasterio.warp import transform_bounds
    print("=" * 60)
    print("CELL 2c - DTM COVERAGE CHECK")
    print("=" * 60)
    with rasterio.open(EA_DTM_TIFF) as ds:
        print(f"File       : {EA_DTM_TIFF}")
        print(f"CRS        : {ds.crs}")
        print(f"Size       : {ds.width} x {ds.height} px   res {ds.res}")
        print(f"Bounds(raw): {tuple(round(b,1) for b in ds.bounds)}")
        w, s, e, n = transform_bounds(ds.crs, "EPSG:4326", *ds.bounds, densify_pts=21)
        print(f"Bounds WGS84: lon[{w:.6f}, {e:.6f}]  lat[{s:.6f}, {n:.6f}]")
        print(f"AOI    WGS84: lon[{SCENE_WEST:.6f}, {SCENE_EAST:.6f}]  "
              f"lat[{SCENE_SOUTH:.6f}, {SCENE_NORTH:.6f}]")
        _tol = 1e-4  # ~10m tolerance for floating point
        _covers = (w <= SCENE_WEST+_tol and e >= SCENE_EAST-_tol and
                   s <= SCENE_SOUTH+_tol and n >= SCENE_NORTH-_tol)
        if _covers:
            print("COVERS AOI : YES - DTM fully encloses the scene bbox.")
        else:
            _short = []
            if w > SCENE_WEST+_tol:  _short.append("West")
            if e < SCENE_EAST-_tol:  _short.append("East")
            if s > SCENE_SOUTH+_tol: _short.append("South")
            if n < SCENE_NORTH-_tol: _short.append("North")
            print(f"COVERS AOI : NO - DTM short on: {', '.join(_short)}")
            print("  -> use a larger tile or shrink the AOI bbox in CELL 0.")
        try:
            _band = ds.read(1, masked=True)
            print(f"Elev range : {float(_band.min()):.1f} .. {float(_band.max()):.1f} m ASL")
            _ndf = float(_band.mask.mean()) * 100 if _band.mask is not None and _band.mask.shape else 0.0
            print(f"NoData frac: {_ndf:.1f}%")
            if _ndf > 5.0:
                print("  ! >5% NoData over the tile - terrain may have holes in the AOI.")
        except Exception as _e:
            print(f"  (elevation/NoData stats unavailable: {_e})")


## CELL 2d — DSM Download + nDSM

Downloads EA LiDAR DSM, computes `nDSM = DSM − DTM` (height of above-ground objects). Used in CELL 4 to replace sparse OSM `height=` tags with measured building heights. Provider-abstracted via `NDSM_PROVIDER`.

**Run after CELL 2b.** Skip when `USE_NDSM_HEIGHTS=False`.


In [ ]:
# ============================================================
# CELL 2d — DSM DOWNLOAD + nDSM = DSM - DTM
# ============================================================
# Downloads LiDAR DSM alongside the DTM already fetched in
# CELL 2b, computes nDSM = DSM - DTM (height of above-ground
# objects), saves ndsm.tif.  Building heights in CELL 4 are
# sampled from nDSM instead of (sparse) OSM height= tags,
# filling clutter gaps automatically.
#
# Provider abstraction (NDSM_PROVIDER in CELL 0):
#   'ea'        -> Environment Agency WCS (England, free)
#   'usgs'      -> USGS 3DEP WCS (USA, free)
#   'opentopo'  -> OpenTopography REST API (global, API key needed)
# ============================================================

if not USE_NDSM_HEIGHTS:
    print('USE_NDSM_HEIGHTS=False — skipping DSM download.')
elif globals().get('FLAT_TERRAIN', True):
    print('FLAT_TERRAIN=True — skipping DSM download.')
else:
    import requests, os
    import numpy as np

    def _dsm_wcs_url(provider, e_min, e_max, n_min, n_max):
        if provider == 'ea':
            return (
                'https://environment.data.gov.uk/spatialdata/lidar-composite-dsm-1m/wcs'
                '?SERVICE=WCS&VERSION=2.0.1&REQUEST=GetCoverage'
                '&COVERAGEID=LIDAR_Composite_DSM_1m'
                f'&SUBSET=E,http://www.opengis.net/def/crs/EPSG/0/27700({e_min:.0f},{e_max:.0f})'
                f'&SUBSET=N,http://www.opengis.net/def/crs/EPSG/0/27700({n_min:.0f},{n_max:.0f})'
                '&FORMAT=image/tiff'
            )
        elif provider == 'usgs':
            raise NotImplementedError(
                'USGS 3DEP: use '
                'https://elevation.nationalmap.gov/arcgis/services/3DEPElevation'
                '/ImageServer/WCSServer with WGS84 bbox. Set NDSM_PROVIDER="usgs".'
            )
        elif provider == 'opentopo':
            raise NotImplementedError(
                'OpenTopography: get API key at https://opentopography.org, '
                'then use their REST API. Set NDSM_PROVIDER="opentopo".'
            )
        else:
            raise ValueError(f'Unknown NDSM_PROVIDER: {provider!r}')

    print('=' * 60)
    print(f'CELL 2d — LiDAR DSM + nDSM  (provider={NDSM_PROVIDER})')
    print('=' * 60)

    if os.path.exists(NDSM_TIFF):
        print(f'nDSM already computed: {NDSM_TIFF}')
        print('Delete ndsm.tif and dsm.tif to force recompute.')
    else:
        if not os.path.exists(EA_DTM_TIFF):
            raise FileNotFoundError(f'DTM not found: {EA_DTM_TIFF} — run CELL 2b first.')

        from pyproj import Transformer as _TrDSM
        _wgs_bng = _TrDSM.from_crs('EPSG:4326', 'EPSG:27700', always_xy=True)
        _e0, _n0 = _wgs_bng.transform(SCENE_WEST,  SCENE_SOUTH)
        _e1, _n1 = _wgs_bng.transform(SCENE_EAST,  SCENE_NORTH)
        _buf = 200
        _e0 -= _buf; _n0 -= _buf; _e1 += _buf; _n1 += _buf

        if not os.path.exists(EA_DSM_TIFF):
            _url = _dsm_wcs_url(NDSM_PROVIDER, _e0, _e1, _n0, _n1)
            print(f'Downloading DSM ({NDSM_PROVIDER.upper()}) ...')
            _r = requests.get(_url, timeout=180, stream=True)
            _r.raise_for_status()
            _total = 0
            with open(EA_DSM_TIFF, 'wb') as _f:
                for _chunk in _r.iter_content(chunk_size=1024*1024):
                    _f.write(_chunk)
                    _total += len(_chunk)
                    print(f'  {_total//1024//1024} MB ...', end='\r')
            print(f'\nSaved: {EA_DSM_TIFF}  ({os.path.getsize(EA_DSM_TIFF)//1024//1024} MB)')
        else:
            print(f'DSM already on disk: {EA_DSM_TIFF}')

        # ── nDSM = DSM - DTM, reprojected onto DTM grid ─────────────────
        import rasterio
        from rasterio.warp import reproject, Resampling

        print('Computing nDSM = DSM - DTM ...')
        with rasterio.open(EA_DTM_TIFF) as _dtm_ds:
            _dtm     = _dtm_ds.read(1).astype(np.float32)
            _profile = _dtm_ds.profile.copy()
            _nodata  = _dtm_ds.nodata or -9999.0

        with rasterio.open(EA_DSM_TIFF) as _dsm_ds:
            _dsm_aligned = np.empty_like(_dtm)
            reproject(
                source      = rasterio.band(_dsm_ds, 1),
                destination = _dsm_aligned,
                src_transform = _dsm_ds.transform,
                src_crs       = _dsm_ds.crs,
                dst_transform = _profile['transform'],
                dst_crs       = _profile['crs'],
                resampling    = Resampling.bilinear,
            )

        _ndsm = np.clip(_dsm_aligned - _dtm, 0.0, None)
        _ndsm[_dtm == _nodata] = 0.0

        _profile.update(dtype=rasterio.float32, nodata=0.0)
        with rasterio.open(NDSM_TIFF, 'w', **_profile) as _out:
            _out.write(_ndsm.astype(np.float32), 1)

        print(f'nDSM saved: {NDSM_TIFF}')
        print(f'  Height range: {_ndsm.min():.1f} – {_ndsm.max():.1f} m')
        print(f'  Pixels > 2 m : {(_ndsm > 2).sum():,}  '
              f'({100*(_ndsm > 2).mean():.1f}% of scene)')
        print(f'  Pixels > 5 m : {(_ndsm > 5).sum():,}  '
              f'({100*(_ndsm > 5).mean():.1f}% of scene)')


## CELL 2e — Merge LiDAR Tiles

Merges individual EA LiDAR 1km tiles (DTM and DSM) into single
GeoTIFFs using GDAL. Skips if merged files already exist and are valid
(non-zero data). Run once per campaign — takes ~1 min per merge.

**Run before CELL 2d.**


In [ ]:
# ============================================================
# CELL 2e — MERGE LIDAR TILES (DTM + DSM)
# ============================================================
# Merges all individual 1km EA LiDAR tiles in LIDAR_DIR into
# two single GeoTIFFs (DTM + DSM). Skips if merged files already
# exist and contain valid (non-zero) data.
# Requires: GDAL (gdal_merge.py on PATH)
# ============================================================
import os, glob
import numpy as np

def _merged_valid(path):
    if not os.path.exists(path):
        return False
    try:
        import rasterio
        with rasterio.open(path) as ds:
            arr = ds.read(1)
            mask = arr != (ds.nodata or -9999)
            return bool(mask.any() and arr[mask].max() > 1.0)
    except Exception:
        return False

def _merge_tiles(pattern, out_path, label):
    if _merged_valid(out_path):
        print(f'{label}: already merged and valid -> {os.path.basename(out_path)}')
        return
    tiles = sorted(glob.glob(os.path.join(LIDAR_DIR, pattern)))
    if not tiles:
        print(f'{label}: no tiles found matching {pattern} in {LIDAR_DIR}')
        return
    print(f'{label}: merging {len(tiles)} tiles -> {os.path.basename(out_path)} ...')
    tile_args = ' '.join(f'"{t}"' for t in tiles)
    cmd = (f'gdal_merge.py -o "{out_path}" -of GTiff '
           f'-co COMPRESS=LZW -co TILED=YES '
           f'-a_nodata -3.4028235e+38 {tile_args}')
    ret = os.system(cmd)
    if ret != 0:
        raise RuntimeError(f'gdal_merge.py failed (exit {ret}). Is GDAL installed?')
    if not _merged_valid(out_path):
        raise RuntimeError(f'Merge produced empty file: {out_path}')
    import rasterio
    with rasterio.open(out_path) as ds:
        arr = ds.read(1)
        mask = arr != (ds.nodata or -9999)
        print(f'  -> {os.path.basename(out_path)}  '
             f'min={arr[mask].min():.1f}  max={arr[mask].max():.1f}  '
             f'mean={arr[mask].mean():.1f} m')

print('=' * 60)
print('CELL 2e — Merge LiDAR Tiles')
print('=' * 60)
print(f'LIDAR_DIR: {LIDAR_DIR}')

_merge_tiles('*_DTM_1m.tif',    EA_DTM_TIFF, 'DTM')
_merge_tiles('*_FZ_DSM_1m.tif', EA_DSM_TIFF, 'DSM')

print('Done. Proceed to CELL 2d to compute nDSM.')


## CELL 3 — Build Terrain PLY

Samples the DTM GeoTIFF onto a regular `TERRAIN_GRID_N × TERRAIN_GRID_N` grid
and writes `terrain.ply`. All building and vegetation base heights are referenced
to this terrain surface so TX/RX positions sit exactly on the loaded scene.

Skip when `FLAT_TERRAIN=True`.


In [ ]:
# ============================================================
# CELL 3 — BUILD TERRAIN PLY
# ============================================================
# FLAT_TERRAIN=True          : flat z=0 plane — no DEM needed
# TERRAIN_SOURCE='aws_dem'   : DEM from AWS tiles  (run CELL 2 first)
# TERRAIN_SOURCE='ea_lidar'  : DEM from EA LiDAR  (run CELL 2b–2e first)
# Skips automatically if terrain.ply already exists on disk.
# ============================================================
_terrain_ply = os.path.join(MESH_DIR, 'terrain.ply')
if os.path.exists(_terrain_ply):
    print(f"terrain.ply already exists — skipping rebuild.")
    print(f"  {_terrain_ply}")
    print("Delete the file and re-run this cell to force a rebuild.")
else:
    import struct, os
    import numpy as np

    N = TERRAIN_GRID_N

    if FLAT_TERRAIN:
        print(f'Building FLAT terrain mesh: {N}×{N} grid ...')
        # Derive terrain extent from merged PLY bounds so terrain covers full scene
        _mdir = globals().get('MERGED_DIR', os.path.join(SCENE_DIR, 'meshes_merged'))
        _xmin, _xmax, _ymin, _ymax = 1e9, -1e9, 1e9, -1e9
        if os.path.isdir(_mdir):
            for _pf in os.listdir(_mdir):
                if not _pf.endswith('.ply') or 'terrain' in _pf: continue
                try:
                    import struct as _st
                    with open(os.path.join(_mdir, _pf), 'rb') as _f:
                        _hdr, _nv = [], 0
                        while True:
                            _l = _f.readline().decode('ascii','ignore').strip()
                            _hdr.append(_l)
                            if _l.startswith('element vertex'): _nv = int(_l.split()[-1])
                            if _l == 'end_header': break
                        if _nv > 0 and any('binary' in h for h in _hdr):
                            _raw = np.frombuffer(_f.read(_nv*12), dtype=np.float32).reshape(-1,3)
                            _xmin=min(_xmin,float(_raw[:,0].min())); _xmax=max(_xmax,float(_raw[:,0].max()))
                            _ymin=min(_ymin,float(_raw[:,1].min())); _ymax=max(_ymax,float(_raw[:,1].max()))
                except Exception: pass
        if _xmin < _xmax:
            _pad  = float(globals().get("TERRAIN_PAD_M", 5000.0))
            x_span = (_xmax - _xmin) + 2*_pad
            y_span = (_ymax - _ymin) + 2*_pad
            _ox = (_xmin + _xmax) / 2
            _oy = (_ymin + _ymax) / 2
            print(f'  Terrain from PLY bounds: {x_span:.0f} x {y_span:.0f} m  (pad={_pad}m)')
        else:
            _ox, _oy = 0.0, 0.0
            x_span = (SCENE_EAST - SCENE_WEST) * 111000 * np.cos(np.radians((SCENE_SOUTH+SCENE_NORTH)/2))
            y_span = (SCENE_NORTH - SCENE_SOUTH) * 111000
            print(f'  Terrain from bbox: {x_span:.0f} x {y_span:.0f} m')
        xs = np.linspace(_ox - x_span/2, _ox + x_span/2, N, dtype=np.float32)
        ys = np.linspace(_oy - y_span/2, _oy + y_span/2, N, dtype=np.float32)
        XX, YY = np.meshgrid(xs, ys, indexing='ij')
        ZZ = np.zeros((N, N), dtype=np.float32)
        origin_elev_asl = 0.0
        def local_z(lon, lat): return 0.0
    else:
        _src = globals().get('TERRAIN_SOURCE', 'aws_dem')
        if _src == 'ea_lidar' and os.path.exists(globals().get('EA_DTM_TIFF','')):
            print(f'Building terrain mesh from EA LiDAR DTM: {N}×{N} grid ...')
            import rasterio as _rio
            from pyproj import Transformer as _Tr
            _bng_to_utm = _Tr.from_crs('EPSG:27700', f'EPSG:{UTM_EPSG}', always_xy=True)
            _utm_to_bng = _Tr.from_crs(f'EPSG:{UTM_EPSG}', 'EPSG:27700', always_xy=True)
            _ds = _rio.open(EA_DTM_TIFF)
            _dem_data = _ds.read(1).astype(np.float32)
            _dem_tf   = _ds.transform
            _dem_nd   = _ds.nodata

            # scene origin elevation from EA DTM
            _ox_bng, _oy_bng = _utm_to_bng.transform(center_utm[0], center_utm[1])
            _oc, _or = ~_dem_tf * (_ox_bng, _oy_bng)
            _or, _oc = int(_or), int(_oc)
            if 0 <= _or < _dem_data.shape[0] and 0 <= _oc < _dem_data.shape[1]:
                origin_elev_asl = float(_dem_data[_or, _oc])
            else:
                origin_elev_asl = 0.0
            print(f'  Origin elevation: {origin_elev_asl:.1f} m ASL')

            def _ea_z(utm_x, utm_y):
                bx, by = _utm_to_bng.transform(utm_x, utm_y)
                col_f, row_f = ~_dem_tf * (bx, by)
                r, c = int(row_f), int(col_f)
                H, W = _dem_data.shape
                if 0 <= r < H-1 and 0 <= c < W-1:
                    dr, dc = row_f-r, col_f-c
                    z = ((1-dr)*(1-dc)*_dem_data[r,c] + (1-dr)*dc*_dem_data[r,c+1] +
                         dr*(1-dc)*_dem_data[r+1,c] + dr*dc*_dem_data[r+1,c+1])
                    if _dem_nd is None or not np.isclose(float(z), _dem_nd):
                        return float(z)
                return origin_elev_asl

            x_span = (SCENE_EAST-SCENE_WEST)*111000*np.cos(np.radians((SCENE_SOUTH+SCENE_NORTH)/2))
            y_span = (SCENE_NORTH-SCENE_SOUTH)*111000
            xs = np.linspace(-x_span/2, x_span/2, N, dtype=np.float32)
            ys = np.linspace(-y_span/2, y_span/2, N, dtype=np.float32)
            XX, YY = np.meshgrid(xs, ys, indexing='ij')
            ZZ = np.zeros((N, N), dtype=np.float32)
            import time as _t; t0 = _t.time()
            for i in range(N):
                for j in range(N):
                    ZZ[i,j] = _ea_z(center_utm[0]+XX[i,j], center_utm[1]+YY[i,j]) - origin_elev_asl
                if (i+1) % max(1,N//5)==0:
                    print(f'  row {i+1}/{N}  ({_t.time()-t0:.0f}s)')
            def local_z(lon, lat):
                ux,uy = to_utm.transform(lon,lat)
                return _ea_z(ux,uy) - origin_elev_asl
        else:
            print(f'Building terrain mesh from AWS DEM: {N}×{N} grid ...')
            x_span = ne_utm[0] - sw_utm[0]
            y_span = ne_utm[1] - sw_utm[1]
            xs = np.linspace(-x_span/2, x_span/2, N, dtype=np.float32)
            ys = np.linspace(-y_span/2, y_span/2, N, dtype=np.float32)
            XX, YY = np.meshgrid(xs, ys, indexing='ij')
            ZZ = np.zeros((N, N), dtype=np.float32)
            import time as _t; t0 = _t.time()
            for i in range(N):
                for j in range(N):
                    utm_x = center_utm[0] + XX[i, j]
                    utm_y = center_utm[1] + YY[i, j]
                    ZZ[i, j] = height_from_utm(utm_x, utm_y) - origin_elev_asl
                if (i+1) % max(1, N//5) == 0:
                    print(f'  row {i+1}/{N}  ({_t.time()-t0:.0f}s)')

    print(f'Terrain Z range: [{ZZ.min():.1f}, {ZZ.max():.1f}] m')

    verts = np.stack([XX.ravel(), YY.ravel(), ZZ.ravel()], axis=1).astype(np.float32)
    faces = []
    for i in range(N-1):
        for j in range(N-1):
            a = i*N+j; b = a+1; c = a+N; d = c+1
            faces.append([a,d,b]); faces.append([a,c,d])  # normals point UP (+z)
    faces = np.array(faces, dtype=np.int32)

    # Write binary PLY
    ply_path = os.path.join(MESH_DIR, 'terrain.ply')  # always save alongside other PLYs
    with open(ply_path, 'wb') as f:
        hdr = (f'ply\nformat binary_little_endian 1.0\n'
               f'element vertex {len(verts)}\nproperty float x\nproperty float y\nproperty float z\n'
               f'element face {len(faces)}\nproperty list uchar int vertex_indices\nend_header\n')
        f.write(hdr.encode())
        f.write(verts.tobytes())
        for fc in faces:
            f.write(struct.pack('<B3i', 3, *fc))

    kb = os.path.getsize(ply_path)//1024
    print(f'terrain.ply: {len(verts):,} verts  {len(faces):,} faces  {kb} KB  → {ply_path}')

    # Save origin elevation for main notebook compatibility
    import json as _json
    _elev_path = os.path.join(SCENE_DIR, 'origin_elev2.json')
    _json.dump({'origin_elev_asl_m': float(origin_elev_asl), 'flat_terrain': FLAT_TERRAIN}, open(_elev_path, 'w'))
    print(f'origin_elev2.json: {_elev_path}')


## CELL 4 — Build OSM Feature PLYs

Downloads and builds PLY meshes for all scene features:

**Buildings** — height priority:
1. LiDAR nDSM at footprint centroid (`USE_NDSM_HEIGHTS=True`)
2. OSM `height=` tag
3. OSM `building:levels × HEIGHT_PER_LEVEL_M`
4. `DEFAULT_HEIGHT_M` fallback

**Roads / Water** — flat polygons at terrain height with appropriate materials.

**Vegetation** — controlled by `VEG_3D_GEOMETRY`:
- `False` (default, recommended): flat ground patches → assigns `itu_vegetation`
  scattering material with no hard-shadow blocking. ITU-R P.833 Weissberger
  attenuation is applied post-hoc in the DEM simulation notebook.
- `True` (legacy): extrudes canopy volumes. **Not recommended at sub-1 GHz** —
  Sionna surfaces are opaque so extruded canopy over-attenuates by 10–15 dB
  in wooded corridors.


In [ ]:
# ============================================================
# CELL 4 — OSM BUILDINGS → BUILDING PLYS
# ============================================================

# Guard: local_z may not be defined if terrain cells were skipped
if 'local_z' not in dir():
    if 'height_from_wgs84' in dir() and 'origin_elev_asl' in dir():
        def local_z(lon, lat):
            return height_from_wgs84(lon, lat) - origin_elev_asl
    else:
        def local_z(lon, lat):
            return 0.0
        print("[WARN] local_z fallback: returning 0.0 (terrain cells not run)")

assert _HAS_OSMNX, 'osmnx required: pip install osmnx'

import osmnx as _ox_ver
_ox_version = tuple(int(x) for x in _ox_ver.__version__.split('.')[:2])

print(f'Downloading OSM buildings (osmnx {_ox_ver.__version__}) ...')
t0 = time.time()
try:
    if _ox_version >= (2, 0):
        gdf_bld = ox.features_from_bbox(
            bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH),
            tags={'building': True}
        )
    elif _ox_version >= (1, 3):
        gdf_bld = ox.features_from_bbox(
            bbox=(SCENE_NORTH, SCENE_SOUTH, SCENE_EAST, SCENE_WEST),
            tags={'building': True}
        )
    else:
        gdf_bld = ox.features_from_bbox(
            north=SCENE_NORTH, south=SCENE_SOUTH,
            east=SCENE_EAST,   west=SCENE_WEST,
            tags={'building': True}
        )
except Exception as e:
    print(f'osmnx error: {e}')
    raise
print(f'  {len(gdf_bld)} raw building features  ({time.time()-t0:.1f}s)')

# ── Material helpers ───────────────────────────────────────────────────────
def _parse_year(date_str):
    """Extract 4-digit year from OSM start_date= tag. Returns int or None."""
    import re
    if not date_str or str(date_str).lower() in ('nan', 'none', ''):
        return None
    m = re.search(r'\b(1[0-9]{3}|20[0-2][0-9])\b', str(date_str))
    return int(m.group(1)) if m else None

def _bld_mat(row):
    mat  = str(row.get('building:material', '')).lower()
    fmat = str(row.get('building:facade:material', '')).lower()
    tag  = str(row.get('building', '')).lower()
    amen = str(row.get('amenity', '')).lower()
    shop = str(row.get('shop', '')).lower()
    off  = str(row.get('office', '')).lower()
    # ── Priority 1: OSM building:material= tag (highest confidence, universal) ─
    _MAT_TAG_MAP = {
        # Concrete/masonry
        'concrete': 'itu_concrete', 'reinforced_concrete': 'itu_concrete',
        'cement': 'itu_concrete',  'precast': 'itu_concrete',
        # Brick/stone (similar EM properties)
        'brick': 'itu_brick', 'stone': 'itu_brick', 'sandstone': 'itu_brick',
        'limestone': 'itu_brick', 'granite': 'itu_brick', 'flint': 'itu_brick',
        'cobblestone': 'itu_brick', 'clay': 'itu_brick',
        # Wood
        'wood': 'itu_wood', 'timber': 'itu_wood', 'wooden': 'itu_wood',
        'logs': 'itu_wood', 'bamboo': 'itu_wood', 'thatch': 'itu_wood',
        'log': 'itu_wood',
        # Glass
        'glass': 'itu_glass', 'glazed': 'itu_glass',
        # Metal
        'metal': 'itu_metal', 'steel': 'itu_metal', 'aluminium': 'itu_metal',
        'aluminum': 'itu_metal', 'tin': 'itu_metal', 'copper': 'itu_metal',
        'zinc': 'itu_metal', 'iron': 'itu_metal', 'corrugated_iron': 'itu_metal',
    }
    for _mkey, _mval in _MAT_TAG_MAP.items():
        if _mkey in mat or _mkey in fmat: return _mval

    # ── Priority 2: building use-type → material (global mapping) ────────────
    # Glazed/curtain-wall structures
    if tag in ('greenhouse', 'glasshouse', 'winter_garden'):    return 'itu_glass'
    if amen in ('shopping_centre', 'mall'):                     return 'itu_glass'
    if shop in ('mall', 'supermarket', 'department_store'):     return 'itu_glass'

    # Metal-frame structures
    _METAL_TYPES = {'industrial','warehouse','factory','shed','barn','hangar',
                    'storage_tank','silo','container','quonset_hut','stable'}
    if tag in _METAL_TYPES: return 'itu_metal'

    # Residential — use age if available, else brick (universal default)
    _RESID_TYPES = {'residential','house','detached','semidetached_house',
                    'semi_detached','terrace','terrace_house','bungalow','hut',
                    'farm','farmhouse','dormitory','apartments','block','cabin',
                    'villa','cottage','chalet','mobile_home'}
    if tag in _RESID_TYPES: _type_mat = 'itu_brick'

    # Retail/commercial
    elif tag in ('retail','commercial','supermarket','kiosk','shop',
                 'convenience','mall'):  _type_mat = 'itu_brick'

    # Religious (stone/brick globally)
    elif tag in ('cathedral','church','chapel','mosque','temple','shrine',
                 'synagogue','monastery','convent'):  _type_mat = 'itu_brick'

    # Civic/institutional → concrete frame (post-1950 global standard)
    elif tag in ('school','university','hospital','civic','public','college',
                 'library','museum','government','courthouse','prison',
                 'police','fire_station','train_station','bus_station',
                 'airport_terminal','sports_centre','stadium'):
        _type_mat = 'itu_concrete'

    # Office — curtain wall or concrete
    elif off: _type_mat = 'itu_concrete'

    # Default — concrete (most common globally for unlabelled buildings)
    else: _type_mat = 'itu_concrete'

    # ── Priority 3: age-based override (Jansen et al. 2019) ──────────────────
    # Pre-1940: solid brick  |  1940-1980: concrete panel  |  post-1980: type-based
    if globals().get('USE_BUILDING_AGE', True):
        _yr = _parse_year(str(row.get('start_date', '') or ''))
        if _yr is not None:
            if _yr < 1940: return 'itu_brick'
            if _yr < 1980: return 'itu_concrete'
    # Building age override — Jansen et al. 2019 wall material classification
    # Pre-1940: Victorian/Edwardian solid brick (30 cm walls, high transmission loss)
    # 1940-1980: post-war concrete panel construction
    # Post-1980: modern construction — fall back to type-based above
    if globals().get('USE_BUILDING_AGE', False):
        _yr = _parse_year(str(row.get('start_date', '') or ''))
        if _yr is not None:
            if _yr < 1940:  return 'itu_brick'
            if _yr < 1980:  return 'itu_concrete'
    return _type_mat

def _roof_mat(row):
    roof_tag = str(row.get('roof:material', '')).lower()
    btag     = str(row.get('building', '')).lower()
    if any(k in roof_tag for k in ['metal','steel','zinc','aluminium','copper','tin']):
        return 'itu_metal'
    if 'glass' in roof_tag:                              return 'itu_glass'
    if any(k in roof_tag for k in ['wood','timber','thatch']): return 'itu_wood'
    if any(k in roof_tag for k in ['tile','concrete','slate','terracotta']):
        return 'itu_concrete'
    if btag in ('industrial','warehouse','factory','shed','barn',
                'retail','supermarket','commercial','garage','garages'):
        return 'itu_metal'
    return 'itu_metal'   # fallback: open-land sheds/barns/houses -> metal roofs

# ── nDSM sampler (open once, reuse for all buildings) ─────────────────────
_ndsm_ds  = None
_ndsm_arr = None
_ndsm_tfm = None
_ndsm_crs = None
if USE_NDSM_HEIGHTS and os.path.exists(NDSM_TIFF):
    import rasterio as _rio_n
    _ndsm_ds  = _rio_n.open(NDSM_TIFF)
    _ndsm_arr = _ndsm_ds.read(1).astype(float)
    _ndsm_tfm = _ndsm_ds.transform
    _ndsm_crs = _ndsm_ds.crs
    print(f'  nDSM loaded: {NDSM_TIFF}')
elif USE_NDSM_HEIGHTS:
    print(f'  WARNING: ndsm.tif not found — run CELL 2d first. Falling back to OSM tags.')

def _ndsm_height(lon, lat):
    if _ndsm_ds is None:
        return None
    from pyproj import Transformer as _TrN
    _to_raster = _TrN.from_crs('EPSG:4326',
                               _ndsm_crs.to_epsg() or 27700, always_xy=True)
    ex, ny = _to_raster.transform(lon, lat)
    from rasterio.transform import rowcol as _rc
    try:
        r, c = int(_rc(_ndsm_tfm, ex, ny)[0]), int(_rc(_ndsm_tfm, ex, ny)[1])
        if 0 <= r < _ndsm_arr.shape[0] and 0 <= c < _ndsm_arr.shape[1]:
            v = float(_ndsm_arr[r, c])
            if v >= NDSM_BUILDING_MIN_M:
                return float(np.clip(v, CITY_MIN_HEIGHT_M, CITY_MAX_HEIGHT_M))
    except Exception:
        pass
    return None

def _bld_height(row):
    # 1. nDSM — LiDAR measured height (most accurate, fills OSM gaps)
    if USE_NDSM_HEIGHTS:
        try:
            cen = row.geometry.centroid
            h = _ndsm_height(cen.x, cen.y)
            if h is not None:
                return h
        except Exception:
            pass
    # 2. OSM height= tag
    try:
        h = float(str(row.get('height','0')).replace('m','').strip())
        if h > 1: return np.clip(h, CITY_MIN_HEIGHT_M, CITY_MAX_HEIGHT_M)
    except: pass
    # 3. OSM building:levels
    try:
        lvl = float(str(row.get('building:levels','0')).strip())
        if lvl > 0: return np.clip(lvl * HEIGHT_PER_LEVEL_M, CITY_MIN_HEIGHT_M, CITY_MAX_HEIGHT_M)
    except: pass
    return DEFAULT_HEIGHT_M

def _roof_height(row, coords_utm):
    """Ridge height (m) above the eave + roof shape string.
    Priority: roof:shape=flat -> flat; roof:height tag; roof:levels;
    else derive from ROOF_PITCH_DEG and footprint short dimension."""
    shape = str(row.get('roof:shape', '')).lower().strip()
    if shape == 'flat':
        return 0.0, 'flat'
    try:
        rh = float(str(row.get('roof:height', '0')).replace('m', '').strip())
        if rh > 0.2:
            return float(np.clip(rh, 0.0, ROOF_MAX_RIDGE_M)), (shape or 'pyramidal')
    except Exception:
        pass
    try:
        rl = float(str(row.get('roof:levels', '0')).strip())
        if rl > 0:
            return float(np.clip(rl * 2.0, 0.0, ROOF_MAX_RIDGE_M)), (shape or 'pyramidal')
    except Exception:
        pass
    try:
        pa = np.asarray(coords_utm, dtype=float)
        span = min(pa[:, 0].max() - pa[:, 0].min(), pa[:, 1].max() - pa[:, 1].min())
        rh = (span / 2.0) * math.tan(math.radians(ROOF_PITCH_DEG))
        return float(np.clip(rh, 0.0, ROOF_MAX_RIDGE_M)), (shape or 'pyramidal')
    except Exception:
        return 0.0, 'flat'

def _write_ply(verts, faces, path):
    verts = np.asarray(verts, dtype=np.float32)
    faces = np.asarray(faces, dtype=np.int32)
    if _HAS_TRIMESH:
        trimesh.Trimesh(vertices=verts, faces=faces, process=False).export(path)
        return
    with open(path, 'w') as f:
        f.write('ply\nformat ascii 1.0\n')
        f.write(f'element vertex {len(verts)}\n')
        f.write('property float x\nproperty float y\nproperty float z\n')
        f.write(f'element face {len(faces)}\n')
        f.write('property list uchar int vertex_indices\nend_header\n')
        for v in verts:   f.write(f'{v[0]:.4f} {v[1]:.4f} {v[2]:.4f}\n')
        for fc in faces:  f.write(f'3 {fc[0]} {fc[1]} {fc[2]}\n')

def _triangulate_roof(pts_2d, top_z):
    """
    Robust Delaunay roof triangulation via Shapely.
    Handles convex AND concave footprints — no degenerate triangles.
    """
    poly = sg.Polygon(pts_2d)
    if not poly.is_valid:
        poly = poly.buffer(0)   # auto-repair self-intersections
    if poly.is_empty or poly.area < 0.1:
        return None

    try:
        tris = so.triangulate(poly)
    except Exception:
        return None

    v_idx = {}
    roof_v = []
    roof_f = []

    for tri in tris:
        # Keep only triangles whose centroid lies inside the footprint
        if not poly.contains(tri.centroid):
            continue
        coords = list(tri.exterior.coords)[:3]
        face_idxs = []
        for cx, cy in coords:
            key = (round(cx, 3), round(cy, 3))
            if key not in v_idx:
                v_idx[key] = len(roof_v)
                roof_v.append([cx, cy, top_z])
            face_idxs.append(v_idx[key])
        # Skip zero-area triangles (cross product check)
        a = np.array(roof_v[face_idxs[1]][:2]) - np.array(roof_v[face_idxs[0]][:2])
        b = np.array(roof_v[face_idxs[2]][:2]) - np.array(roof_v[face_idxs[0]][:2])
        if abs(a[0]*b[1] - a[1]*b[0]) > 1e-6:
            roof_f.append(face_idxs)

    if not roof_v or not roof_f:
        return None
    return np.array(roof_v, np.float32), np.array(roof_f, np.int32)

def _extrude_building(poly_utm, base_z, height, roof_h=0.0, roof_shape=''):
    pts = np.array(poly_utm, dtype=np.float32)
    if len(pts) < 3:
        return None
    if np.allclose(pts[0], pts[-1]):
        pts = pts[:-1]
    n = len(pts)
    top_z = base_z + height
    bot_z = base_z

    # ── Walls ──────────────────────────────────────────────────────────────
    wall_v, wall_f = [], []
    for i in range(n):
        j = (i+1) % n
        v_base = len(wall_v)
        wall_v += [
            [pts[i,0], pts[i,1], bot_z],
            [pts[j,0], pts[j,1], bot_z],
            [pts[j,0], pts[j,1], top_z],
            [pts[i,0], pts[i,1], top_z],
        ]
        wall_f += [
            [v_base,   v_base+1, v_base+2],
            [v_base,   v_base+2, v_base+3],
        ]

    # ── Roof ──────────────────────────────────────────────────────────────
    # Pitched (pyramidal/hip) roof when roof_h>0 and shape!='flat'; else a
    # flat Delaunay cap. The apex sits at the footprint representative point
    # raised by roof_h, so the roof is gap-free for any footprint and
    # presents sloped faces for realistic diffuse scattering.
    if roof_h > 0.2 and roof_shape not in ('flat',):
        cx, cy = float(pts[:,0].mean()), float(pts[:,1].mean())
        try:
            _c = sg.Polygon([(pts[i,0], pts[i,1]) for i in range(n)]).representative_point()
            cx, cy = float(_c.x), float(_c.y)
        except Exception:
            pass
        rv = [[pts[i,0], pts[i,1], top_z] for i in range(n)]
        apex = len(rv)
        rv.append([cx, cy, top_z + roof_h])
        rf = [[i, (i+1) % n, apex] for i in range(n)]
        rv = np.array(rv, np.float32); rf = np.array(rf, np.int32)
    else:
        roof_result = _triangulate_roof([(pts[i,0], pts[i,1]) for i in range(n)], top_z)
        if roof_result is None:
            return None   # skip buildings with unfixable geometry
        rv, rf = roof_result
    return (np.array(wall_v, np.float32), np.array(wall_f, np.int32), rv, rf)

# ── Process buildings — accumulate geometry per material ────────────────────
mat_geom = {}   # mat -> {'verts': [], 'faces': [], 'offset': 0}
n_ok = n_skip = 0
_n_total = len(gdf_bld)
t0 = time.time()

for idx, (oid, row) in enumerate(gdf_bld.iterrows()):
    geom = row.geometry
    if geom is None or geom.is_empty:
        n_skip += 1; continue

    if isinstance(geom, MultiPolygon):
        geom = max(geom.geoms, key=lambda g: g.area)
    if not isinstance(geom, Polygon):
        n_skip += 1; continue

    btag = str(row.get('building','')).lower()
    if btag in EXCLUDE_BUILDING_TYPES:
        n_skip += 1; continue

    coords_wgs = list(geom.exterior.coords)
    coords_utm = []
    for lon, lat in coords_wgs:
        ex, ny = to_utm.transform(lon, lat)
        coords_utm.append((ex - center_utm[0], ny - center_utm[1]))

    area_m2 = sg.Polygon(coords_utm).area
    if area_m2 < MIN_BUILDING_AREA_M2:
        n_skip += 1; continue

    c_lon, c_lat = geom.centroid.x, geom.centroid.y
    base_z_local = local_z(c_lon, c_lat)
    h = _bld_height(row)
    roof_h, roof_shape = _roof_height(row, coords_utm)

    result = _extrude_building(coords_utm, base_z_local, h, roof_h, roof_shape)
    if result is None:
        n_skip += 1; continue
    wv, wf, rv, rf = result

    w_mat = _bld_mat(row)
    r_mat = _roof_mat(row)

    for mat, verts, faces in [(w_mat, wv, wf), (r_mat, rv, rf)]:
        if mat not in mat_geom:
            mat_geom[mat] = {'verts': [], 'faces': [], 'offset': 0}
        g = mat_geom[mat]
        g['faces'].append(faces + g['offset'])
        g['verts'].append(verts)
        g['offset'] += len(verts)

    n_ok += 1
    if n_ok % 100 == 0:
        print(f'  {n_ok}/{_n_total} buildings  ({time.time()-t0:.0f}s elapsed) ...')

print(f'\nBuildings : {n_ok} exported, {n_skip} skipped')
print(f'Materials : {list(mat_geom.keys())}')

# ── Write one merged PLY per material ───────────────────────────────────────
mat_plys = {}
for mat, g in mat_geom.items():
    all_v = np.concatenate(g['verts'], axis=0).astype(np.float32)
    all_f = np.concatenate(g['faces'], axis=0).astype(np.int32)
    ply_name = f'bld_{mat}.ply'
    ply_path = os.path.join(MESH_DIR, ply_name)
    _write_ply(all_v, all_f, ply_path)
    mat_plys[mat] = [('meshes/' + ply_name, 'buildings')]
    print(f'  {mat}: {len(all_v)} verts, {len(all_f)} faces → {ply_name}')

print(f'\nPLY files written: {len(mat_plys)} (one per material)')

# ── Roads → itu_asphalt PLY ──────────────────────────────────────────────────
if INCLUDE_ROADS:
    print('\nDownloading OSM roads ...')
    try:
        _road_tags = {'highway': True}  # all OSM highway types
        if _ox_version >= (2, 0):
            gdf_roads = ox.features_from_bbox(
                bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH), tags=_road_tags)
        elif _ox_version >= (1, 3):
            gdf_roads = ox.features_from_bbox(
                bbox=(SCENE_NORTH, SCENE_SOUTH, SCENE_EAST, SCENE_WEST), tags=_road_tags)
        else:
            gdf_roads = ox.features_from_bbox(
                north=SCENE_NORTH, south=SCENE_SOUTH,
                east=SCENE_EAST, west=SCENE_WEST, tags=_road_tags)
        _ROAD_WIDTH = {'motorway':12,'trunk':10,'primary':8,'secondary':7,
                       'tertiary':6,'residential':5,'unclassified':4,'service':3,'road':4}
        r_verts, r_faces, r_off = [], [], 0
        from shapely.geometry import LineString as _LS, MultiLineString as _MLS
        for _, row in gdf_roads.iterrows():
            geom = row.geometry
            if geom is None or geom.is_empty: continue
            if globals().get('EXCLUDE_TUNNELS', True) and \
               str(row.get('tunnel', '')).lower() in ('yes', 'building_passage', 'avalanche_protector'):
                continue
            hw = str(row.get('highway','')).lower()
            width = _ROAD_WIDTH.get(hw, 4) / 2.0
            lines = geom.geoms if isinstance(geom, _MLS) else [geom]
            for line in lines:
                if not isinstance(line, _LS): continue
                coords_utm = []
                for lon, lat in line.coords:
                    ex, ny = to_utm.transform(lon, lat)
                    coords_utm.append((ex - center_utm[0], ny - center_utm[1]))
                if len(coords_utm) < 2: continue
                poly = sg.LineString(coords_utm).buffer(width, cap_style=2, join_style=2)
                if poly.is_empty: continue
                pts = np.array(poly.exterior.coords[:-1], np.float32)
                n = len(pts)
                if n < 3: continue
                verts = np.hstack([pts, np.zeros((n,1), np.float32)])
                cx, cy = pts[:,0].mean(), pts[:,1].mean()
                base = r_off
                r_verts.append(verts)
                r_verts.append(np.array([[cx, cy, 0.0]], np.float32))
                for i in range(n):
                    r_faces.append([base+i, base+(i+1)%n, base+n])
                r_off += n + 1
        if r_verts:
            rv_all = np.concatenate(r_verts).astype(np.float32)
            rf_all = np.array(r_faces, np.int32)
            ply_name = 'road_itu_asphalt.ply'
            _write_ply(rv_all, rf_all, os.path.join(MESH_DIR, ply_name))
            mat_plys['itu_asphalt'] = [('meshes/' + ply_name, 'roads')]
            print(f'  Roads: {len(rv_all):,} verts, {len(rf_all):,} faces → {ply_name}')
        else:
            print('  Roads: no geometry generated')
    except Exception as _e:
        print(f'  Roads download failed: {_e}')
else:
    print('Roads: skipped (INCLUDE_ROADS=False)')

# ── Water → itu_water PLY ────────────────────────────────────────────────────
if INCLUDE_WATER:
    print('\nDownloading OSM water ...')
    try:
        _water_tags = {'natural': ['water','river','stream','lake'],
                       'waterway': ['river','stream','canal','drain']}
        if _ox_version >= (2, 0):
            gdf_water = ox.features_from_bbox(
                bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH), tags=_water_tags)
        elif _ox_version >= (1, 3):
            gdf_water = ox.features_from_bbox(
                bbox=(SCENE_NORTH, SCENE_SOUTH, SCENE_EAST, SCENE_WEST), tags=_water_tags)
        else:
            gdf_water = ox.features_from_bbox(
                north=SCENE_NORTH, south=SCENE_SOUTH,
                east=SCENE_EAST, west=SCENE_WEST, tags=_water_tags)
        from shapely.geometry import Polygon as _WPoly, MultiPolygon as _WMPoly
        from shapely.geometry import LineString as _WLS, MultiLineString as _WMLS
        w_verts, w_faces, w_off = [], [], 0
        for _, row in gdf_water.iterrows():
            geom = row.geometry
            if geom is None or geom.is_empty: continue
            polys = []
            if isinstance(geom, _WPoly): polys = [geom]
            elif isinstance(geom, _WMPoly): polys = list(geom.geoms)
            elif isinstance(geom, (_WLS, _WMLS)):
                for line in (geom.geoms if isinstance(geom, _WMLS) else [geom]):
                    # buffer in UTM metres (5 m half-width) not degrees
                    _lcoords = []
                    for lon, lat in line.coords:
                        ex, ny = to_utm.transform(lon, lat)
                        _lcoords.append((ex - center_utm[0], ny - center_utm[1]))
                    if len(_lcoords) >= 2:
                        wp = sg.LineString(_lcoords).buffer(5.0)
                        if not wp.is_empty: polys.append(wp)
                    continue   # already in UTM; skip second conversion below
            for poly in polys:
                if poly.is_empty: continue
                # flatten MultiPolygon into individual polygons
                from shapely.geometry import MultiPolygon as _MP2
                sub_polys = list(poly.geoms) if isinstance(poly, _MP2) else [poly]
                for poly in sub_polys:
                    if poly.exterior is None or poly.is_empty: continue
                # check if already UTM (from line buffer above) or still WGS84
                _first = poly.exterior.coords[0]
                _is_utm = abs(_first[0]) > 180 or abs(_first[1]) > 90
                if _is_utm:
                    coords_utm = [(x, y) for x, y in poly.exterior.coords[:-1]]
                else:
                    coords_utm = []
                    for lon, lat in poly.exterior.coords[:-1]:
                        ex, ny = to_utm.transform(lon, lat)
                        coords_utm.append((ex - center_utm[0], ny - center_utm[1]))
                n = len(coords_utm)
                if n < 3: continue
                if sg.Polygon(coords_utm).area < 1.0: continue   # m^2
                pts = np.array(coords_utm, np.float32)
                verts = np.hstack([pts, np.zeros((n,1), np.float32)])
                cx, cy = pts[:,0].mean(), pts[:,1].mean()
                base = w_off
                w_verts.append(verts)
                w_verts.append(np.array([[cx, cy, 0.0]], np.float32))
                for i in range(n):
                    w_faces.append([base+i, base+(i+1)%n, base+n])
                w_off += n + 1
        if w_verts:
            wv_all = np.concatenate(w_verts).astype(np.float32)
            wf_all = np.array(w_faces, np.int32)
            ply_name = 'water_itu_water.ply'
            _write_ply(wv_all, wf_all, os.path.join(MESH_DIR, ply_name))
            mat_plys['itu_water'] = [('meshes/' + ply_name, 'water')]
            print(f'  Water: {len(wv_all):,} verts, {len(wf_all):,} faces → {ply_name}')
        else:
            print('  Water: no geometry generated')
    except Exception as _e:
        print(f'  Water download failed: {_e}')
else:
    print('Water: skipped (INCLUDE_WATER=False)')

# ── Vegetation → itu_vegetation PLY ─────────────────────────────────────────
if INCLUDE_VEGETATION:
    print('\nDownloading OSM vegetation ...')
    try:
        _veg_tags = {'landuse': ['forest'],
                     'natural': ['wood']}
        if _ox_version >= (2, 0):
            gdf_veg = ox.features_from_bbox(
                bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH), tags=_veg_tags)
        elif _ox_version >= (1, 3):
            gdf_veg = ox.features_from_bbox(
                bbox=(SCENE_NORTH, SCENE_SOUTH, SCENE_EAST, SCENE_WEST), tags=_veg_tags)
        else:
            gdf_veg = ox.features_from_bbox(
                north=SCENE_NORTH, south=SCENE_SOUTH,
                east=SCENE_EAST, west=SCENE_WEST, tags=_veg_tags)
        from shapely.geometry import Polygon as _VPoly, MultiPolygon as _VMPoly
        vg_verts, vg_faces, vg_off = [], [], 0
        _veg_min  = float(globals().get('VEG_MIN_AREA_M2', 50.0))
        _veg_hmap = dict(globals().get('VEG_CLASS_HEIGHT_M', {}))
        _veg_tall = float(globals().get('VEG_TALL_MIN_M', 1.5))
        _veg_def  = float(globals().get('VEG_CANOPY_M', 12.0))
        def _veg_height(row):
            # height from the vegetation class tag (landuse/natural)
            for _k in ('natural', 'landuse'):
                _t = str(row.get(_k, '') or '').lower()
                if _t in _veg_hmap:
                    return _veg_hmap[_t]
            return _veg_def
        _veg_skip = 0
        for _, row in gdf_veg.iterrows():
            geom = row.geometry
            if geom is None or geom.is_empty: continue
            polys = []
            if isinstance(geom, _VPoly): polys = [geom]
            elif isinstance(geom, _VMPoly): polys = list(geom.geoms)
            for poly in polys:
                if poly.is_empty: continue
                coords_utm = []
                for lon, lat in poly.exterior.coords[:-1]:
                    ex, ny = to_utm.transform(lon, lat)
                    coords_utm.append((ex - center_utm[0], ny - center_utm[1]))
                n = len(coords_utm)
                if n < 3: continue
                if sg.Polygon(coords_utm).area < _veg_min: continue   # m^2
                # VEG_3D_GEOMETRY=False (best practice, default):
                #   flat patch at terrain height -> itu_vegetation material
                #   assigned to ground area; no hard shadow blocking.
                #   ITU-R P.833 attenuation applied post-hoc in DEM notebook.
                # VEG_3D_GEOMETRY=True: legacy canopy extrusion (over-blocks at 915 MHz)
                try:
                    _cen = poly.centroid
                    base_z = 0.0 if FLAT_TERRAIN else float(local_z(_cen.x, _cen.y))
                except Exception:
                    base_z = 0.0
                if not VEG_3D_GEOMETRY:
                    # flat fan triangulation at terrain height
                    _n = len(coords_utm)
                    _vp = np.array([[x, y, base_z] for x, y in coords_utm], np.float32)
                    for _fi in range(_n - 2):
                        vg_faces.append([vg_off, vg_off + _fi + 1, vg_off + _fi + 2])
                    vg_verts.append(_vp)
                    vg_off += _n
                    continue
                # ── legacy extrusion path (VEG_3D_GEOMETRY=True) ────────────────────
                _veg_h = _veg_height(row)
                if _veg_h < _veg_tall:
                    _veg_skip += 1; continue
                _res = _extrude_building(coords_utm, base_z, _veg_h, roof_h=0.0, roof_shape='flat')
                if _res is None: continue
                _wv, _wf, _rv, _rf = _res
                for _vv, _ff in ((_wv, _wf), (_rv, _rf)):
                    if _vv is None or len(_vv) == 0 or _ff is None or len(_ff) == 0:
                        continue
                    _base = vg_off
                    _arr = np.asarray(_vv, np.float32)
                    vg_verts.append(_arr)
                    for _f in _ff:
                        vg_faces.append([_base+int(_f[0]), _base+int(_f[1]), _base+int(_f[2])])
                    vg_off += len(_arr)
        if not VEG_3D_GEOMETRY:
            print(f'  Vegetation: {len(vg_faces):,} flat ground patches '
                  f'(VEG_3D_GEOMETRY=False — P.833 attenuation in DEM notebook)')
        elif _veg_skip > 0:
            print(f'  Vegetation: skipped {_veg_skip} ground-cover polygons (< {_veg_tall} m)')
        if vg_verts:
            vv_all = np.concatenate(vg_verts).astype(np.float32)
            vf_all = np.array(vg_faces, np.int32)
            ply_name = 'veg_itu_vegetation.ply'
            _write_ply(vv_all, vf_all, os.path.join(MESH_DIR, ply_name))
            mat_plys['itu_vegetation'] = [('meshes/' + ply_name, 'vegetation')]
            print(f'  Vegetation: {len(vv_all):,} verts, {len(vf_all):,} faces → {ply_name}')
            print(f'  Vegetation: skipped {_veg_skip} ground-cover polygons (< {_veg_tall} m)')
        else:
            print('  Vegetation: no geometry generated')
    except Exception as _e:
        print(f'  Vegetation download failed: {_e}')
else:
    print('Vegetation: skipped (INCLUDE_VEGETATION=False)')


# ── Individual trees → itu_vegetation disks ─────────────────────────────────
# OSM natural=tree points → flat disk at canopy height → itu_vegetation material
# Represents per-tree diffraction / scattering obstacle (ITU-R P.833-10 §3)
# Uses nDSM-sampled height when available; falls back to TREE_DEFAULT_HT_M
if globals().get('INCLUDE_TREES', True):
    print('\nDownloading OSM individual trees ...')
    try:
        _tree_tags = {'natural': 'tree'}
        if _ox_version >= (2, 0):
            gdf_trees = ox.features_from_bbox(
                bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH), tags=_tree_tags)
        elif _ox_version >= (1, 3):
            gdf_trees = ox.features_from_bbox(
                bbox=(SCENE_NORTH, SCENE_SOUTH, SCENE_EAST, SCENE_WEST), tags=_tree_tags)
        else:
            gdf_trees = ox.features_from_bbox(
                north=SCENE_NORTH, south=SCENE_SOUTH,
                east=SCENE_EAST,   west=SCENE_WEST, tags=_tree_tags)
        _tree_def_h = float(globals().get('TREE_DEFAULT_HT_M',  8.0))
        _tree_r     = float(globals().get('TREE_DISK_RADIUS_M', 3.0))
        _tree_segs  = 8      # octagon disk — enough for RF, fast to build
        tv_verts, tv_faces, tv_off = [], [], 0
        _tree_count = 0
        for _, trow in gdf_trees.iterrows():
            geom = trow.geometry
            if geom is None or geom.is_empty: continue
            # Accept point geometry only (individual trees, not rows/areas)
            if geom.geom_type != 'Point': continue
            tx_utm, ty_utm = to_utm.transform(geom.x, geom.y)
            lx = tx_utm - center_utm[0]
            ly = ty_utm - center_utm[1]
            # Canopy height: OSM height= tag → nDSM sample → default
            _ht = None
            _h_tag = str(trow.get('height', '') or '')
            if _h_tag and _h_tag not in ('nan','None',''):
                try: _ht = float(_h_tag.replace('m','').strip())
                except ValueError: pass
            if _ht is None and _ndsm_arr is not None:
                try: _ht = float(local_z(geom.x, geom.y))
                except Exception: pass
            if _ht is None or _ht < 1.5:
                _ht = _tree_def_h
            # Build octagon disk at canopy height (flat horizontal polygon)
            _angles = [2 * np.pi * k / _tree_segs for k in range(_tree_segs)]
            _disk   = np.array(
                [[lx + _tree_r * np.cos(a), ly + _tree_r * np.sin(a), _ht]
                 for a in _angles], dtype=np.float32)
            tv_verts.append(_disk)
            for fi in range(_tree_segs - 2):
                tv_faces.append([tv_off, tv_off + fi + 1, tv_off + fi + 2])
            tv_off += _tree_segs
            _tree_count += 1
        if tv_verts:
            # Merge with existing vegetation PLY if present, else create new
            _tv_v = np.concatenate(tv_verts).astype(np.float32)
            _tv_f = np.array(tv_faces, np.int32)
            _tree_ply = 'trees_itu_vegetation.ply'
            _write_ply(_tv_v, _tv_f, os.path.join(MESH_DIR, _tree_ply))
            if 'itu_vegetation' in mat_plys:
                mat_plys['itu_vegetation'].append(('meshes/' + _tree_ply, 'trees'))
            else:
                mat_plys['itu_vegetation'] = [('meshes/' + _tree_ply, 'trees')]
            print(f'  Trees: {_tree_count} individual trees → {_tree_ply} '
                  f'({len(_tv_v):,} verts, {len(_tv_f):,} faces)')
        else:
            print('  Trees: no individual tree points found in scene bbox')
    except Exception as _e:
        print(f'  Trees download failed: {_e}')
else:
    print('Trees: skipped (INCLUDE_TREES=False)')

# ── Bridges → itu_concrete PLY ───────────────────────────────────────────────
if globals().get('INCLUDE_BRIDGES', False):
    print('\nDownloading OSM bridges ...')
    try:
        # man_made=bridge returns actual bridge STRUCTURE polygons (deck footprints).
        # bridge=True returns road ways tagged bridge=yes (LineStrings) — no geometry.
        _br_tags = {'man_made': 'bridge'}
        if _ox_version >= (2, 0):
            gdf_br = ox.features_from_bbox(
                bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH), tags=_br_tags)
        elif _ox_version >= (1, 3):
            gdf_br = ox.features_from_bbox(
                bbox=(SCENE_NORTH, SCENE_SOUTH, SCENE_EAST, SCENE_WEST), tags=_br_tags)
        else:
            gdf_br = ox.features_from_bbox(north=SCENE_NORTH, south=SCENE_SOUTH,
                                            east=SCENE_EAST, west=SCENE_WEST, tags=_br_tags)
        _br_polys = gdf_br[gdf_br.geometry.geom_type.isin(['Polygon','MultiPolygon'])]
        print(f'  {len(_br_polys)} bridge structure polygons')
        _br_v, _br_f, _br_off = [], [], 0
        _br_count = 0
        _DECK_THICKNESS = 1.5   # concrete slab thickness (m) — creates side walls + bottom face
        _DECK_CLEARANCE = 5.0   # height of deck bottom above ground (m)
        for _, row in _br_polys.iterrows():
            try:
                geom = row.geometry
                polys = list(geom.geoms) if geom.geom_type == 'MultiPolygon' else [geom]
                for poly in polys:
                    coords_utm = []
                    for lon, lat in poly.exterior.coords[:-1]:
                        ex, ny = to_utm.transform(lon, lat)
                        coords_utm.append((ex - center_utm[0], ny - center_utm[1]))
                    _cen = poly.centroid
                    base_z = local_z(_cen.x, _cen.y)
                    # base_z = bottom of slab; height = slab thickness → top + bottom + 4 side walls
                    slab_base = base_z + _DECK_CLEARANCE
                    res = _extrude_building(coords_utm, slab_base, _DECK_THICKNESS,
                                           roof_h=0.0, roof_shape='flat')
                    if res is None: continue
                    _wv, _wf, _rv, _rf = res
                    for _vv, _ff in ((_wv, _wf), (_rv, _rf)):
                        if _vv is None or len(_vv) == 0 or _ff is None or len(_ff) == 0: continue
                        _br_f_s = [[fi + _br_off for fi in face] for face in _ff]
                        _br_v.append(_vv); _br_f.extend(_br_f_s); _br_off += len(_vv)
                    _br_count += 1
            except Exception: pass
        if _br_v:
            _brv = np.concatenate(_br_v).astype(np.float32)
            _brf = np.array(_br_f, np.int32)
            _write_ply(_brv, _brf, os.path.join(MESH_DIR, 'bld_itu_concrete_bridges.ply'))
            mat_plys.setdefault('itu_concrete', []).append(
                (os.path.basename(MESH_DIR)+'/bld_itu_concrete_bridges.ply', 'buildings'))
            print(f'  Bridges: {_br_count} → bld_itu_concrete_bridges.ply ({len(_brv):,} verts, {len(_brf):,} faces)')
        else:
            print('  Bridges: none found (man_made=bridge) — check bbox or OSM coverage')
    except Exception as _e:
        print(f'  Bridges download failed: {_e}')
else:
    print('Bridges: skipped (INCLUDE_BRIDGES=False)')

# ── Railway embankments → itu_concrete PLY ───────────────────────────────────
if globals().get('INCLUDE_EMBANKMENTS', False):
    print('\nDownloading OSM railway embankments ...')
    try:
        _emb_tags = {'railway': ['rail','light_rail','tram','subway'],
                     'man_made': ['embankment']}
        if _ox_version >= (2, 0):
            gdf_emb = ox.features_from_bbox(
                bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH), tags=_emb_tags)
        else:
            gdf_emb = ox.features_from_bbox(
                bbox=(SCENE_NORTH, SCENE_SOUTH, SCENE_EAST, SCENE_WEST), tags=_emb_tags)
        _emb_lines = gdf_emb[gdf_emb.geometry.geom_type.isin(['LineString','MultiLineString'])]
        print(f'  {len(_emb_lines)} railway/embankment lines')
        ev_all, ef_all, e_off = [], [], 0
        _EMB_H = 4.0   # typical embankment height (m)
        _EMB_W = 8.0   # half-width (m)
        for _, row in _emb_lines.iterrows():
            try:
                geom = row.geometry
                lines = list(geom.geoms) if 'Multi' in geom.geom_type else [geom]
                for line in lines:
                    coords = list(line.coords)
                    _lcoords = []
                    for x, y in coords:
                        ex, ny = to_utm.transform(x, y)
                        _lcoords.append((ex, ny))
                    poly = sg.LineString(_lcoords).buffer(_EMB_W, cap_style=2)
                    base_z = local_z(*to_wgs84.transform(poly.centroid.x, poly.centroid.y))
                    # Extrude buffered polygon → solid wall (top + sides + bottom)
                    ext_coords_utm = [(x - center_utm[0], y - center_utm[1])
                                      for x, y in poly.exterior.coords[:-1]]
                    res = _extrude_building(ext_coords_utm, base_z, _EMB_H,
                                           roof_h=0.0, roof_shape='flat')
                    if res is None: continue
                    _wv, _wf, _rv, _rf = res
                    for _vv, _ff in ((_wv, _wf), (_rv, _rf)):
                        if _vv is None or len(_vv) == 0 or _ff is None or len(_ff) == 0: continue
                        _ef_s = [[fi + e_off for fi in face] for face in _ff]
                        ev_all.append(_vv); ef_all.extend(_ef_s); e_off += len(_vv)
            except Exception: pass
        if ev_all:
            ev = np.concatenate(ev_all).astype(np.float32)
            ef = np.array(ef_all, np.int32)
            _write_ply(ev, ef, os.path.join(MESH_DIR, 'bld_itu_concrete_embankments.ply'))
            mat_plys.setdefault('itu_concrete', []).append(
                (os.path.basename(MESH_DIR)+'/bld_itu_concrete_embankments.ply', 'buildings'))
            print(f'  Embankments: {len(ev):,} verts, {len(ef):,} faces → bld_itu_concrete_embankments.ply')
        else:
            print('  Embankments: no geometry found')
    except Exception as _e:
        print(f'  Embankments download failed: {_e}')
else:
    print('Embankments: skipped (INCLUDE_EMBANKMENTS=False)')

# ── Road embankments (highway + embankment=yes) → itu_concrete PLY ──────────
# M1, A610, A52 raised road sections — earth/concrete mounds blocking lateral scatter
if globals().get('INCLUDE_ROAD_EMBANKMENTS', True):
    print('\nDownloading OSM road embankments ...')
    try:
        _rdmb_tags = {'highway': ['motorway','trunk','primary','secondary']}
        if _ox_version >= (2, 0):
            gdf_rdmb = ox.features_from_bbox(
                bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH), tags=_rdmb_tags)
        else:
            gdf_rdmb = ox.features_from_bbox(
                bbox=(SCENE_NORTH, SCENE_SOUTH, SCENE_EAST, SCENE_WEST), tags=_rdmb_tags)
        if 'embankment' in gdf_rdmb.columns:
            _rdmb_mask = gdf_rdmb['embankment'].fillna('') == 'yes'
        else:
            _rdmb_mask = gdf_rdmb.index.map(lambda _: False)
        _rdmb_lines = gdf_rdmb[_rdmb_mask & gdf_rdmb.geometry.geom_type.isin(['LineString','MultiLineString'])]
        print(f'  {len(_rdmb_lines)} road embankment ways')
        rdmb_v, rdmb_f, rdmb_off = [], [], 0
        _RDMB_H = 4.0    # typical road embankment height (m)
        _RDMB_W = 12.0   # half-width buffer — motorway wider than railway
        for _, row in _rdmb_lines.iterrows():
            try:
                geom = row.geometry
                lines = list(geom.geoms) if 'Multi' in geom.geom_type else [geom]
                for line in lines:
                    _lcoords = [(*(to_utm.transform(x, y)),) for x, y in list(line.coords)]
                    poly = sg.LineString(_lcoords).buffer(_RDMB_W, cap_style=2)
                    base_z = local_z(*to_wgs84.transform(poly.centroid.x, poly.centroid.y))
                    ext_coords = [(x - center_utm[0], y - center_utm[1])
                                  for x, y in poly.exterior.coords[:-1]]
                    res = _extrude_building(ext_coords, base_z, _RDMB_H, roof_h=0.0, roof_shape='flat')
                    if res is None: continue
                    for _vv, _ff in zip(res[0::2], res[1::2]):
                        if _vv is None or len(_vv) == 0: continue
                        rdmb_f.extend([[fi + rdmb_off for fi in face] for face in _ff])
                        rdmb_v.append(_vv); rdmb_off += len(_vv)
            except Exception: pass
        if rdmb_v:
            _rv = np.concatenate(rdmb_v).astype(np.float32)
            _rf = np.array(rdmb_f, np.int32)
            _write_ply(_rv, _rf, os.path.join(MESH_DIR, 'infra_itu_concrete_road_embankments.ply'))
            mat_plys.setdefault('itu_concrete', []).append(
                (os.path.basename(MESH_DIR)+'/infra_itu_concrete_road_embankments.ply', 'infrastructure'))
            print(f'  Road embankments: {len(_rv):,} verts, {len(_rf):,} faces → infra_itu_concrete_road_embankments.ply')
        else:
            print('  Road embankments: none found (highway + embankment=yes) — check OSM data for bbox')
    except Exception as _e:
        print(f'  Road embankments download failed: {_e}')
else:
    print('Road embankments: skipped (INCLUDE_ROAD_EMBANKMENTS=False)')

# ── Road cuttings (highway + cutting=yes) → itu_concrete wall PLY ───────────
# A610, A52 depressed sections — vertical retaining walls on both sides block
# scatter paths entering/exiting the cut (ITU-R P.2040-2 concrete walls)
if globals().get('INCLUDE_ROAD_CUTTINGS', True):
    print('\nDownloading OSM road cuttings ...')
    try:
        _cut_tags = {'highway': ['motorway','trunk','primary','secondary']}
        if _ox_version >= (2, 0):
            gdf_cut = ox.features_from_bbox(
                bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH), tags=_cut_tags)
        else:
            gdf_cut = ox.features_from_bbox(
                bbox=(SCENE_NORTH, SCENE_SOUTH, SCENE_EAST, SCENE_WEST), tags=_cut_tags)
        if 'cutting' in gdf_cut.columns:
            _cut_mask = gdf_cut['cutting'].fillna('').isin(['yes','left','right','both'])
        else:
            _cut_mask = gdf_cut.index.map(lambda _: False)
        _cut_lines = gdf_cut[_cut_mask & gdf_cut.geometry.geom_type.isin(['LineString','MultiLineString'])]
        print(f'  {len(_cut_lines)} road cutting ways')
        cut_v, cut_f, cut_off = [], [], 0
        _CUT_W = 10.0   # half-width from centreline to cutting wall face (m)
        _CUT_H =  3.0   # cutting retaining wall height above road level (m)
        for _, row in _cut_lines.iterrows():
            try:
                geom = row.geometry
                lines = list(geom.geoms) if 'Multi' in geom.geom_type else [geom]
                for line in lines:
                    _lcoords = [(*(to_utm.transform(x, y)),) for x, y in list(line.coords)]
                    ring_poly = sg.LineString(_lcoords).buffer(_CUT_W, cap_style=2)
                    ring_pts = [(x - center_utm[0], y - center_utm[1])
                                for x, y in ring_poly.exterior.coords[:-1]]
                    base_z = local_z(*to_wgs84.transform(ring_poly.centroid.x, ring_poly.centroid.y))
                    n = len(ring_pts)
                    for k in range(n):
                        x0, y0 = ring_pts[k]
                        x1, y1 = ring_pts[(k+1) % n]
                        verts = np.array([[x0, y0, base_z],
                                          [x1, y1, base_z],
                                          [x1, y1, base_z + _CUT_H],
                                          [x0, y0, base_z + _CUT_H]], dtype=np.float32)
                        cut_f.extend([[cut_off, cut_off+1, cut_off+2],
                                      [cut_off, cut_off+2, cut_off+3]])
                        cut_v.append(verts); cut_off += 4
            except Exception: pass
        if cut_v:
            _cv = np.concatenate(cut_v).astype(np.float32)
            _cf = np.array(cut_f, np.int32)
            _write_ply(_cv, _cf, os.path.join(MESH_DIR, 'infra_itu_concrete_road_cuttings.ply'))
            mat_plys.setdefault('itu_concrete', []).append(
                (os.path.basename(MESH_DIR)+'/infra_itu_concrete_road_cuttings.ply', 'infrastructure'))
            print(f'  Road cuttings: {len(_cv):,} verts, {len(_cf):,} faces → infra_itu_concrete_road_cuttings.ply')
        else:
            print('  Road cuttings: none found (highway + cutting=yes) — check OSM data for bbox')
    except Exception as _e:
        print(f'  Road cuttings download failed: {_e}')
else:
    print('Road cuttings: skipped (INCLUDE_ROAD_CUTTINGS=False)')

# ── Highway bridge decks (highway + bridge=yes) → itu_concrete PLY ──────────
# Motorway flyovers, trunk road bridges — deck slab as flat concrete panel
# Width derived from highway type; clearance 5 m above terrain
if globals().get('INCLUDE_HWY_BRIDGES', True):
    print('\nDownloading OSM highway bridges ...')
    try:
        _hbr_tags = {'highway': ['motorway','trunk','primary','secondary','tertiary']}
        if _ox_version >= (2, 0):
            gdf_hbr = ox.features_from_bbox(
                bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH), tags=_hbr_tags)
        else:
            gdf_hbr = ox.features_from_bbox(
                bbox=(SCENE_NORTH, SCENE_SOUTH, SCENE_EAST, SCENE_WEST), tags=_hbr_tags)
        if 'bridge' in gdf_hbr.columns:
            _hbr_mask = gdf_hbr['bridge'].fillna('').isin(['yes','viaduct','movable'])
        else:
            _hbr_mask = gdf_hbr.index.map(lambda _: False)
        _hbr_lines = gdf_hbr[_hbr_mask & gdf_hbr.geometry.geom_type.isin(['LineString','MultiLineString'])]
        print(f'  {len(_hbr_lines)} highway bridge ways')
        hbr_v, hbr_f, hbr_off = [], [], 0
        _HBR_WIDTHS = {'motorway': 15.0, 'trunk': 12.0, 'primary': 9.0,
                       'secondary': 7.0, 'tertiary': 6.0}
        _DECK_THICKNESS = 1.5
        _DECK_CLEARANCE = 5.0
        _hbr_count = 0
        for _, row in _hbr_lines.iterrows():
            try:
                hw_type = str(row.get('highway', 'primary') or 'primary').lower().split('_')[0]
                half_w = _HBR_WIDTHS.get(hw_type, 8.0) / 2.0
                geom = row.geometry
                lines = list(geom.geoms) if 'Multi' in geom.geom_type else [geom]
                for line in lines:
                    _lcoords = [(*(to_utm.transform(x, y)),) for x, y in list(line.coords)]
                    deck = sg.LineString(_lcoords).buffer(half_w, cap_style=2)
                    deck_coords = [(x - center_utm[0], y - center_utm[1])
                                   for x, y in deck.exterior.coords[:-1]]
                    base_z = local_z(*to_wgs84.transform(deck.centroid.x, deck.centroid.y))
                    res = _extrude_building(deck_coords, base_z + _DECK_CLEARANCE,
                                           _DECK_THICKNESS, roof_h=0.0, roof_shape='flat')
                    if res is None: continue
                    for _vv, _ff in zip(res[0::2], res[1::2]):
                        if _vv is None or len(_vv) == 0: continue
                        hbr_f.extend([[fi + hbr_off for fi in face] for face in _ff])
                        hbr_v.append(_vv); hbr_off += len(_vv)
                    _hbr_count += 1
            except Exception: pass
        if hbr_v:
            _hv = np.concatenate(hbr_v).astype(np.float32)
            _hf = np.array(hbr_f, np.int32)
            _write_ply(_hv, _hf, os.path.join(MESH_DIR, 'infra_itu_concrete_hwy_bridges.ply'))
            mat_plys.setdefault('itu_concrete', []).append(
                (os.path.basename(MESH_DIR)+'/infra_itu_concrete_hwy_bridges.ply', 'infrastructure'))
            print(f'  Hwy bridges: {_hbr_count} decks → infra_itu_concrete_hwy_bridges.ply ({len(_hv):,} verts, {len(_hf):,} faces)')
        else:
            print('  Highway bridges: none found (highway + bridge=yes) — check OSM data for bbox')
    except Exception as _e:
        print(f'  Highway bridges download failed: {_e}')
else:
    print('Highway bridges: skipped (INCLUDE_HWY_BRIDGES=False)')

# ── Railway tracks → itu_metal PLY → itu_metal PLY ───────────────────────────────────────────
# OSM railway=rail/light_rail/tram lines → flat ribbon mesh at terrain height
# Steel rails are strong specular reflectors at 915 MHz (εr≈1, σ=1e7 S/m)
# Effect: 2-4 dB signal enhancement along tram/rail corridors (Lienard 1997)
if globals().get('INCLUDE_RAILWAYS', True):
    print('\nDownloading OSM railway tracks ...')
    try:
        _rail_tags = {'railway': ['rail', 'light_rail', 'tram', 'subway', 'narrow_gauge']}
        if _ox_version >= (2, 0):
            gdf_rail = ox.features_from_bbox(
                bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH), tags=_rail_tags)
        elif _ox_version >= (1, 3):
            gdf_rail = ox.features_from_bbox(
                bbox=(SCENE_NORTH, SCENE_SOUTH, SCENE_EAST, SCENE_WEST), tags=_rail_tags)
        else:
            gdf_rail = ox.features_from_bbox(
                north=SCENE_NORTH, south=SCENE_SOUTH,
                east=SCENE_EAST,   west=SCENE_WEST, tags=_rail_tags)
        _RAIL_W = 1.5    # track half-width (m) — standard gauge 1435 mm
        rv_all, rf_all, rv_off = [], [], 0
        for _, rrow in gdf_rail.iterrows():
            geom = rrow.geometry
            if geom is None or geom.is_empty: continue
            if globals().get('EXCLUDE_TUNNELS', True) and \
               str(rrow.get('tunnel', '')).lower() in ('yes', 'building_passage', 'subway'):
                continue
            lines_geom = []
            if geom.geom_type == 'LineString':
                lines_geom = [geom]
            elif geom.geom_type == 'MultiLineString':
                lines_geom = list(geom.geoms)
            for lg in lines_geom:
                coords = list(lg.coords)
                if len(coords) < 2: continue
                pts_utm = []
                for lon, lat in coords:
                    rx_, ry_ = to_utm.transform(lon, lat)
                    pts_utm.append((rx_ - center_utm[0], ry_ - center_utm[1]))
                # Build ribbon along track — one quad per segment
                for k in range(len(pts_utm) - 1):
                    x0, y0 = pts_utm[k]
                    x1, y1 = pts_utm[k + 1]
                    dx, dy = x1 - x0, y1 - y0
                    seg_len = (dx**2 + dy**2)**0.5
                    if seg_len < 0.1: continue
                    nx, ny = -dy / seg_len * _RAIL_W, dx / seg_len * _RAIL_W
                    try:
                        z0 = 0.0 if FLAT_TERRAIN else float(local_z(
                            *to_wgs84.transform(x0 + center_utm[0], y0 + center_utm[1])))
                        z1 = 0.0 if FLAT_TERRAIN else float(local_z(
                            *to_wgs84.transform(x1 + center_utm[0], y1 + center_utm[1])))
                    except Exception:
                        z0 = z1 = 0.0
                    quad = np.array([
                        [x0 - nx, y0 - ny, z0],
                        [x0 + nx, y0 + ny, z0],
                        [x1 + nx, y1 + ny, z1],
                        [x1 - nx, y1 - ny, z1],
                    ], dtype=np.float32)
                    rv_all.append(quad)
                    rf_all.append([rv_off, rv_off + 1, rv_off + 2])
                    rf_all.append([rv_off, rv_off + 2, rv_off + 3])
                    rv_off += 4
        if rv_all:
            rv = np.concatenate(rv_all).astype(np.float32)
            rf = np.array(rf_all, np.int32)
            _write_ply(rv, rf, os.path.join(MESH_DIR, 'rail_itu_metal.ply'))
            mat_plys.setdefault('itu_metal', []).append(
                ('meshes/' + 'rail_itu_metal.ply', 'railways'))
            print(f'  Railways: {len(rv):,} verts, {len(rf):,} faces → rail_itu_metal.ply')
        else:
            print('  Railways: no geometry found')
    except Exception as _e:
        print(f'  Railways download failed: {_e}')
else:
    print('Railways: skipped (INCLUDE_RAILWAYS=False)')

# ── Barriers (walls / fences / noise barriers) → PLY ─────────────────────────
# OSM barrier=wall/noise_barrier → itu_concrete  (brick walls, noise screens)
#     barrier=fence/railing      → itu_metal     (metal fence panels)
# Effect: 1-8 dB shadow behind wall depending on height and frequency
# Noise barriers along major roads are 3-5 m concrete → 5-8 dB NLOS attenuation
if globals().get('INCLUDE_BARRIERS', True):
    print('\nDownloading OSM barriers ...')
    try:
        _bar_tags = {'barrier': ['wall', 'fence', 'noise_barrier', 'retaining_wall',
                                  'guard_rail', 'railing', 'jersey_barrier']}
        if _ox_version >= (2, 0):
            gdf_bar = ox.features_from_bbox(
                bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH), tags=_bar_tags)
        elif _ox_version >= (1, 3):
            gdf_bar = ox.features_from_bbox(
                bbox=(SCENE_NORTH, SCENE_SOUTH, SCENE_EAST, SCENE_WEST), tags=_bar_tags)
        else:
            gdf_bar = ox.features_from_bbox(
                north=SCENE_NORTH, south=SCENE_SOUTH,
                east=SCENE_EAST,   west=SCENE_WEST, tags=_bar_tags)
        # Default heights per barrier type (m)
        _BAR_HEIGHTS = {
            'wall': 2.0, 'noise_barrier': 4.0, 'retaining_wall': 2.5,
            'fence': 1.8, 'railing': 1.2, 'guard_rail': 0.8, 'jersey_barrier': 1.0,
        }
        # Material per barrier type
        _BAR_MAT = {
            'wall': 'itu_concrete', 'noise_barrier': 'itu_concrete',
            'retaining_wall': 'itu_concrete', 'jersey_barrier': 'itu_concrete',
            'fence': 'itu_metal', 'railing': 'itu_metal', 'guard_rail': 'itu_metal',
        }
        _bar_min_len = float(globals().get('BARRIER_MIN_LEN_M', 10.0))
        bar_geom = {}   # mat → (verts_list, faces_list, offset)
        _bar_count = 0
        for _, brow in gdf_bar.iterrows():
            geom = brow.geometry
            if geom is None or geom.is_empty: continue
            b_type = str(brow.get('barrier', 'wall') or 'wall').lower()
            b_mat  = _BAR_MAT.get(b_type, 'itu_concrete')
            b_h    = _BAR_HEIGHTS.get(b_type, 2.0)
            # Override height from OSM height= tag if present
            _ht = str(brow.get('height', '') or '')
            if _ht and _ht not in ('nan', 'None', ''):
                try: b_h = float(_ht.replace('m','').strip())
                except ValueError: pass
            lines_geom = []
            if geom.geom_type == 'LineString':     lines_geom = [geom]
            elif geom.geom_type == 'MultiLineString': lines_geom = list(geom.geoms)
            elif geom.geom_type in ('Polygon','MultiPolygon'):
                # Barrier polygons → use exterior ring as wall line
                polys = [geom] if geom.geom_type == 'Polygon' else list(geom.geoms)
                for p in polys:
                    from shapely.geometry import LineString as _LS
                    lines_geom.append(_LS(p.exterior.coords))
            for lg in lines_geom:
                coords = list(lg.coords)
                if len(coords) < 2: continue
                pts_utm = []
                for lon, lat in coords:
                    bx_, by_ = to_utm.transform(lon, lat)
                    pts_utm.append((bx_ - center_utm[0], by_ - center_utm[1]))
                # Check total length — skip trivially short segments
                _seg_len_total = sum(
                    ((pts_utm[k+1][0]-pts_utm[k][0])**2 +
                     (pts_utm[k+1][1]-pts_utm[k][1])**2)**0.5
                    for k in range(len(pts_utm)-1))
                if _seg_len_total < _bar_min_len: continue
                if b_mat not in bar_geom:
                    bar_geom[b_mat] = ([], [], 0)
                bv_l, bf_l, b_off = bar_geom[b_mat]
                for k in range(len(pts_utm) - 1):
                    x0, y0 = pts_utm[k]
                    x1, y1 = pts_utm[k + 1]
                    try:
                        z0 = 0.0 if FLAT_TERRAIN else float(local_z(
                            *to_wgs84.transform(x0+center_utm[0], y0+center_utm[1])))
                        z1 = 0.0 if FLAT_TERRAIN else float(local_z(
                            *to_wgs84.transform(x1+center_utm[0], y1+center_utm[1])))
                    except Exception:
                        z0 = z1 = 0.0
                    # Vertical quad: bottom-left, bottom-right, top-right, top-left
                    quad = np.array([
                        [x0, y0, z0],
                        [x1, y1, z1],
                        [x1, y1, z1 + b_h],
                        [x0, y0, z0 + b_h],
                    ], dtype=np.float32)
                    bv_l.append(quad)
                    bf_l.append([b_off, b_off + 1, b_off + 2])
                    bf_l.append([b_off, b_off + 2, b_off + 3])
                    b_off += 4
                bar_geom[b_mat] = (bv_l, bf_l, b_off)
                _bar_count += 1
        if bar_geom:
            for b_mat, (bv_l, bf_l, _) in bar_geom.items():
                if not bv_l: continue
                bv = np.concatenate(bv_l).astype(np.float32)
                bf = np.array(bf_l, np.int32)
                _mat_tag = b_mat.replace('itu_', '')
                _bar_ply = f'barriers_{b_mat}.ply'
                _write_ply(bv, bf, os.path.join(MESH_DIR, _bar_ply))
                mat_plys.setdefault(b_mat, []).append(
                    ('meshes/' + _bar_ply, 'barriers'))
                print(f'  Barriers ({_mat_tag}): {len(bv):,} verts, {len(bf):,} faces → {_bar_ply}')
            print(f'  Barriers total: {_bar_count} segments processed')
        else:
            print('  Barriers: no geometry found')
    except Exception as _e:
        print(f'  Barriers download failed: {_e}')
else:
    print('Barriers: skipped (INCLUDE_BARRIERS=False)')


# ── Power pylons / transmission towers → itu_metal PLY ───────────────────────
# Lattice steel towers are dominant metallic diffractors at 915 MHz [DeE04].
# Approximated as solid box of equivalent projected cross-section.
if globals().get('INCLUDE_PYLONS', False):
    print('\nDownloading OSM power pylons ...')
    try:
        _pyl_tags = {'power': ['tower', 'pole']}
        if _ox_version >= (2, 0):
            gdf_pyl = ox.features_from_bbox(bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH), tags=_pyl_tags)
        elif _ox_version >= (1, 3):
            gdf_pyl = ox.features_from_bbox(bbox=(SCENE_NORTH, SCENE_SOUTH, SCENE_EAST, SCENE_WEST), tags=_pyl_tags)
        else:
            gdf_pyl = ox.features_from_bbox(north=SCENE_NORTH, south=SCENE_SOUTH, east=SCENE_EAST, west=SCENE_WEST, tags=_pyl_tags)
        _pyl_v, _pyl_f, _pyl_off = [], [], 0
        _pyl_count = 0
        _PYL_H = {'tower': 30.0, 'pole': 8.0}
        _PYL_W = {'tower':  3.0, 'pole': 0.4}
        for _, prow in gdf_pyl.iterrows():
            geom = prow.geometry
            if geom is None or geom.is_empty: continue
            cx, cy = (geom.x, geom.y) if geom.geom_type == 'Point' else (geom.centroid.x, geom.centroid.y)
            px, py = to_utm.transform(cx, cy)
            lx = px - center_utm[0]; ly = py - center_utm[1]
            base_z = local_z(cx, cy)
            ptype = str(prow.get('power', 'tower') or 'tower').lower()
            _ht = _PYL_H.get(ptype, 25.0)
            _ht_tag = str(prow.get('height', '') or '')
            if _ht_tag not in ('nan', 'None', ''):
                try: _ht = float(_ht_tag.replace('m', '').strip())
                except ValueError: pass
            hw = _PYL_W.get(ptype, 2.0) / 2.0
            sq = [(lx-hw, ly-hw), (lx+hw, ly-hw), (lx+hw, ly+hw), (lx-hw, ly+hw)]
            res = _extrude_building(sq, base_z, _ht, roof_h=0.0, roof_shape='flat')
            if res is None: continue
            _wv, _wf, _rv, _rf = res
            for _vv, _ff in ((_wv, _wf), (_rv, _rf)):
                if _vv is None or len(_vv) == 0 or _ff is None or len(_ff) == 0: continue
                _pyl_f_s = [[fi + _pyl_off for fi in face] for face in _ff]
                _pyl_v.append(_vv); _pyl_f.extend(_pyl_f_s); _pyl_off += len(_vv)
            _pyl_count += 1
        if _pyl_v:
            _pv = np.concatenate(_pyl_v).astype(np.float32)
            _pf = np.array(_pyl_f, np.int32)
            _ply_name = 'infra_itu_metal_pylons.ply'
            _write_ply(_pv, _pf, os.path.join(MESH_DIR, _ply_name))
            mat_plys.setdefault('itu_metal', []).append(('meshes/' + _ply_name, 'infrastructure'))
            print(f'  Pylons: {_pyl_count} → {_ply_name} ({len(_pv):,} verts, {len(_pf):,} faces)')
        else:
            print('  Pylons: none found in bbox')
    except Exception as _e:
        print(f'  Pylons: download failed — {_e}')
else:
    print('Pylons: skipped (INCLUDE_PYLONS=False)')

# ── Telecom masts / broadcast towers → itu_metal PLY ─────────────────────────
# Tall slender metal masts are strong vertical diffraction edges at 915 MHz.
if globals().get('INCLUDE_MASTS', False):
    print('\nDownloading OSM telecom masts ...')
    try:
        _mst_tags = {'man_made': ['mast', 'communications_tower', 'tower']}
        if _ox_version >= (2, 0):
            gdf_mst = ox.features_from_bbox(bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH), tags=_mst_tags)
        elif _ox_version >= (1, 3):
            gdf_mst = ox.features_from_bbox(bbox=(SCENE_NORTH, SCENE_SOUTH, SCENE_EAST, SCENE_WEST), tags=_mst_tags)
        else:
            gdf_mst = ox.features_from_bbox(north=SCENE_NORTH, south=SCENE_SOUTH, east=SCENE_EAST, west=SCENE_WEST, tags=_mst_tags)
        _mst_v, _mst_f, _mst_off = [], [], 0
        _mst_count = 0
        for _, mrow in gdf_mst.iterrows():
            geom = mrow.geometry
            if geom is None or geom.is_empty: continue
            cx, cy = (geom.x, geom.y) if geom.geom_type == 'Point' else (geom.centroid.x, geom.centroid.y)
            px, py = to_utm.transform(cx, cy)
            lx = px - center_utm[0]; ly = py - center_utm[1]
            base_z = local_z(cx, cy)
            _ht = 30.0
            _ht_tag = str(mrow.get('height', '') or '')
            if _ht_tag not in ('nan', 'None', ''):
                try: _ht = float(_ht_tag.replace('m', '').strip())
                except ValueError: pass
            hw = 1.0  # 2m width column
            sq = [(lx-hw, ly-hw), (lx+hw, ly-hw), (lx+hw, ly+hw), (lx-hw, ly+hw)]
            res = _extrude_building(sq, base_z, _ht, roof_h=0.0, roof_shape='flat')
            if res is None: continue
            _wv, _wf, _rv, _rf = res
            for _vv, _ff in ((_wv, _wf), (_rv, _rf)):
                if _vv is None or len(_vv) == 0 or _ff is None or len(_ff) == 0: continue
                _mst_f_s = [[fi + _mst_off for fi in face] for face in _ff]
                _mst_v.append(_vv); _mst_f.extend(_mst_f_s); _mst_off += len(_vv)
            _mst_count += 1
        if _mst_v:
            _mv = np.concatenate(_mst_v).astype(np.float32)
            _mf = np.array(_mst_f, np.int32)
            _ply_name = 'infra_itu_metal_masts.ply'
            _write_ply(_mv, _mf, os.path.join(MESH_DIR, _ply_name))
            mat_plys.setdefault('itu_metal', []).append(('meshes/' + _ply_name, 'infrastructure'))
            print(f'  Masts: {_mst_count} → {_ply_name} ({len(_mv):,} verts, {len(_mf):,} faces)')
        else:
            print('  Masts: none found in bbox')
    except Exception as _e:
        print(f'  Masts: download failed — {_e}')
else:
    print('Masts: skipped (INCLUDE_MASTS=False)')

# ── Industrial chimneys → itu_concrete PLY ───────────────────────────────────
# Circular concrete stacks are significant vertical diffractors.
if globals().get('INCLUDE_CHIMNEYS', False):
    print('\nDownloading OSM chimneys ...')
    try:
        _chi_tags = {'man_made': 'chimney'}
        if _ox_version >= (2, 0):
            gdf_chi = ox.features_from_bbox(bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH), tags=_chi_tags)
        elif _ox_version >= (1, 3):
            gdf_chi = ox.features_from_bbox(bbox=(SCENE_NORTH, SCENE_SOUTH, SCENE_EAST, SCENE_WEST), tags=_chi_tags)
        else:
            gdf_chi = ox.features_from_bbox(north=SCENE_NORTH, south=SCENE_SOUTH, east=SCENE_EAST, west=SCENE_WEST, tags=_chi_tags)
        _chi_v, _chi_f, _chi_off = [], [], 0
        _chi_count = 0
        for _, crow in gdf_chi.iterrows():
            geom = crow.geometry
            if geom is None or geom.is_empty: continue
            cx, cy = (geom.x, geom.y) if geom.geom_type == 'Point' else (geom.centroid.x, geom.centroid.y)
            px, py = to_utm.transform(cx, cy)
            lx = px - center_utm[0]; ly = py - center_utm[1]
            base_z = local_z(cx, cy)
            _ht = 25.0
            _ht_tag = str(crow.get('height', '') or '')
            if _ht_tag not in ('nan', 'None', ''):
                try: _ht = float(_ht_tag.replace('m', '').strip())
                except ValueError: pass
            hw = 1.5  # 3m diameter approximation
            sq = [(lx-hw, ly-hw), (lx+hw, ly-hw), (lx+hw, ly+hw), (lx-hw, ly+hw)]
            res = _extrude_building(sq, base_z, _ht, roof_h=0.0, roof_shape='flat')
            if res is None: continue
            _wv, _wf, _rv, _rf = res
            for _vv, _ff in ((_wv, _wf), (_rv, _rf)):
                if _vv is None or len(_vv) == 0 or _ff is None or len(_ff) == 0: continue
                _chi_f_s = [[fi + _chi_off for fi in face] for face in _ff]
                _chi_v.append(_vv); _chi_f.extend(_chi_f_s); _chi_off += len(_vv)
            _chi_count += 1
        if _chi_v:
            _cv = np.concatenate(_chi_v).astype(np.float32)
            _cf = np.array(_chi_f, np.int32)
            _ply_name = 'infra_itu_concrete_chimneys.ply'
            _write_ply(_cv, _cf, os.path.join(MESH_DIR, _ply_name))
            mat_plys.setdefault('itu_concrete', []).append(('meshes/' + _ply_name, 'infrastructure'))
            print(f'  Chimneys: {_chi_count} → {_ply_name} ({len(_cv):,} verts, {len(_cf):,} faces)')
        else:
            print('  Chimneys: none found in bbox')
    except Exception as _e:
        print(f'  Chimneys: download failed — {_e}')
else:
    print('Chimneys: skipped (INCLUDE_CHIMNEYS=False)')

# ── Water towers → itu_metal PLY ─────────────────────────────────────────────
if globals().get('INCLUDE_WATER_TOWERS', False):
    print('\nDownloading OSM water towers ...')
    try:
        _wt_tags = {'man_made': 'water_tower'}
        if _ox_version >= (2, 0):
            gdf_wt = ox.features_from_bbox(bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH), tags=_wt_tags)
        elif _ox_version >= (1, 3):
            gdf_wt = ox.features_from_bbox(bbox=(SCENE_NORTH, SCENE_SOUTH, SCENE_EAST, SCENE_WEST), tags=_wt_tags)
        else:
            gdf_wt = ox.features_from_bbox(north=SCENE_NORTH, south=SCENE_SOUTH, east=SCENE_EAST, west=SCENE_WEST, tags=_wt_tags)
        _wt_v, _wt_f, _wt_off = [], [], 0
        _wt_count = 0
        for _, wrow in gdf_wt.iterrows():
            geom = wrow.geometry
            if geom is None or geom.is_empty: continue
            cx, cy = (geom.x, geom.y) if geom.geom_type == 'Point' else (geom.centroid.x, geom.centroid.y)
            px, py = to_utm.transform(cx, cy)
            lx = px - center_utm[0]; ly = py - center_utm[1]
            base_z = local_z(cx, cy)
            _ht = 15.0
            _ht_tag = str(wrow.get('height', '') or '')
            if _ht_tag not in ('nan', 'None', ''):
                try: _ht = float(_ht_tag.replace('m', '').strip())
                except ValueError: pass
            hw = 2.5  # 5m diameter approximation
            sq = [(lx-hw, ly-hw), (lx+hw, ly-hw), (lx+hw, ly+hw), (lx-hw, ly+hw)]
            res = _extrude_building(sq, base_z, _ht, roof_h=0.0, roof_shape='flat')
            if res is None: continue
            _wv, _wf, _rv, _rf = res
            for _vv, _ff in ((_wv, _wf), (_rv, _rf)):
                if _vv is None or len(_vv) == 0 or _ff is None or len(_ff) == 0: continue
                _wt_f_s = [[fi + _wt_off for fi in face] for face in _ff]
                _wt_v.append(_vv); _wt_f.extend(_wt_f_s); _wt_off += len(_vv)
            _wt_count += 1
        if _wt_v:
            _wtv = np.concatenate(_wt_v).astype(np.float32)
            _wtf = np.array(_wt_f, np.int32)
            _ply_name = 'infra_itu_metal_watertowers.ply'
            _write_ply(_wtv, _wtf, os.path.join(MESH_DIR, _ply_name))
            mat_plys.setdefault('itu_metal', []).append(('meshes/' + _ply_name, 'infrastructure'))
            print(f'  Water towers: {_wt_count} → {_ply_name} ({len(_wtv):,} verts, {len(_wtf):,} faces)')
        else:
            print('  Water towers: none found in bbox')
    except Exception as _e:
        print(f'  Water towers: download failed — {_e}')
else:
    print('Water towers: skipped (INCLUDE_WATER_TOWERS=False)')

# ── Storage tanks → itu_metal PLY ────────────────────────────────────────────
# Curved cylindrical metal surfaces produce strong backscatter at sub-GHz [DeE04 §IV].
if globals().get('INCLUDE_STORAGE_TANKS', False):
    print('\nDownloading OSM storage tanks ...')
    try:
        _stk_tags = {'man_made': ['storage_tank', 'silo']}
        if _ox_version >= (2, 0):
            gdf_stk = ox.features_from_bbox(bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH), tags=_stk_tags)
        elif _ox_version >= (1, 3):
            gdf_stk = ox.features_from_bbox(bbox=(SCENE_NORTH, SCENE_SOUTH, SCENE_EAST, SCENE_WEST), tags=_stk_tags)
        else:
            gdf_stk = ox.features_from_bbox(north=SCENE_NORTH, south=SCENE_SOUTH, east=SCENE_EAST, west=SCENE_WEST, tags=_stk_tags)
        _stk_v, _stk_f, _stk_off = [], [], 0
        _stk_count = 0
        for _, srow in gdf_stk.iterrows():
            geom = srow.geometry
            if geom is None or geom.is_empty: continue
            _ht = 8.0
            _ht_tag = str(srow.get('height', '') or '')
            if _ht_tag not in ('nan', 'None', ''):
                try: _ht = float(_ht_tag.replace('m', '').strip())
                except ValueError: pass
            if geom.geom_type in ('Polygon', 'MultiPolygon'):
                polys = geom.geoms if geom.geom_type == 'MultiPolygon' else [geom]
                for poly in polys:
                    coords_utm = []
                    for lon, lat in poly.exterior.coords[:-1]:
                        ex, ny = to_utm.transform(lon, lat)
                        coords_utm.append((ex - center_utm[0], ny - center_utm[1]))
                    base_z = local_z(poly.centroid.x, poly.centroid.y)
                    res = _extrude_building(coords_utm, base_z, _ht, roof_h=0.0, roof_shape='flat')
                    if res is None: continue
                    _wv, _wf, _rv, _rf = res
                    for _vv, _ff in ((_wv, _wf), (_rv, _rf)):
                        if _vv is None or len(_vv) == 0 or _ff is None or len(_ff) == 0: continue
                        _stk_f_s = [[fi + _stk_off for fi in face] for face in _ff]
                        _stk_v.append(_vv); _stk_f.extend(_stk_f_s); _stk_off += len(_vv)
                    _stk_count += 1
            elif geom.geom_type == 'Point':
                px, py = to_utm.transform(geom.x, geom.y)
                lx = px - center_utm[0]; ly = py - center_utm[1]
                base_z = local_z(geom.x, geom.y)
                hw = 5.0
                sq = [(lx-hw, ly-hw), (lx+hw, ly-hw), (lx+hw, ly+hw), (lx-hw, ly+hw)]
                res = _extrude_building(sq, base_z, _ht, roof_h=0.0, roof_shape='flat')
                if res is None: continue
                _wv, _wf, _rv, _rf = res
                for _vv, _ff in ((_wv, _wf), (_rv, _rf)):
                    if _vv is None or len(_vv) == 0 or _ff is None or len(_ff) == 0: continue
                    _stk_f_s = [[fi + _stk_off for fi in face] for face in _ff]
                    _stk_v.append(_vv); _stk_f.extend(_stk_f_s); _stk_off += len(_vv)
                _stk_count += 1
        if _stk_v:
            _sv = np.concatenate(_stk_v).astype(np.float32)
            _sf = np.array(_stk_f, np.int32)
            _ply_name = 'infra_itu_metal_tanks.ply'
            _write_ply(_sv, _sf, os.path.join(MESH_DIR, _ply_name))
            mat_plys.setdefault('itu_metal', []).append(('meshes/' + _ply_name, 'infrastructure'))
            print(f'  Storage tanks: {_stk_count} → {_ply_name} ({len(_sv):,} verts, {len(_sf):,} faces)')
        else:
            print('  Storage tanks: none found in bbox')
    except Exception as _e:
        print(f'  Storage tanks: download failed — {_e}')
else:
    print('Storage tanks: skipped (INCLUDE_STORAGE_TANKS=False)')

# ── Stadiums → itu_metal PLY ──────────────────────────────────────────────────
# Large metal-roofed sports venues are significant reflectors [Xia24].
# Only include leisure=stadium features NOT already captured by building export.
if globals().get('INCLUDE_STADIUMS', False):
    print('\nDownloading OSM stadiums ...')
    try:
        _std_tags = {'leisure': 'stadium'}
        if _ox_version >= (2, 0):
            gdf_std = ox.features_from_bbox(bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH), tags=_std_tags)
        elif _ox_version >= (1, 3):
            gdf_std = ox.features_from_bbox(bbox=(SCENE_NORTH, SCENE_SOUTH, SCENE_EAST, SCENE_WEST), tags=_std_tags)
        else:
            gdf_std = ox.features_from_bbox(north=SCENE_NORTH, south=SCENE_SOUTH, east=SCENE_EAST, west=SCENE_WEST, tags=_std_tags)
        _std_v, _std_f, _std_off = [], [], 0
        _std_count = 0
        for _, srow in gdf_std.iterrows():
            geom = srow.geometry
            if geom is None or geom.is_empty: continue
            if str(srow.get('building', '') or '').lower() in ('yes','stadium','sports_hall'):
                continue  # already in building export
            if geom.geom_type not in ('Polygon', 'MultiPolygon'): continue
            _ht = 15.0  # typical stand roof height
            _ht_tag = str(srow.get('height', '') or '')
            if _ht_tag not in ('nan', 'None', ''):
                try: _ht = float(_ht_tag.replace('m', '').strip())
                except ValueError: pass
            polys = geom.geoms if geom.geom_type == 'MultiPolygon' else [geom]
            for poly in polys:
                coords_utm = []
                for lon, lat in poly.exterior.coords[:-1]:
                    ex, ny = to_utm.transform(lon, lat)
                    coords_utm.append((ex - center_utm[0], ny - center_utm[1]))
                base_z = local_z(poly.centroid.x, poly.centroid.y)
                res = _extrude_building(coords_utm, base_z, _ht, roof_h=0.0, roof_shape='flat')
                if res is None: continue
                _wv, _wf, _rv, _rf = res
                for _vv, _ff in ((_wv, _wf), (_rv, _rf)):
                    if _vv is None or len(_vv) == 0 or _ff is None or len(_ff) == 0: continue
                    _std_f_s = [[fi + _std_off for fi in face] for face in _ff]
                    _std_v.append(_vv); _std_f.extend(_std_f_s); _std_off += len(_vv)
                _std_count += 1
        if _std_v:
            _stv = np.concatenate(_std_v).astype(np.float32)
            _stf = np.array(_std_f, np.int32)
            _ply_name = 'infra_itu_metal_stadiums.ply'
            _write_ply(_stv, _stf, os.path.join(MESH_DIR, _ply_name))
            mat_plys.setdefault('itu_metal', []).append(('meshes/' + _ply_name, 'infrastructure'))
            print(f'  Stadiums: {_std_count} → {_ply_name} ({len(_stv):,} verts, {len(_stf):,} faces)')
        else:
            print('  Stadiums: none found in bbox (all may already be in building export)')
    except Exception as _e:
        print(f'  Stadiums: download failed — {_e}')
else:
    print('Stadiums: skipped (INCLUDE_STADIUMS=False)')

# ── Power substations → itu_metal PLY ────────────────────────────────────────
# Fenced transformer enclosures with metallic equipment inside.
if globals().get('INCLUDE_SUBSTATIONS', False):
    print('\nDownloading OSM power substations ...')
    try:
        _sub_tags = {'power': 'substation'}
        if _ox_version >= (2, 0):
            gdf_sub = ox.features_from_bbox(bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH), tags=_sub_tags)
        elif _ox_version >= (1, 3):
            gdf_sub = ox.features_from_bbox(bbox=(SCENE_NORTH, SCENE_SOUTH, SCENE_EAST, SCENE_WEST), tags=_sub_tags)
        else:
            gdf_sub = ox.features_from_bbox(north=SCENE_NORTH, south=SCENE_SOUTH, east=SCENE_EAST, west=SCENE_WEST, tags=_sub_tags)
        _sub_v, _sub_f, _sub_off = [], [], 0
        _sub_count = 0
        for _, subrow in gdf_sub.iterrows():
            geom = subrow.geometry
            if geom is None or geom.is_empty: continue
            if geom.geom_type not in ('Polygon', 'MultiPolygon'): continue
            _ht = 4.0  # substation fence/equipment height
            polys = geom.geoms if geom.geom_type == 'MultiPolygon' else [geom]
            for poly in polys:
                coords_utm = []
                for lon, lat in poly.exterior.coords[:-1]:
                    ex, ny = to_utm.transform(lon, lat)
                    coords_utm.append((ex - center_utm[0], ny - center_utm[1]))
                base_z = local_z(poly.centroid.x, poly.centroid.y)
                res = _extrude_building(coords_utm, base_z, _ht, roof_h=0.0, roof_shape='flat')
                if res is None: continue
                _wv, _wf, _rv, _rf = res
                for _vv, _ff in ((_wv, _wf), (_rv, _rf)):
                    if _vv is None or len(_vv) == 0 or _ff is None or len(_ff) == 0: continue
                    _sub_f_s = [[fi + _sub_off for fi in face] for face in _ff]
                    _sub_v.append(_vv); _sub_f.extend(_sub_f_s); _sub_off += len(_vv)
                _sub_count += 1
        if _sub_v:
            _sbv = np.concatenate(_sub_v).astype(np.float32)
            _sbf = np.array(_sub_f, np.int32)
            _ply_name = 'infra_itu_metal_substations.ply'
            _write_ply(_sbv, _sbf, os.path.join(MESH_DIR, _ply_name))
            mat_plys.setdefault('itu_metal', []).append(('meshes/' + _ply_name, 'infrastructure'))
            print(f'  Substations: {_sub_count} → {_ply_name} ({len(_sbv):,} verts, {len(_sbf):,} faces)')
        else:
            print('  Substations: none found in bbox')
    except Exception as _e:
        print(f'  Substations: download failed — {_e}')
else:
    print('Substations: skipped (INCLUDE_SUBSTATIONS=False)')


# ── Multi-storey car parks → itu_concrete PLY ────────────────────────────────
# Large open-floor concrete structures with different EM signature from solid buildings.
# Primary cause of −13 dB bias in 300–700m band along Nottingham city centre [GS22].
if globals().get('INCLUDE_CAR_PARKS', False):
    print('\nDownloading OSM multi-storey car parks ...')
    try:
        _cp_tags = {'building': ['parking', 'car_park', 'carpark']}
        if _ox_version >= (2, 0):
            gdf_cp = ox.features_from_bbox(bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH), tags=_cp_tags)
        elif _ox_version >= (1, 3):
            gdf_cp = ox.features_from_bbox(bbox=(SCENE_NORTH, SCENE_SOUTH, SCENE_EAST, SCENE_WEST), tags=_cp_tags)
        else:
            gdf_cp = ox.features_from_bbox(north=SCENE_NORTH, south=SCENE_SOUTH, east=SCENE_EAST, west=SCENE_WEST, tags=_cp_tags)
        _cp_v, _cp_f, _cp_off = [], [], 0
        _cp_count = 0
        for _, cprow in gdf_cp.iterrows():
            geom = cprow.geometry
            if geom is None or geom.is_empty: continue
            if geom.geom_type not in ('Polygon', 'MultiPolygon'): continue
            _ht = 12.0  # typical 4-level MSCP ~12m
            _ht_tag = str(cprow.get('height', '') or '')
            if _ht_tag not in ('nan', 'None', ''):
                try: _ht = float(_ht_tag.replace('m', '').strip())
                except ValueError: pass
            _lvls = str(cprow.get('building:levels', '') or '')
            if _lvls not in ('nan', 'None', ''):
                try: _ht = max(_ht, float(_lvls) * 3.0)
                except ValueError: pass
            polys = geom.geoms if geom.geom_type == 'MultiPolygon' else [geom]
            for poly in polys:
                coords_utm = []
                for lon, lat in poly.exterior.coords[:-1]:
                    ex, ny = to_utm.transform(lon, lat)
                    coords_utm.append((ex - center_utm[0], ny - center_utm[1]))
                base_z = local_z(poly.centroid.x, poly.centroid.y)
                res = _extrude_building(coords_utm, base_z, _ht, roof_h=0.0, roof_shape='flat')
                if res is None: continue
                _wv, _wf, _rv, _rf = res
                for _vv, _ff in ((_wv, _wf), (_rv, _rf)):
                    if _vv is None or len(_vv) == 0 or _ff is None or len(_ff) == 0: continue
                    _cp_f_s = [[fi + _cp_off for fi in face] for face in _ff]
                    _cp_v.append(_vv); _cp_f.extend(_cp_f_s); _cp_off += len(_vv)
                _cp_count += 1
        if _cp_v:
            _cpv = np.concatenate(_cp_v).astype(np.float32)
            _cpf = np.array(_cp_f, np.int32)
            _ply_name = 'infra_itu_concrete_carparks.ply'
            _write_ply(_cpv, _cpf, os.path.join(MESH_DIR, _ply_name))
            mat_plys.setdefault('itu_concrete', []).append(('meshes/' + _ply_name, 'infrastructure'))
            print(f'  Car parks: {_cp_count} → {_ply_name} ({len(_cpv):,} verts, {len(_cpf):,} faces)')
        else:
            print('  Car parks: none found in bbox')
    except Exception as _e:
        print(f'  Car parks: download failed — {_e}')
else:
    print('Car parks: skipped (INCLUDE_CAR_PARKS=False)')

# ── Cooling towers → itu_concrete PLY ────────────────────────────────────────
# Large cylindrical concrete structures — dominant long-range reflectors at sub-GHz.
# Ratcliffe-on-Soar power station is within 10 km south of Nottingham city centre.
if globals().get('INCLUDE_COOLING_TOWERS', False):
    print('\nDownloading OSM cooling towers ...')
    try:
        _ct_tags = {'man_made': 'cooling_tower'}
        if _ox_version >= (2, 0):
            gdf_ct = ox.features_from_bbox(bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH), tags=_ct_tags)
        elif _ox_version >= (1, 3):
            gdf_ct = ox.features_from_bbox(bbox=(SCENE_NORTH, SCENE_SOUTH, SCENE_EAST, SCENE_WEST), tags=_ct_tags)
        else:
            gdf_ct = ox.features_from_bbox(north=SCENE_NORTH, south=SCENE_SOUTH, east=SCENE_EAST, west=SCENE_WEST, tags=_ct_tags)
        _ct_v, _ct_f, _ct_off = [], [], 0
        _ct_count = 0
        for _, ctrow in gdf_ct.iterrows():
            geom = ctrow.geometry
            if geom is None or geom.is_empty: continue
            _ht = 60.0  # Ratcliffe towers ~114m; generic default 60m
            _ht_tag = str(ctrow.get('height', '') or '')
            if _ht_tag not in ('nan', 'None', ''):
                try: _ht = float(_ht_tag.replace('m', '').strip())
                except ValueError: pass
            if geom.geom_type in ('Polygon', 'MultiPolygon'):
                polys = geom.geoms if geom.geom_type == 'MultiPolygon' else [geom]
                for poly in polys:
                    coords_utm = []
                    for lon, lat in poly.exterior.coords[:-1]:
                        ex, ny = to_utm.transform(lon, lat)
                        coords_utm.append((ex - center_utm[0], ny - center_utm[1]))
                    base_z = local_z(poly.centroid.x, poly.centroid.y)
                    res = _extrude_building(coords_utm, base_z, _ht, roof_h=0.0, roof_shape='flat')
                    if res is None: continue
                    _wv, _wf, _rv, _rf = res
                    for _vv, _ff in ((_wv, _wf), (_rv, _rf)):
                        if _vv is None or len(_vv) == 0 or _ff is None or len(_ff) == 0: continue
                        _ct_f_s = [[fi + _ct_off for fi in face] for face in _ff]
                        _ct_v.append(_vv); _ct_f.extend(_ct_f_s); _ct_off += len(_vv)
                    _ct_count += 1
            elif geom.geom_type == 'Point':
                px, py = to_utm.transform(geom.x, geom.y)
                lx = px - center_utm[0]; ly = py - center_utm[1]
                base_z = local_z(geom.x, geom.y)
                hw = 10.0  # 20m diameter approx
                sq = [(lx-hw, ly-hw), (lx+hw, ly-hw), (lx+hw, ly+hw), (lx-hw, ly+hw)]
                res = _extrude_building(sq, base_z, _ht, roof_h=0.0, roof_shape='flat')
                if res is None: continue
                _wv, _wf, _rv, _rf = res
                for _vv, _ff in ((_wv, _wf), (_rv, _rf)):
                    if _vv is None or len(_vv) == 0 or _ff is None or len(_ff) == 0: continue
                    _ct_f_s = [[fi + _ct_off for fi in face] for face in _ff]
                    _ct_v.append(_vv); _ct_f.extend(_ct_f_s); _ct_off += len(_vv)
                _ct_count += 1
        if _ct_v:
            _ctv = np.concatenate(_ct_v).astype(np.float32)
            _ctf = np.array(_ct_f, np.int32)
            _ply_name = 'infra_itu_concrete_coolingtowers.ply'
            _write_ply(_ctv, _ctf, os.path.join(MESH_DIR, _ply_name))
            mat_plys.setdefault('itu_concrete', []).append(('meshes/' + _ply_name, 'infrastructure'))
            print(f'  Cooling towers: {_ct_count} → {_ply_name} ({len(_ctv):,} verts, {len(_ctf):,} faces)')
        else:
            print('  Cooling towers: none found in bbox')
    except Exception as _e:
        print(f'  Cooling towers: download failed — {_e}')
else:
    print('Cooling towers: skipped (INCLUDE_COOLING_TOWERS=False)')

# ── Fuel station canopies → itu_metal PLY ────────────────────────────────────
# amenity=fuel → flat metal canopy roof at +5m clearance, 0.3m thick slab
# ITU-R P.2040-2: itu_metal  er=1.0  σ=1e7  s=0.05  xpd=0.01  wt=0.005m
# Effect: strong specular reflector (metal roof) — visible at both 915 MHz & 3.6 GHz
if globals().get('INCLUDE_FUEL_CANOPIES', True):
    print('\nDownloading OSM fuel station canopies ...')
    try:
        _fuel_tags = {'amenity': 'fuel'}
        if _ox_version >= (2, 0):
            gdf_fuel = ox.features_from_bbox(
                bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH), tags=_fuel_tags)
        elif _ox_version >= (1, 3):
            gdf_fuel = ox.features_from_bbox(
                bbox=(SCENE_NORTH, SCENE_SOUTH, SCENE_EAST, SCENE_WEST), tags=_fuel_tags)
        else:
            gdf_fuel = ox.features_from_bbox(
                north=SCENE_NORTH, south=SCENE_SOUTH,
                east=SCENE_EAST,   west=SCENE_WEST, tags=_fuel_tags)
        _FUEL_CLEARANCE  = 5.0   # m — typical forecourt canopy clearance
        _FUEL_THICKNESS  = 0.3   # m — metal roof slab thickness
        _FUEL_BUF        = 12.0  # m — point feature buffer radius
        _fuel_v, _fuel_f, _fuel_off, _fuel_cnt = [], [], 0, 0
        for _, row in gdf_fuel.iterrows():
            try:
                geom = row.geometry
                if geom is None or geom.is_empty: continue
                if geom.geom_type == 'Point':
                    _px, _py = to_utm.transform(geom.x, geom.y)
                    import shapely.geometry as _sg2
                    geom = _sg2.Point(_px, _py).buffer(_FUEL_BUF)
                    _poly_list = [geom]
                elif geom.geom_type == 'Polygon':
                    _poly_list = [geom]
                    geom = None
                    for lon, lat in _poly_list[0].exterior.coords[:-1]:
                        pass
                    _pts = []
                    for lon, lat in _poly_list[0].exterior.coords[:-1]:
                        ex, ny = to_utm.transform(lon, lat)
                        _pts.append((ex, ny))
                    import shapely.geometry as _sg2
                    _poly_list = [_sg2.Polygon(_pts)]
                elif geom.geom_type == 'MultiPolygon':
                    _all_pts = []
                    for _sp in geom.geoms:
                        _pts = []
                        for lon, lat in _sp.exterior.coords[:-1]:
                            ex, ny = to_utm.transform(lon, lat)
                            _pts.append((ex, ny))
                        import shapely.geometry as _sg2
                        _all_pts.append(_sg2.Polygon(_pts))
                    _poly_list = _all_pts
                else:
                    continue
                for _poly in _poly_list:
                    _cen_lon, _cen_lat = to_wgs84.transform(
                        _poly.centroid.x, _poly.centroid.y)
                    _bz = 0.0 if FLAT_TERRAIN else float(local_z(_cen_lon, _cen_lat))
                    _coords_local = [(_p[0] - center_utm[0], _p[1] - center_utm[1])
                                     for _p in _poly.exterior.coords[:-1]]
                    if len(_coords_local) < 3: continue
                    _res = _extrude_building(_coords_local, _bz + _FUEL_CLEARANCE,
                                             _FUEL_THICKNESS, roof_h=0.0, roof_shape='flat')
                    if _res is None: continue
                    _wv, _wf, _rv, _rf = _res
                    for _vv, _ff in ((_wv, _wf), (_rv, _rf)):
                        if _vv is None or len(_vv) == 0: continue
                        _fuel_f.extend([[fi + _fuel_off for fi in face] for face in _ff])
                        _fuel_v.append(_vv); _fuel_off += len(_vv)
                    _fuel_cnt += 1
            except Exception: pass
        if _fuel_v:
            _fv = np.concatenate(_fuel_v).astype(np.float32)
            _ff2 = np.array(_fuel_f, np.int32)
            _write_ply(_fv, _ff2, os.path.join(MESH_DIR, 'infra_itu_metal_fuel_canopies.ply'))
            mat_plys.setdefault('itu_metal', []).append(
                ('meshes/infra_itu_metal_fuel_canopies.ply', 'fuel_canopies'))
            print(f'  Fuel canopies: {_fuel_cnt} → infra_itu_metal_fuel_canopies.ply '
                  f'({len(_fv):,} verts, {len(_ff2):,} faces)')
        else:
            print('  Fuel canopies: none found or no polygon geometry')
    except Exception as _e:
        print(f'  Fuel canopies download failed: {_e}')
else:
    print('Fuel canopies: skipped (INCLUDE_FUEL_CANOPIES=False)')

# ── Bus stations → itu_metal PLY ─────────────────────────────────────────────
# amenity=bus_station → extruded metal structure 6m high
# ITU-R P.2040-2: itu_metal  er=1.0  σ=1e7  s=0.05  xpd=0.01  wt=0.005m
# Effect: large covered roof in city centres — specular + diffraction edges
if globals().get('INCLUDE_BUS_STATIONS', True):
    print('\nDownloading OSM bus stations ...')
    try:
        _bus_tags = {'amenity': 'bus_station'}
        if _ox_version >= (2, 0):
            gdf_bus = ox.features_from_bbox(
                bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH), tags=_bus_tags)
        elif _ox_version >= (1, 3):
            gdf_bus = ox.features_from_bbox(
                bbox=(SCENE_NORTH, SCENE_SOUTH, SCENE_EAST, SCENE_WEST), tags=_bus_tags)
        else:
            gdf_bus = ox.features_from_bbox(
                north=SCENE_NORTH, south=SCENE_SOUTH,
                east=SCENE_EAST,   west=SCENE_WEST, tags=_bus_tags)
        _BUS_H   = 6.0   # m — typical bus station roof height
        _BUS_BUF = 15.0  # m — point feature buffer radius
        _bus_v, _bus_f, _bus_off, _bus_cnt = [], [], 0, 0
        for _, row in gdf_bus.iterrows():
            try:
                geom = row.geometry
                if geom is None or geom.is_empty: continue
                _ht = _BUS_H
                _ht_tag = str(row.get('height', '') or '')
                if _ht_tag not in ('nan', 'None', ''):
                    try: _ht = float(_ht_tag.replace('m', '').strip())
                    except ValueError: pass
                if geom.geom_type == 'Point':
                    _px, _py = to_utm.transform(geom.x, geom.y)
                    import shapely.geometry as _sg2
                    geom = _sg2.Point(_px, _py).buffer(_BUS_BUF)
                    _poly_list = [geom]
                elif geom.geom_type in ('Polygon', 'MultiPolygon'):
                    _poly_list = list(geom.geoms) if geom.geom_type == 'MultiPolygon' else [geom]
                    _utm_polys = []
                    for _sp in _poly_list:
                        _pts = []
                        for lon, lat in _sp.exterior.coords[:-1]:
                            ex, ny = to_utm.transform(lon, lat)
                            _pts.append((ex, ny))
                        import shapely.geometry as _sg2
                        _utm_polys.append(_sg2.Polygon(_pts))
                    _poly_list = _utm_polys
                else:
                    continue
                for _poly in _poly_list:
                    _cen_lon, _cen_lat = to_wgs84.transform(
                        _poly.centroid.x, _poly.centroid.y)
                    _bz = 0.0 if FLAT_TERRAIN else float(local_z(_cen_lon, _cen_lat))
                    _coords_local = [(_p[0] - center_utm[0], _p[1] - center_utm[1])
                                     for _p in _poly.exterior.coords[:-1]]
                    if len(_coords_local) < 3: continue
                    _res = _extrude_building(_coords_local, _bz, _ht,
                                             roof_h=0.0, roof_shape='flat')
                    if _res is None: continue
                    _wv, _wf, _rv, _rf = _res
                    for _vv, _ff in ((_wv, _wf), (_rv, _rf)):
                        if _vv is None or len(_vv) == 0: continue
                        _bus_f.extend([[fi + _bus_off for fi in face] for face in _ff])
                        _bus_v.append(_vv); _bus_off += len(_vv)
                    _bus_cnt += 1
            except Exception: pass
        if _bus_v:
            _bv = np.concatenate(_bus_v).astype(np.float32)
            _bf = np.array(_bus_f, np.int32)
            _write_ply(_bv, _bf, os.path.join(MESH_DIR, 'infra_itu_metal_bus_stations.ply'))
            mat_plys.setdefault('itu_metal', []).append(
                ('meshes/infra_itu_metal_bus_stations.ply', 'bus_stations'))
            print(f'  Bus stations: {_bus_cnt} → infra_itu_metal_bus_stations.ply '
                  f'({len(_bv):,} verts, {len(_bf):,} faces)')
        else:
            print('  Bus stations: none found or no polygon geometry')
    except Exception as _e:
        print(f'  Bus stations download failed: {_e}')
else:
    print('Bus stations: skipped (INCLUDE_BUS_STATIONS=False)')

# ── Surface car parks → itu_asphalt flat patch ───────────────────────────────
# amenity=parking (surface/open-air) → flat asphalt ground patch
# ITU-R P.2040-2: itu_asphalt  er=2.56  σ=0.005  s=0.30  xpd=0.15  wt=0.05m
# Effect: improves ground reflection accuracy — critical at 3.6 GHz (shorter Fresnel zone)
# Excludes multi-storey (parking=multi-storey) — already in INCLUDE_CAR_PARKS
if globals().get('INCLUDE_SURFACE_PARKS', True):
    print('\nDownloading OSM surface car parks ...')
    try:
        _spk_tags = {'amenity': 'parking'}
        if _ox_version >= (2, 0):
            gdf_spk = ox.features_from_bbox(
                bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH), tags=_spk_tags)
        elif _ox_version >= (1, 3):
            gdf_spk = ox.features_from_bbox(
                bbox=(SCENE_NORTH, SCENE_SOUTH, SCENE_EAST, SCENE_WEST), tags=_spk_tags)
        else:
            gdf_spk = ox.features_from_bbox(
                north=SCENE_NORTH, south=SCENE_SOUTH,
                east=SCENE_EAST,   west=SCENE_WEST, tags=_spk_tags)
        _MULTI_STOREY = {'multi-storey', 'multi_storey', 'multistorey',
                          'parking_garage', 'underground', 'rooftop'}
        _spk_v, _spk_f, _spk_off, _spk_cnt = [], [], 0, 0
        for _, row in gdf_spk.iterrows():
            try:
                geom = row.geometry
                if geom is None or geom.is_empty: continue
                _pk_type = str(row.get('parking', '') or '').lower()
                if _pk_type in _MULTI_STOREY: continue
                _bldg = str(row.get('building', '') or '').lower()
                if _bldg in ('parking', 'car_park', 'carpark', 'yes'): continue
                if geom.geom_type not in ('Polygon', 'MultiPolygon'): continue
                _poly_list = list(geom.geoms) if geom.geom_type == 'MultiPolygon' else [geom]
                for _poly in _poly_list:
                    _coords_utm = []
                    for lon, lat in _poly.exterior.coords[:-1]:
                        ex, ny = to_utm.transform(lon, lat)
                        _coords_utm.append((ex - center_utm[0], ny - center_utm[1]))
                    if len(_coords_utm) < 3: continue
                    _cen_lon, _cen_lat = to_wgs84.transform(
                        _poly.centroid.x, _poly.centroid.y)
                    _bz = 0.0 if FLAT_TERRAIN else float(local_z(_cen_lon, _cen_lat))
                    _n = len(_coords_utm)
                    _verts = np.array([[x, y, _bz] for x, y in _coords_utm], np.float32)
                    _cx2 = _verts[:, 0].mean(); _cy2 = _verts[:, 1].mean()
                    _cen_v = np.array([[_cx2, _cy2, _bz]], np.float32)
                    _spk_v.append(_verts); _spk_v.append(_cen_v)
                    for _i in range(_n):
                        _spk_f.append([_spk_off + _i,
                                        _spk_off + (_i + 1) % _n,
                                        _spk_off + _n])
                    _spk_off += _n + 1
                    _spk_cnt += 1
            except Exception: pass
        if _spk_v:
            _sv = np.concatenate(_spk_v).astype(np.float32)
            _sf = np.array(_spk_f, np.int32)
            _write_ply(_sv, _sf, os.path.join(MESH_DIR, 'surface_itu_asphalt_carparks.ply'))
            mat_plys.setdefault('itu_asphalt', []).append(
                ('meshes/surface_itu_asphalt_carparks.ply', 'surface_parks'))
            print(f'  Surface car parks: {_spk_cnt} → surface_itu_asphalt_carparks.ply '
                  f'({len(_sv):,} verts, {len(_sf):,} faces)')
        else:
            print('  Surface car parks: none found or no polygon geometry')
    except Exception as _e:
        print(f'  Surface car parks download failed: {_e}')
else:
    print('Surface car parks: skipped (INCLUDE_SURFACE_PARKS=False)')

# ── Greenhouses → itu_glass PLY ──────────────────────────────────────────────
# building=greenhouse/glasshouse → extruded glass walls + roof
# ITU-R P.2040-2: itu_glass  er=6.27(f^0)  σ=0.0043·f^1.1925  s=0.08  xpd=0.02  wt=0.012m
# Effect: glass transmits more RF than brick — prevents over-attenuation in horticultural areas
if globals().get('INCLUDE_GREENHOUSES', True):
    print('\nDownloading OSM greenhouses ...')
    try:
        _gh_tags = {'building': ['greenhouse', 'glasshouse', 'conservatory']}
        if _ox_version >= (2, 0):
            gdf_gh = ox.features_from_bbox(
                bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH), tags=_gh_tags)
        elif _ox_version >= (1, 3):
            gdf_gh = ox.features_from_bbox(
                bbox=(SCENE_NORTH, SCENE_SOUTH, SCENE_EAST, SCENE_WEST), tags=_gh_tags)
        else:
            gdf_gh = ox.features_from_bbox(
                north=SCENE_NORTH, south=SCENE_SOUTH,
                east=SCENE_EAST,   west=SCENE_WEST, tags=_gh_tags)
        _GH_H_DEFAULT = 4.0   # m — typical greenhouse height
        _gh_v, _gh_f, _gh_off, _gh_cnt = [], [], 0, 0
        for _, row in gdf_gh.iterrows():
            try:
                geom = row.geometry
                if geom is None or geom.is_empty: continue
                if geom.geom_type not in ('Polygon', 'MultiPolygon'): continue
                _ht = _GH_H_DEFAULT
                for _htag in ('height', 'building:height'):
                    _hv = str(row.get(_htag, '') or '')
                    if _hv not in ('nan', 'None', ''):
                        try: _ht = float(_hv.replace('m', '').strip()); break
                        except ValueError: pass
                _poly_list = list(geom.geoms) if geom.geom_type == 'MultiPolygon' else [geom]
                for _poly in _poly_list:
                    _coords_utm = []
                    for lon, lat in _poly.exterior.coords[:-1]:
                        ex, ny = to_utm.transform(lon, lat)
                        _coords_utm.append((ex - center_utm[0], ny - center_utm[1]))
                    if len(_coords_utm) < 3: continue
                    _cen_lon, _cen_lat = to_wgs84.transform(
                        _poly.centroid.x, _poly.centroid.y)
                    _bz = 0.0 if FLAT_TERRAIN else float(local_z(_cen_lon, _cen_lat))
                    _res = _extrude_building(_coords_utm, _bz, _ht,
                                             roof_h=0.0, roof_shape='flat')
                    if _res is None: continue
                    _wv, _wf, _rv, _rf = _res
                    for _vv, _ff in ((_wv, _wf), (_rv, _rf)):
                        if _vv is None or len(_vv) == 0: continue
                        _gh_f.extend([[fi + _gh_off for fi in face] for face in _ff])
                        _gh_v.append(_vv); _gh_off += len(_vv)
                    _gh_cnt += 1
            except Exception: pass
        if _gh_v:
            _gv = np.concatenate(_gh_v).astype(np.float32)
            _gf = np.array(_gh_f, np.int32)
            _write_ply(_gv, _gf, os.path.join(MESH_DIR, 'bld_itu_glass_greenhouses.ply'))
            mat_plys.setdefault('itu_glass', []).append(
                ('meshes/bld_itu_glass_greenhouses.ply', 'greenhouses'))
            print(f'  Greenhouses: {_gh_cnt} → bld_itu_glass_greenhouses.ply '
                  f'({len(_gv):,} verts, {len(_gf):,} faces)')
        else:
            print('  Greenhouses: none found in bbox')
    except Exception as _e:
        print(f'  Greenhouses download failed: {_e}')
else:
    print('Greenhouses: skipped (INCLUDE_GREENHOUSES=False)')



# ====================================================================
# ====================================================================
# V3 BLOCK -- ROOFTOP EQUIPMENT  (itu_metal, ITU-R P.2040-2 Table 3)
# HVAC / plant-room boxes on commercial/industrial rooftops.
# Hard diffraction edges critical above 2.8 GHz.
# er=1.0, sigma=1e7 S/m  (ITU-R P.2040-2 metal Table 3)
# Guard: entire block in try/except -- scene build never stops.
# ====================================================================
if globals().get('INCLUDE_ROOFTOP_EQUIPMENT', False):
    print('\nBuilding rooftop equipment meshes (V3, itu_metal) ...')
    try:
        import math as _mth_rte
        if 'gdf_bld' not in dir() or gdf_bld is None or len(gdf_bld) == 0:
            raise RuntimeError('gdf_bld not available -- run building download first')
        _rte_bld_types = set(globals().get('ROOFTOP_BLDG_TYPES',
            {'commercial','retail','industrial','office','supermarket',
             'warehouse','factory','hospital','university','school'}))
        _rte_h    = max(0.5, float(globals().get('ROOFTOP_EQUIP_HEIGHT_M', 2.5)))
        _rte_frac = min(0.9, max(0.05, float(globals().get('ROOFTOP_EQUIP_FRACTION', 0.25))))
        _rte_verts, _rte_faces, _rte_off, _rte_cnt, _rte_skip = [], [], 0, 0, 0
        for _, row in gdf_bld.iterrows():
            try:
                _btag = str(row.get('building', '') or '').lower()
                if _btag not in _rte_bld_types:
                    continue
                geom = row.geometry
                if geom is None or geom.is_empty:
                    continue
                _poly_list = (list(geom.geoms) if geom.geom_type == 'MultiPolygon'
                              else [geom])
                for _poly in _poly_list:
                    if _poly.geom_type != 'Polygon' or _poly.is_empty:
                        continue
                    _scale    = _mth_rte.sqrt(_rte_frac)
                    _buf_dist = -(_poly.length * (1 - _scale) / (4 * _mth_rte.pi))
                    try:
                        _small = _poly.buffer(_buf_dist)
                    except Exception:
                        _small = None
                    if _small is None or _small.is_empty:
                        continue
                    _bh = float(globals().get('DEFAULT_HEIGHT_M', 8.0))
                    for _ht in ('height', 'building:height'):
                        _hv = str(row.get(_ht, '') or '')
                        if _hv not in ('nan', 'None', '', 'none'):
                            try:
                                _bh = float(_hv.replace('m','').replace(' ','').strip())
                                break
                            except (ValueError, AttributeError):
                                pass
                    _bh = max(1.0, _bh)
                    try:
                        _cen_lon = float(_poly.centroid.x)
                        _cen_lat = float(_poly.centroid.y)
                        _base_z  = (0.0 if FLAT_TERRAIN
                                    else float(local_z(_cen_lon, _cen_lat)))
                    except Exception:
                        _base_z = 0.0
                    _coords_box = []
                    try:
                        _ext = (_small if _small.geom_type == 'Polygon'
                                else list(_small.geoms)[0])
                        for lon, lat in _ext.exterior.coords[:-1]:
                            ex, ny = to_utm.transform(lon, lat)
                            _coords_box.append((ex - center_utm[0], ny - center_utm[1]))
                    except Exception:
                        continue
                    if len(_coords_box) < 3:
                        continue
                    _res = _extrude_building(_coords_box, _base_z + _bh,
                                             _rte_h, roof_h=0.0, roof_shape='flat')
                    if _res is None:
                        continue
                    _wv, _wf, _rv, _rf = _res
                    for _vv, _ff in ((_wv, _wf), (_rv, _rf)):
                        if _vv is None or len(_vv) == 0:
                            continue
                        _rte_faces.extend([[fi + _rte_off for fi in face] for face in _ff])
                        _rte_verts.append(np.asarray(_vv, np.float32))
                        _rte_off += len(_vv)
                    _rte_cnt += 1
            except Exception:
                _rte_skip += 1
                continue
        if _rte_verts:
            _rv_arr  = np.concatenate(_rte_verts).astype(np.float32)
            _rf_arr  = np.array(_rte_faces, np.int32)
            _rte_ply = os.path.join(MESH_DIR, 'infra_itu_metal_rooftop_equip.ply')
            _write_ply(_rv_arr, _rf_arr, _rte_ply)
            mat_plys.setdefault('itu_metal', []).append(
                ('meshes/infra_itu_metal_rooftop_equip.ply', 'rooftop_equipment'))
            print(f'  Rooftop equipment: {_rte_cnt} boxes, {_rte_skip} skipped '
                  f'-> infra_itu_metal_rooftop_equip.ply '
                  f'({len(_rv_arr):,} verts, {len(_rf_arr):,} faces)')
        else:
            print(f'  Rooftop equipment: none generated (0 matching or all skipped={_rte_skip})')
    except Exception as _e_rte:
        print(f'  [WARN] Rooftop equipment failed: {_e_rte} -- continuing')
else:
    print('Rooftop equipment: skipped (INCLUDE_ROOFTOP_EQUIPMENT=False)')


# ====================================================================
# V4 HELPERS -- shared box + cylinder mesh generators
# ====================================================================
def _box_mesh(cx, cy, base_z, length, width, height, angle_rad):
    import math as _mb
    _hl, _hw2 = length / 2, width / 2
    _c, _s    = _mb.cos(angle_rad), _mb.sin(angle_rad)
    _corners  = [(-_hl,-_hw2), (_hl,-_hw2), (_hl,_hw2), (-_hl,_hw2)]
    _pts = [(cx + _c*lx - _s*ly, cy + _s*lx + _c*ly) for lx, ly in _corners]
    _vt  = np.array([
        [_pts[0][0], _pts[0][1], base_z],         [_pts[1][0], _pts[1][1], base_z],
        [_pts[2][0], _pts[2][1], base_z],         [_pts[3][0], _pts[3][1], base_z],
        [_pts[0][0], _pts[0][1], base_z+height],  [_pts[1][0], _pts[1][1], base_z+height],
        [_pts[2][0], _pts[2][1], base_z+height],  [_pts[3][0], _pts[3][1], base_z+height],
    ], np.float32)
    _fc = np.array([[0,1,2],[0,2,3],[4,6,5],[4,7,6],
                    [0,1,5],[0,5,4],[2,3,7],[2,7,6],
                    [1,2,6],[1,6,5],[3,0,4],[3,4,7]], np.int32)
    return _vt, _fc

def _cylinder_mesh(cx, cy, base_z, radius, height, n_segs):
    import math as _mc
    n_segs = max(3, int(n_segs))
    radius = max(1e-4, float(radius))
    height = max(1e-4, float(height))
    _angs   = [2 * _mc.pi * i / n_segs for i in range(n_segs)]
    _ring_b = np.array([[cx+radius*_mc.cos(a), cy+radius*_mc.sin(a), base_z]
                         for a in _angs], np.float32)
    _ring_t = np.array([[cx+radius*_mc.cos(a), cy+radius*_mc.sin(a), base_z+height]
                         for a in _angs], np.float32)
    _cap_b  = np.array([[cx, cy, base_z]],           np.float32)
    _cap_t  = np.array([[cx, cy, base_z + height]],  np.float32)
    _verts  = np.concatenate([_ring_b, _ring_t, _cap_b, _cap_t])
    _faces  = []
    for _i in range(n_segs):
        _j = (_i + 1) % n_segs
        _faces += [[_i, _j, n_segs+_j], [_i, n_segs+_j, n_segs+_i]]
        _faces += [[2*n_segs,   _j,         _i         ]]
        _faces += [[2*n_segs+1, n_segs+_i,  n_segs+_j  ]]
    return _verts, np.array(_faces, np.int32)


# ====================================================================
# V4 BLOCK A -- PARKED VEHICLES  (itu_metal, ITU-R P.2040-2 Table 3)
# Box 4.5x1.8x1.4m along OSM parking lanes/areas.
# Full Fresnel-zone blocker at 28 GHz. Guard: OSM fail = skip silently.
# ====================================================================
if globals().get('INCLUDE_PARKED_VEHICLES', False):
    print('\nBuilding parked vehicle meshes (V4, itu_metal) ...')
    try:
        import math as _mth_pv
        _veh_l  = max(1.0, float(globals().get('VEHICLE_LENGTH_M',  4.5)))
        _veh_w  = max(0.5, float(globals().get('VEHICLE_WIDTH_M',   1.8)))
        _veh_h  = max(0.3, float(globals().get('VEHICLE_HEIGHT_M',  1.4)))
        _veh_sp = max(_veh_l + 0.1, float(globals().get('VEHICLE_SPACING_M', 6.0)))
        _veh_of = max(0.0, float(globals().get('VEHICLE_OFFSET_M',  2.5)))
        gdf_pk  = None
        try:
            _pk_tags = {'amenity': 'parking',
                        'parking': ['lane','street_side','on_street']}
            gdf_pk = (ox.features_from_bbox(
                          bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH),
                          tags=_pk_tags)
                      if _ox_version >= (2, 0) else
                      ox.features_from_bbox(
                          bbox=(SCENE_NORTH, SCENE_SOUTH, SCENE_EAST, SCENE_WEST),
                          tags=_pk_tags))
            print(f'  Parking features: {len(gdf_pk)}')
        except Exception as _epk:
            print(f'  [WARN] Parking OSM download failed: {_epk} -- no vehicles placed')
        _pveh_verts, _pveh_faces, _pveh_off, _pveh_cnt, _pveh_skip = [], [], 0, 0, 0
        if gdf_pk is not None and len(gdf_pk) > 0:
            for _, row in gdf_pk.iterrows():
                try:
                    geom = row.geometry
                    if geom is None or geom.is_empty:
                        continue
                    _lines_utm = []
                    if geom.geom_type == 'LineString':
                        _pts = []
                        for lon, lat in geom.coords:
                            ex, ny = to_utm.transform(lon, lat)
                            _pts.append((ex - center_utm[0], ny - center_utm[1]))
                        if len(_pts) >= 2:
                            _lines_utm.append(_pts)
                    elif geom.geom_type in ('Polygon', 'MultiPolygon'):
                        _plist = (list(geom.geoms) if geom.geom_type == 'MultiPolygon'
                                  else [geom])
                        for _pp in _plist:
                            if _pp.is_empty:
                                continue
                            _pts = []
                            for lon, lat in _pp.exterior.coords[:-1]:
                                ex, ny = to_utm.transform(lon, lat)
                                _pts.append((ex - center_utm[0], ny - center_utm[1]))
                            if len(_pts) >= 2:
                                _lines_utm.append(_pts)
                    for _pts_l in _lines_utm:
                        for _si in range(len(_pts_l) - 1):
                            _x0, _y0 = _pts_l[_si]
                            _x1, _y1 = _pts_l[_si + 1]
                            _seg_len  = _mth_pv.hypot(_x1-_x0, _y1-_y0)
                            if _seg_len < _veh_l:
                                continue
                            _dx, _dy = (_x1-_x0)/_seg_len, (_y1-_y0)/_seg_len
                            _nx, _ny = -_dy, _dx
                            _angle   = _mth_pv.atan2(_dy, _dx)
                            _pos = 0.0
                            while _pos + _veh_l <= _seg_len:
                                _mid = _pos + _veh_l / 2
                                _vcx = _x0 + _dx*_mid + _nx*_veh_of
                                _vcy = _y0 + _dy*_mid + _ny*_veh_of
                                try:
                                    _lon_v, _lat_v = to_wgs84.transform(
                                        _vcx + center_utm[0], _vcy + center_utm[1])
                                    _bz_v = (0.0 if FLAT_TERRAIN
                                             else float(local_z(_lon_v, _lat_v)))
                                except Exception:
                                    _bz_v = 0.0
                                _vt, _fc = _box_mesh(_vcx, _vcy, _bz_v,
                                                      _veh_l, _veh_w, _veh_h, _angle)
                                _pveh_faces.extend([[fi+_pveh_off for fi in f] for f in _fc])
                                _pveh_verts.append(_vt)
                                _pveh_off += 8
                                _pveh_cnt += 1
                                _pos += _veh_sp
                except Exception:
                    _pveh_skip += 1
                    continue
        if _pveh_verts:
            _pv_arr = np.concatenate(_pveh_verts).astype(np.float32)
            _pf_arr = np.array(_pveh_faces, np.int32)
            _pv_ply = os.path.join(MESH_DIR, 'infra_itu_metal_parked_vehicles.ply')
            _write_ply(_pv_arr, _pf_arr, _pv_ply)
            mat_plys.setdefault('itu_metal', []).append(
                ('meshes/infra_itu_metal_parked_vehicles.ply', 'parked_vehicles'))
            print(f'  Parked vehicles: {_pveh_cnt} boxes, {_pveh_skip} skipped '
                  f'-> infra_itu_metal_parked_vehicles.ply '
                  f'({len(_pv_arr):,} verts, {len(_pf_arr):,} faces)')
        else:
            print(f'  Parked vehicles: none placed (no geometry or all skipped={_pveh_skip})')
    except Exception as _e_pv:
        print(f'  [WARN] Parked vehicles failed: {_e_pv} -- continuing')
else:
    print('Parked vehicles: skipped (INCLUDE_PARKED_VEHICLES=False)')


# ====================================================================
# V4 BLOCK B -- STREET LAMPS  (itu_metal, ITU-R P.2040-2 Table 3)
# Hexagonal cylinders r=0.05m h=6m every 35m on road edges.
# Point scatterers above 10 GHz. Guard: road download fail = skip.
# ====================================================================
if globals().get('INCLUDE_STREET_LAMPS', False):
    print('\nBuilding street lamp meshes (V4, itu_metal) ...')
    try:
        import math as _mth_lmp
        _lmp_h   = max(1.0,  float(globals().get('LAMP_HEIGHT_M',      6.0)))
        _lmp_r   = max(0.01, float(globals().get('LAMP_RADIUS_M',      0.05)))
        _lmp_sp  = max(5.0,  float(globals().get('LAMP_SPACING_M',     35.0)))
        _lmp_of  = max(0.0,  float(globals().get('LAMP_ROAD_OFFSET_M', 2.5)))
        _lmp_rts = set(globals().get('LAMP_ROAD_TYPES',
                        {'primary','secondary','tertiary','residential','unclassified'}))
        _N_POLE_SEGS = 6
        _edges_road  = None
        try:
            _G_roads = (ox.graph_from_bbox(
                            bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH))
                        if _ox_version >= (2, 0) else
                        ox.graph_from_bbox(
                            north=SCENE_NORTH, south=SCENE_SOUTH,
                            east=SCENE_EAST,   west=SCENE_WEST,
                            network_type='drive'))
            _edges_road = ox.graph_to_gdfs(_G_roads, nodes=False, edges=True)
            print(f'  Road edges: {len(_edges_road)}')
        except Exception as _elmp:
            print(f'  [WARN] Road network download failed: {_elmp} -- no lamps placed')
        _lmp_verts, _lmp_faces, _lmp_off, _lmp_cnt, _lmp_skip = [], [], 0, 0, 0
        if _edges_road is not None and len(_edges_road) > 0:
            for _, row in _edges_road.iterrows():
                try:
                    _hw = row.get('highway', '')
                    if isinstance(_hw, list):
                        _hw = _hw[0] if _hw else ''
                    if str(_hw) not in _lmp_rts:
                        continue
                    geom = row.geometry
                    if geom is None or geom.is_empty:
                        continue
                    _pts_utm = []
                    for lon, lat in geom.coords:
                        ex, ny = to_utm.transform(lon, lat)
                        _pts_utm.append((ex - center_utm[0], ny - center_utm[1]))
                    if len(_pts_utm) < 2:
                        continue
                    for _si in range(len(_pts_utm) - 1):
                        _x0, _y0 = _pts_utm[_si]
                        _x1, _y1 = _pts_utm[_si + 1]
                        _seg_len  = _mth_lmp.hypot(_x1-_x0, _y1-_y0)
                        if _seg_len < 1.0:
                            continue
                        _dx, _dy = (_x1-_x0)/_seg_len, (_y1-_y0)/_seg_len
                        _nx, _ny = -_dy, _dx
                        _n_lamps = max(1, int(_seg_len / _lmp_sp))
                        for _li in range(_n_lamps):
                            _t  = (_li + 0.5) * _seg_len / _n_lamps
                            _px = _x0 + _dx*_t + _nx*_lmp_of
                            _py = _y0 + _dy*_t + _ny*_lmp_of
                            try:
                                _lon_l, _lat_l = to_wgs84.transform(
                                    _px + center_utm[0], _py + center_utm[1])
                                _bz_l = (0.0 if FLAT_TERRAIN
                                         else float(local_z(_lon_l, _lat_l)))
                            except Exception:
                                _bz_l = 0.0
                            _vt, _fc = _cylinder_mesh(
                                _px, _py, _bz_l, _lmp_r, _lmp_h, _N_POLE_SEGS)
                            _lmp_faces.extend([[fi+_lmp_off for fi in f] for f in _fc])
                            _lmp_verts.append(_vt)
                            _lmp_off += len(_vt)
                            _lmp_cnt += 1
                except Exception:
                    _lmp_skip += 1
                    continue
        if _lmp_verts:
            _lv_arr = np.concatenate(_lmp_verts).astype(np.float32)
            _lf_arr = np.array(_lmp_faces, np.int32)
            _lp_ply = os.path.join(MESH_DIR, 'infra_itu_metal_street_lamps.ply')
            _write_ply(_lv_arr, _lf_arr, _lp_ply)
            mat_plys.setdefault('itu_metal', []).append(
                ('meshes/infra_itu_metal_street_lamps.ply', 'street_lamps'))
            print(f'  Street lamps: {_lmp_cnt} poles, {_lmp_skip} skipped '
                  f'-> infra_itu_metal_street_lamps.ply '
                  f'({len(_lv_arr):,} verts, {len(_lf_arr):,} faces)')
        else:
            print(f'  Street lamps: none placed (no matching edges or all skipped={_lmp_skip})')
    except Exception as _e_lmp:
        print(f'  [WARN] Street lamps failed: {_e_lmp} -- continuing')
else:
    print('Street lamps: skipped (INCLUDE_STREET_LAMPS=False)')


# ====================================================================
# V5 BLOCK A -- HUMAN BODY PHANTOMS  (itu_concrete, ITU-R P.1238 / P.2040-2)
# Cylinder r=0.2m h=1.7m at each RX. 15-20 dB blockage at 60 GHz.
# Material: itu_concrete (er=5.31, ITU-R P.2040-2 Table 3).
# Guard: missing/bad CSV rows silently skipped.
# ====================================================================
if globals().get('INCLUDE_BODY_PHANTOMS', False):
    print('\nBuilding human body phantom meshes (V5, itu_concrete) ...')
    try:
        _ph_h    = max(0.5,  float(globals().get('PHANTOM_HEIGHT_M', 1.7)))
        _ph_r    = max(0.05, float(globals().get('PHANTOM_RADIUS_M', 0.2)))
        _rx_agl  = float(globals().get('RX_AGL_M', 1.5))
        _N_PH_SEGS = 8
        _RX_CSV  = globals().get('MEASUREMENT_CSV', globals().get('RX_CSV', None))
        if _RX_CSV is None:
            print('  [SKIP] MEASUREMENT_CSV not set in Cell 0')
        elif not os.path.exists(str(_RX_CSV)):
            print(f'  [SKIP] File not found: {_RX_CSV}')
        else:
            import pandas as _pd_ph
            try:
                _df_rx = _pd_ph.read_csv(str(_RX_CSV))
            except Exception as _ecsv:
                _df_rx = None
                print(f'  [SKIP] Cannot read CSV: {_ecsv}')
            if _df_rx is not None and len(_df_rx) > 0:
                _lon_col = next((c for c in _df_rx.columns
                                  if 'lon' in c.lower() or 'lng' in c.lower()), None)
                _lat_col = next((c for c in _df_rx.columns
                                  if 'lat' in c.lower() and 'lon' not in c.lower()), None)
                if _lat_col is None:
                    _lat_col = next((c for c in _df_rx.columns
                                     if 'lat' in c.lower()), None)
                if _lon_col is None or _lat_col is None:
                    print(f'  [SKIP] No lon/lat columns in CSV. '
                          f'Available: {list(_df_rx.columns)}')
                else:
                    _ph_verts, _ph_faces, _ph_off = [], [], 0
                    _ph_cnt, _ph_skip = 0, 0
                    for _, row in _df_rx.iterrows():
                        try:
                            _lon_p = float(row[_lon_col])
                            _lat_p = float(row[_lat_col])
                            if not (-180 <= _lon_p <= 180 and -90 <= _lat_p <= 90):
                                _ph_skip += 1; continue
                            _ex, _ny = to_utm.transform(_lon_p, _lat_p)
                            _pxp  = _ex - center_utm[0]
                            _pyp  = _ny - center_utm[1]
                            _bz_p = (0.0 if FLAT_TERRAIN
                                     else float(local_z(_lon_p, _lat_p)))
                            _bot  = _bz_p + _rx_agl - _ph_h
                            _vt, _fc = _cylinder_mesh(
                                _pxp, _pyp, _bot, _ph_r, _ph_h, _N_PH_SEGS)
                            _ph_faces.extend([[fi+_ph_off for fi in f] for f in _fc])
                            _ph_verts.append(_vt)
                            _ph_off += len(_vt)
                            _ph_cnt += 1
                        except Exception:
                            _ph_skip += 1
                            continue
                    if _ph_verts:
                        _phv_arr = np.concatenate(_ph_verts).astype(np.float32)
                        _phf_arr = np.array(_ph_faces, np.int32)
                        _pp_ply  = os.path.join(
                            MESH_DIR, 'infra_itu_concrete_body_phantoms.ply')
                        _write_ply(_phv_arr, _phf_arr, _pp_ply)
                        mat_plys.setdefault('itu_concrete', []).append(
                            ('meshes/infra_itu_concrete_body_phantoms.ply',
                             'body_phantoms'))
                        print(f'  Body phantoms: {_ph_cnt} written, {_ph_skip} skipped '
                              f'-> infra_itu_concrete_body_phantoms.ply '
                              f'({len(_phv_arr):,} verts, {len(_phf_arr):,} faces)')
                    else:
                        print(f'  Body phantoms: none placed (all skipped={_ph_skip})')
    except Exception as _e_ph:
        print(f'  [WARN] Body phantoms failed: {_e_ph} -- continuing')
else:
    print('Body phantoms: skipped (INCLUDE_BODY_PHANTOMS=False)')


# ====================================================================
# V5 BLOCK B -- BUS SHELTERS / KIOSKS  (itu_metal + itu_glass, ITU-R P.2040-2)
# Metal roof (er=1, sigma=1e7) + 3 glass sides (er=6.27, sigma~0).
# Glass transparent at 915 MHz; hard blocker at 28-60 GHz.
# Guard: OSM fail + per-feature errors silently skip.
# ====================================================================
if globals().get('INCLUDE_SHELTERS', False):
    print('\nBuilding bus shelter / kiosk meshes (V5, itu_metal + itu_glass) ...')
    try:
        _sh_h = max(1.0, float(globals().get('SHELTER_HEIGHT_M', 2.5)))
        _sh_d = max(0.5, float(globals().get('SHELTER_DEPTH_M',  1.5)))
        _sh_w = max(0.5, float(globals().get('SHELTER_WIDTH_M',  3.0)))
        gdf_sh = None
        try:
            _sh_tags = {'amenity': ['shelter','telephone','vending_machine'],
                        'man_made': ['telephone_box']}
            gdf_sh = (ox.features_from_bbox(
                          bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH),
                          tags=_sh_tags)
                      if _ox_version >= (2, 0) else
                      ox.features_from_bbox(
                          bbox=(SCENE_NORTH, SCENE_SOUTH, SCENE_EAST, SCENE_WEST),
                          tags=_sh_tags))
            print(f'  Shelter features: {len(gdf_sh)}')
        except Exception as _esh:
            print(f'  [WARN] Shelter OSM download failed: {_esh} -- no shelters placed')
        _sh_metal_v, _sh_metal_f, _sh_m_off = [], [], 0
        _sh_glass_v, _sh_glass_f, _sh_g_off = [], [], 0
        _sh_cnt, _sh_skip = 0, 0
        if gdf_sh is not None and len(gdf_sh) > 0:
            for _, row in gdf_sh.iterrows():
                try:
                    geom = row.geometry
                    if geom is None or geom.is_empty:
                        continue
                    _cx_wgs = float(geom.centroid.x)
                    _cy_wgs = float(geom.centroid.y)
                    _ex, _ny = to_utm.transform(_cx_wgs, _cy_wgs)
                    _sx   = _ex - center_utm[0]
                    _sy   = _ny - center_utm[1]
                    _bz_s = (0.0 if FLAT_TERRAIN
                              else float(local_z(_cx_wgs, _cy_wgs)))
                    _hw_s, _hd_s = _sh_w / 2, _sh_d / 2
                    _roof_v = np.array([
                        [_sx-_hw_s, _sy-_hd_s, _bz_s+_sh_h],
                        [_sx+_hw_s, _sy-_hd_s, _bz_s+_sh_h],
                        [_sx+_hw_s, _sy+_hd_s, _bz_s+_sh_h],
                        [_sx-_hw_s, _sy+_hd_s, _bz_s+_sh_h],
                    ], np.float32)
                    _roof_f = np.array([[0,1,2],[0,2,3]], np.int32)
                    _sh_metal_f.extend([[fi+_sh_m_off for fi in f] for f in _roof_f])
                    _sh_metal_v.append(_roof_v); _sh_m_off += 4
                    for _p0, _p1 in [
                        ([_sx-_hw_s, _sy-_hd_s], [_sx+_hw_s, _sy-_hd_s]),
                        ([_sx-_hw_s, _sy-_hd_s], [_sx-_hw_s, _sy+_hd_s]),
                        ([_sx+_hw_s, _sy-_hd_s], [_sx+_hw_s, _sy+_hd_s]),
                    ]:
                        _panel_v = np.array([
                            [_p0[0], _p0[1], _bz_s],
                            [_p1[0], _p1[1], _bz_s],
                            [_p1[0], _p1[1], _bz_s+_sh_h],
                            [_p0[0], _p0[1], _bz_s+_sh_h],
                        ], np.float32)
                        _panel_f = np.array([[0,1,2],[0,2,3]], np.int32)
                        _sh_glass_f.extend([[fi+_sh_g_off for fi in f] for f in _panel_f])
                        _sh_glass_v.append(_panel_v); _sh_g_off += 4
                    _sh_cnt += 1
                except Exception:
                    _sh_skip += 1
                    continue
        for _mat_key, _vlist, _flist, _fname, _label in [
            ('itu_metal', _sh_metal_v, _sh_metal_f,
             'infra_itu_metal_shelter_roofs.ply', 'shelter_roofs'),
            ('itu_glass', _sh_glass_v, _sh_glass_f,
             'infra_itu_glass_shelter_walls.ply', 'shelter_walls'),
        ]:
            if _vlist:
                _sv_arr = np.concatenate(_vlist).astype(np.float32)
                _sf_arr = np.array(_flist, np.int32)
                _sp_ply = os.path.join(MESH_DIR, _fname)
                _write_ply(_sv_arr, _sf_arr, _sp_ply)
                mat_plys.setdefault(_mat_key, []).append((f'meshes/{_fname}', _label))
                print(f'  Shelters ({_mat_key}): {_sh_cnt} written, '
                      f'{_sh_skip} skipped -> {_fname} '
                      f'({len(_sv_arr):,} verts, {len(_sf_arr):,} faces)')
        if not _sh_metal_v:
            print(f'  Shelters: none placed (no OSM features; skipped={_sh_skip})')
    except Exception as _e_sh:
        print(f'  [WARN] Shelters failed: {_e_sh} -- continuing')
else:
    print('Shelters: skipped (INCLUDE_SHELTERS=False)')

# ── Save vegetation footprints for P.833 post-processing ─────────────────────
if INCLUDE_VEGETATION and 'gdf_veg' in dir() and len(gdf_veg) > 0:
    import geopandas as _gpd
    _veg_utm = gdf_veg.to_crs(epsg=UTM_EPSG)
    _veg_polys = _veg_utm[_veg_utm.geometry.geom_type.isin(['Polygon','MultiPolygon'])].copy()
    _veg_out = os.path.join(SCENE_DIR, 'vegetation_footprints.geojson')
    _keep_cols = ['geometry'] + [c for c in ['natural', 'landuse'] if c in _veg_polys.columns]
    _veg_polys[_keep_cols].to_file(_veg_out, driver='GeoJSON')
    print(f'  Vegetation footprints saved: {len(_veg_polys)} polygons → {_veg_out}')


## CELL B3 — Write scene_sionna2.xml (Sionna 2.0)

Assembles all PLY meshes into a Sionna 2.0 / Mitsuba 2.1.0 compatible scene XML.

**This is the final output cell for the standard pipeline.** After this runs,
load `scene_sionna2.xml` in the DEM simulation notebook.

Non-ITU materials (asphalt, vegetation, water) are written as plain
`RadioMaterial` entries so the DEM notebook can assign EM properties
(permittivity, conductivity, scattering coefficient) at runtime.


In [ ]:
# ============================================================
# CELL 5 — WRITE SCENE.XML  (Mitsuba 2.1.0 / Sionna 0.19)
# ============================================================
# ITU-R P.2040-2 material definitions as used in Sionna 0.19.
# Each shape references a PLY file and a material by name.

# Sionna 0.19 valid ITU-R P.2040-2 built-in names + custom radio-materials
# Valid built-ins: concrete, brick, plywood, glass, ceiling_board, chipboard,
#                  floorboard, metal, very_dry_ground, medium_dry_ground, wet_ground
# NOT valid in 0.19: itu_wood → itu_plywood  |  itu_asphalt/vegetation/water → ground built-ins
# ── ITU-R P.2040-2 (2023) Table 3 — frequency-portable material parameters ──
# Formula: er(f) = a*f^b   sigma(f) = c*f^d   f in GHz
# s=scattering coefficient  xpd=cross-pol discrimination
# Static columns (b=0 or d=0) are frequency-independent.
_ITU_P2040_PARAMS = {
    #                    a       b       c        d      s     xpd
    'itu_concrete' : (5.31,  0.000,  0.0326, 0.8095, 0.30, 0.10),
    'itu_brick'    : (3.91,  0.000,  0.0238, 0.0000, 0.30, 0.10),
    'itu_glass'    : (6.27,  0.000,  0.0043, 1.1925, 0.10, 0.05),
    'itu_plywood'  : (1.99,  0.000,  0.0047, 1.0718, 0.20, 0.10),  # P.2040 wood -> plywood in 0.19
    'itu_metal'    : (1.00,  0.000,  1.0e7,  0.0000, 0.05, 0.05),  # perfect conductor
    'itu_wet_ground'        : (30.0, -0.400, 0.1500, 1.3000, 0.10, 0.05),
    'itu_water'      : (80.0,  0.000, 0.0100, 0.0000, 0.40, 0.05),  # ITU-R P.527 fresh water (εr=80, rough surface S=0.4)
    'itu_medium_dry_ground'  : (15.0, -0.100, 0.0350, 1.6300, 0.10, 0.05),  # ITU-R P.2040-2 Table 3
    'itu_very_dry_ground'    : ( 3.0,  0.000, 0.00015,2.5200, 0.10, 0.05),  # ITU-R P.2040-2 Table 3
}
_f_ghz_c5 = float(globals().get('FREQUENCY_HZ', 915.95e6)) / 1e9
ITU_MATERIALS = {}
for _mn, (_a, _b, _c, _d, _s, _xpd) in _ITU_P2040_PARAMS.items():
    _er  = round(_a * (_f_ghz_c5 ** _b), 6)
    _sig = round(_c * (_f_ghz_c5 ** _d), 8)
    ITU_MATERIALS[_mn] = (_er, _sig, _s, _xpd)
print(f'ITU_MATERIALS computed at {_f_ghz_c5*1e3:.2f} MHz (ITU-R P.2040-2):')
for _mn, (_er, _sig, _s, _xpd) in ITU_MATERIALS.items():
    print(f'  {_mn:<22}  er={_er:.4f}  sigma={_sig:.6f}  S={_s:.2f}')
# Material name remapping: mat_plys keys → ITU_MATERIALS keys
_MAT_REMAP_019 = {
    'itu_wood'       : 'itu_plywood',          # itu_wood not in 0.19 registry
    'itu_asphalt'    : 'itu_medium_dry_ground', # closest Sionna 0.19 built-in for road
    'itu_vegetation' : 'itu_very_dry_ground',   # low sigma; P.833 attenuation in DEM
    # 'itu_water' now has its own entry in _ITU_P2040_PARAMS (ITU-R P.527)
}
TERRAIN_MATERIAL = 'itu_wet_ground'

# Collect all materials actually used
used_mats = set(mat_plys.keys()) | {TERRAIN_MATERIAL}

lines = []
lines.append('<?xml version="1.0" encoding="utf-8"?>')
lines.append('<scene version="2.1.0">')
lines.append('')
lines.append('  <!-- ── ITU-R P.2040-2 Materials ────────────────────── -->')

# Apply mat_plys key remapping before writing
_used_mats_remapped = set()
for m in used_mats:
    _used_mats_remapped.add(_MAT_REMAP_019.get(m, m))
_used_mats_remapped.add(TERRAIN_MATERIAL)

for mat_name, (eps, sigma, scat, xpd) in ITU_MATERIALS.items():
    if mat_name not in _used_mats_remapped:
        continue
    lines.append(f'  <bsdf type="radio-material" id="{mat_name}">')
    lines.append(f'    <float name="relative_permittivity"  value="{eps}"/>')
    lines.append(f'    <float name="conductivity"           value="{sigma}"/>')
    lines.append(f'    <float name="scattering_coefficient" value="{scat}"/>')
    lines.append(f'    <float name="xpd_coefficient"        value="{xpd}"/>')
    lines.append(f'  </bsdf>')
    lines.append('')

lines.append('  <!-- ── Terrain ─────────────────────────────────────── -->')
lines.append('  <shape type="ply">')
lines.append('    <string name="filename" value="meshes/terrain.ply"/>')
lines.append('    <ref id="{mat}" name="bsdf"/>'.replace('{mat}', TERRAIN_MATERIAL))
lines.append('  </shape>')
lines.append('')

lines.append('  <!-- ── Buildings ───────────────────────────────────── -->')
for mat_name, ply_list in sorted(mat_plys.items()):
    for ply_path, role in ply_list:
        _ref_mat = _MAT_REMAP_019.get(mat_name, mat_name)
        lines.append(f'  <shape type="ply">')
        lines.append(f'    <string name="filename" value="{ply_path}"/>')
        lines.append(f'    <ref id="{_ref_mat}" name="bsdf"/>')
        lines.append(f'  </shape>')

lines.append('')
lines.append('</scene>')

scene_xml = os.path.join(SCENE_DIR, 'scene.xml')
with open(scene_xml, 'w') as f:
    f.write('\n'.join(lines))

print(f'Wrote: {scene_xml}')
print(f'  Materials : {len(used_mats)}')
total_shapes = 1 + sum(len(v) for v in mat_plys.values())
print(f'  Shapes    : {total_shapes}  (1 terrain + {total_shapes-1} building parts)')

# Save scene metadata for main notebook
meta = {
    'scene_center_lon'  : center_lon,
    'scene_center_lat'  : center_lat,
    'origin_elev_asl_m' : origin_elev_asl,
    'utm_epsg'          : UTM_EPSG,
    'bbox'              : {'west': SCENE_WEST, 'east': SCENE_EAST,
                           'south': SCENE_SOUTH, 'north': SCENE_NORTH},
    'n_buildings'       : n_ok,
    'terrain_grid_n'    : TERRAIN_GRID_N,
    'tile_zoom'         : TILE_ZOOM,
}
params_json = os.path.join(BASE_DIR, 'scene_parameters.json')
with open(params_json, 'w') as f:
    json.dump(meta, f, indent=2)
print(f'  Metadata  : {params_json}')

In [ ]:
# ============================================================
# CELL 6 — VERIFY SCENE
# ============================================================
# Quick sanity checks before handing the scene to the main notebook.

import glob as glob_mod

ply_files = sorted(glob_mod.glob(os.path.join(MESH_DIR, '*.ply')))
total_kb = sum(os.path.getsize(p) for p in ply_files) / 1024

print('=' * 60)
print('SCENE VERIFICATION')
print('=' * 60)
print(f'PLY files   : {len(ply_files)}')
print(f'Total size  : {total_kb/1024:.1f} MB')
print()

print(f'scene.xml   : {os.path.getsize(scene_xml)/1024:.0f} KB')

# Load with Mitsuba to verify (requires Sionna env)
try:
    import mitsuba as mi
    mi.set_variant('scalar_rgb')
    scene_mi = mi.load_file(scene_xml)
    bbox = scene_mi.bbox()
    print()
    print(f'Mitsuba load: OK')
    print(f'  BBox X    : [{float(bbox.min[0]):.1f}, {float(bbox.max[0]):.1f}] m')
    print(f'  BBox Y    : [{float(bbox.min[1]):.1f}, {float(bbox.max[1]):.1f}] m')
    print(f'  BBox Z    : [{float(bbox.min[2]):.1f}, {float(bbox.max[2]):.1f}] m')
except Exception as e:
    print(f'Mitsuba load: {e}')

print()
print('Scene metadata (scene_parameters.json):')
with open(params_json) as f:
    print(json.dumps(json.load(f), indent=2))

print()
print('DONE — scene ready for sionna019_main_simulation.ipynb')
print(f'Set BASE_DIR = "{BASE_DIR}" in Cell 0c of the main notebook.')

## CELL 7 — 2D Scene Map: OSM Buildings + Receiver Locations + RSSI Heatmap


In [ ]:
# ============================================================
# CELL 7 — 2D MAP: OSM BUILDINGS + TX/RX DOTS
# ============================================================

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
import numpy as np
import re, warnings

# ── Config — edit here ────────────────────────────────────────────────────
OFCOM_CSV   = os.path.join(BASE_DIR, 'nottingham915.csv')

# Number of RX points to plot (first N rows in CSV order, same as simulation).
# Set to None to plot all rows in the file.
NUM_RX_PLOT = 1200      # ← adjust to match NUM_RX in simulation notebook

SAVE_FIG_2D = True
FIG_DPI_2D  = 150

# ── Auto-detect header row + parse CSV ───────────────────────────────────
_tx_lat = _tx_lon = None
_rx_rows = []

if not os.path.exists(OFCOM_CSV):
    print(f'CSV not found: {OFCOM_CSV}')
else:
    # Read TX site coordinates from metadata lines
    with open(OFCOM_CSV, 'r', encoding='utf-8', errors='replace') as _f:
        _all_lines = _f.readlines()

    for _line in _all_lines[:40]:
        if 'latitude'  in _line.lower() and _tx_lat is None:
            _m = re.search(r'([-+]?\d+\.\d+)', _line)
            if _m: _tx_lat = float(_m.group(1))
        if 'longitude' in _line.lower() and _tx_lon is None:
            _m = re.search(r'([-+]?\d+\.\d+)', _line)
            if _m: _tx_lon = float(_m.group(1))

    # Find header row: first line containing both 'Latitude' and 'Longitude'
    _hdr_idx = next(
        (i for i, l in enumerate(_all_lines) if 'Latitude' in l and 'Longitude' in l),
        None
    )

    if _hdr_idx is None:
        print('WARNING: could not find header row in CSV')
    else:
        _df = pd.read_csv(OFCOM_CSV, skiprows=_hdr_idx, low_memory=False)

        # Flexible column matching
        def _fcol(df, *kws):
            for c in df.columns:
                if all(k.lower() in c.strip().lower() for k in kws):
                    return c
            return None

        _lat_col  = _fcol(_df, 'latitude')
        _lon_col  = _fcol(_df, 'longitude')
        _rssi_col = _fcol(_df, 'measurement') or _fcol(_df, 'dbm')

        if _lat_col and _lon_col and _rssi_col:
            for _c in [_lat_col, _lon_col, _rssi_col]:
                _df[_c] = pd.to_numeric(_df[_c], errors='coerce')
            _sel = _df[[_lat_col, _lon_col, _rssi_col]].dropna().reset_index(drop=True)

            # Take first NUM_RX_PLOT rows (same order as simulation)
            if NUM_RX_PLOT is not None:
                _sel = _sel.head(NUM_RX_PLOT)

            _rx_rows = list(zip(_sel[_lon_col], _sel[_lat_col], _sel[_rssi_col]))
            print(f'Loaded {len(_rx_rows)} RX points  '
                  f'(RSSI {float(_sel[_rssi_col].min()):.1f} – {float(_sel[_rssi_col].max()):.1f} dBm)')
        else:
            print(f'WARNING: could not find lat/lon/rssi columns. '
                  f'Available: {list(_df.columns)}')

# ── Plot ──────────────────────────────────────────────────────────────────
fig2d, ax2d = plt.subplots(figsize=(11, 10), dpi=FIG_DPI_2D)

# Building footprints
_gdf_plot = gdf_bld.copy()
if hasattr(_gdf_plot, 'crs') and _gdf_plot.crs and str(_gdf_plot.crs) != 'EPSG:4326':
    _gdf_plot = _gdf_plot.to_crs('EPSG:4326')
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    _gdf_plot.plot(ax=ax2d, facecolor='#d8d0c4', edgecolor='#999999',
                   linewidth=0.15, alpha=0.85, zorder=2)

ax2d.set_xlim(SCENE_WEST, SCENE_EAST)
ax2d.set_ylim(SCENE_SOUTH, SCENE_NORTH)
ax2d.set_aspect('equal')
ax2d.set_facecolor('#eef2f5')
ax2d.set_xlabel('Longitude', fontsize=10)
ax2d.set_ylabel('Latitude',  fontsize=10)
ax2d.tick_params(labelsize=8)
ax2d.grid(True, linestyle='--', linewidth=0.4, alpha=0.5, zorder=1)

# Scene bbox border
from matplotlib.patches import Rectangle as _Rect
ax2d.add_patch(_Rect((SCENE_WEST, SCENE_SOUTH),
                      SCENE_EAST - SCENE_WEST, SCENE_NORTH - SCENE_SOUTH,
                      linewidth=1.5, edgecolor='black', facecolor='none',
                      linestyle='--', zorder=6))

# RX dots coloured by RSSI
if _rx_rows:
    _lons_p = np.array([p[0] for p in _rx_rows])
    _lats_p = np.array([p[1] for p in _rx_rows])
    _rssi_p = np.array([p[2] for p in _rx_rows])
    _vmin_p = float(np.percentile(_rssi_p, 2))
    _vmax_p = float(np.percentile(_rssi_p, 98))
    _sc2d = ax2d.scatter(_lons_p, _lats_p, c=_rssi_p, s=2.5,
                         cmap='RdYlGn', vmin=_vmin_p, vmax=_vmax_p,
                         alpha=0.65, linewidths=0, zorder=5, label='RX (RSSI)')
    _cb2d = fig2d.colorbar(_sc2d, ax=ax2d, fraction=0.025, pad=0.01, shrink=0.7)
    _cb2d.set_label('RSSI (dBm)', fontsize=9)
    _cb2d.ax.tick_params(labelsize=8)

# TX star
if _tx_lon and _tx_lat:
    ax2d.plot(_tx_lon, _tx_lat, marker='*', markersize=16, color='red',
              markeredgecolor='darkred', markeredgewidth=0.8,
              zorder=10, label='TX (transmitter)')
    ax2d.annotate('TX', (_tx_lon, _tx_lat),
                  textcoords='offset points', xytext=(8, 5),
                  fontsize=9, color='darkred', fontweight='bold', zorder=11)

_n_bld = len(_gdf_plot) if '_gdf_plot' in dir() else 0
ax2d.set_title(
    f'Nottingham 915 MHz — OSM Scene Map\n'
    f'{_n_bld:,} buildings  |  {len(_rx_rows):,} RX (first {NUM_RX_PLOT})  |  '
    f'{(SCENE_EAST-SCENE_WEST)*111.32*np.cos(np.radians((SCENE_SOUTH+SCENE_NORTH)/2)):.1f} × '
    f'{(SCENE_NORTH-SCENE_SOUTH)*111.32:.1f} km',
    fontsize=11)

ax2d.legend(loc='upper right', fontsize=9, markerscale=3,
            framealpha=0.85, edgecolor='grey')

plt.tight_layout()

if SAVE_FIG_2D:
    _fig2d_path = os.path.join(BASE_DIR, 'scene_map_2d.png')
    fig2d.savefig(_fig2d_path, dpi=FIG_DPI_2D, bbox_inches='tight')
    print(f'Saved: {_fig2d_path}')

plt.show()
print('2D map done.')

## CELL 8 — Interactive 3D RSSI Heatmap (Plotly)


## CELL 9 — Customised Antenna Patterns (Ofcom 915 MHz)

Defines TX and RX antenna pattern functions matching the Ofcom drive-test equipment,
then exports all link-budget parameters to `scene_parameters.json` for automatic
pick-up by the simulation notebook.


In [ ]:
# ============================================================
# CELL 9 — CUSTOMISED ANTENNA PATTERNS  (Ofcom 915 MHz)
# ============================================================
# Defines TX / RX antenna pattern callables matching the Ofcom
# drive-test equipment used in the nottingham915.csv dataset.
#
# TX : collinear omni mast — donut shaped, peak = TX_ANTENNA_GAIN_DBI
# RX : vehicle rooftop — effectively isotropic; equipment losses
#      (cable, splitter, BPF) folded into RX_EXTRA_GAIN_DB
#
# Pattern functions are TensorFlow callables accepted by Sionna's
# PlanarArray.  A safe fallback to 'hw_dipole' / 'iso' is provided
# if the Sionna version does not accept callables.
#
# After defining patterns, all link-budget parameters are written to
# scene_parameters.json so the simulation notebook reads them
# automatically.
# ============================================================

import numpy as np
import json as _json

# ── Half-wave dipole directivity (linear) ─────────────────────────────────
_D_HW = 1.6409   # D of ideal half-wave dipole (= 2.15 dBi in linear)

# ── TX : donut omni scaled to TX_ANTENNA_GAIN_DBI ─────────────────────────
def _tx_pattern_915(theta, phi):
    """
    Ofcom 915 MHz TX — collinear omni mast.
    Pattern: donut (half-wave dipole shape) scaled so peak = TX_ANTENNA_GAIN_DBI.
    F_theta = scale * cos(pi/2 * cos(theta)) / sin(theta)
    F_phi   = 0
    """
    import tensorflow as tf
    _scale = np.float32((10 ** (TX_ANTENNA_GAIN_DBI / 10) / _D_HW) ** 0.5)
    cos_t  = tf.cos(theta)
    sin_t  = tf.sin(theta)
    safe_s = tf.where(tf.abs(sin_t) < 1e-6,
                      tf.ones_like(sin_t) * 1e-6, sin_t)
    f_theta = tf.cast(_scale * tf.cos(np.float32(np.pi / 2) * cos_t) / safe_s,
                      tf.complex64)
    f_phi   = tf.zeros_like(f_theta)
    return f_theta, f_phi

# ── RX : isotropic — equipment losses in RX_EXTRA_GAIN_DB ─────────────────
def _rx_pattern_915(theta, phi):
    """
    Ofcom 915 MHz RX — vehicle rooftop omni.
    Treated as isotropic (0 dBi); drive-test NLOS paths arrive from varied
    elevations so a directional pattern would introduce azimuth bias.
    Cable / splitter / filter losses are captured in RX_EXTRA_GAIN_DB.
    """
    import tensorflow as tf
    _iso = tf.cast(tf.ones_like(theta) / np.float32(np.sqrt(4.0 * np.pi)),
                   tf.complex64)
    f_phi = tf.zeros_like(_iso)
    return _iso, f_phi

# ── Helper: build PlanarArray with fallback ───────────────────────────────
def _make_antenna_array(pattern_fn, fallback='hw_dipole'):
    """
    Try to create a 1×1 PlanarArray with a callable pattern.
    Falls back to a named built-in if Sionna rejects the callable.
    Returns (array, mode_string).
    """
    try:
        from sionna.rt import PlanarArray
        arr = PlanarArray(num_rows=1, num_cols=1,
                          vertical_spacing=0.5, horizontal_spacing=0.5,
                          pattern=pattern_fn, polarization='V')
        return arr, 'callable'
    except Exception as _e:
        try:
            from sionna.rt import PlanarArray
            arr = PlanarArray(num_rows=1, num_cols=1,
                              vertical_spacing=0.5, horizontal_spacing=0.5,
                              pattern=fallback, polarization='V')
            return arr, f'fallback={fallback} ({_e})'
        except Exception as _e2:
            return None, f'failed ({_e2})'

# ── Verify pattern shape at sample angles ─────────────────────────────────
print('Antenna pattern check (915 MHz):')
print(f'  TX pattern : donut omni  peak = {TX_ANTENNA_GAIN_DBI:+.1f} dBi')
print(f'  RX pattern : isotropic   system gain = {RX_EXTRA_GAIN_DB:.1f} dB')
print(f'  Frequency  : {FREQUENCY_HZ/1e6:.2f} MHz')
print(f'  TX AGL     : {TX_AGL_M:.1f} m   RX AGL : {RX_AGL_M:.1f} m')
print(f'  TX EIRP    : {TX_CONDUCTED_DBM + TX_ANTENNA_GAIN_DBI:.1f} dBm  '
      f'(conducted={TX_CONDUCTED_DBM:.1f} + antenna={TX_ANTENNA_GAIN_DBI:.1f})')
print()

# Evaluate TX pattern at horizon (theta=pi/2) and zenith (theta=0)
try:
    import tensorflow as tf
    _th_h = tf.constant([np.pi/2], dtype=tf.float32)
    _ph_h = tf.constant([0.0],     dtype=tf.float32)
    _fth, _ = _tx_pattern_915(_th_h, _ph_h)
    _g_horizon = 10 * np.log10(float(tf.reduce_sum(tf.abs(_fth)**2).numpy()) * _D_HW)
    print(f'  TX gain at horizon (theta=90°): {_g_horizon:.2f} dBi  '
          f'(expected {TX_ANTENNA_GAIN_DBI:.2f} dBi)')

    _th_z = tf.constant([0.01], dtype=tf.float32)   # avoid exact 0
    _fth_z, _ = _tx_pattern_915(_th_z, _ph_h)
    _g_zenith = 10 * np.log10(max(float(tf.reduce_sum(tf.abs(_fth_z)**2).numpy()), 1e-12) * _D_HW)
    print(f'  TX gain at zenith  (theta≈0°) : {_g_zenith:.2f} dBi  (expected deep null)')
except Exception as _e:
    print(f'  Pattern evaluation skipped (TensorFlow not available in this env): {_e}')

# ── Optional: build arrays if sionna.rt is importable ─────────────────────
print()
try:
    _tx_arr, _tx_mode = _make_antenna_array(_tx_pattern_915, fallback='hw_dipole')
    _rx_arr, _rx_mode = _make_antenna_array(_rx_pattern_915, fallback='iso')
    print(f'  PlanarArray TX : [{_tx_mode}]')
    print(f'  PlanarArray RX : [{_rx_mode}]')
    print('  (Arrays ready — assign to scene.tx_array / scene.rx_array in sim notebook)')
    # Store for downstream cells
    _tx_array_915 = _tx_arr
    _rx_array_915 = _rx_arr
except Exception as _e:
    print(f'  sionna.rt not available in scene builder env — arrays not built: {_e}')
    print('  Pattern functions _tx_pattern_915 / _rx_pattern_915 are defined and')
    print('  can be copied to / imported by the simulation notebook.')

# ── Export antenna + link budget params to scene_parameters.json ──────────
_params_path = os.path.join(BASE_DIR, 'scene_parameters.json')
try:
    with open(_params_path) as _f:
        _meta = _json.load(_f)
except FileNotFoundError:
    _meta = {}

_meta.update({
    'frequency_hz'         : FREQUENCY_HZ,
    'antenna_pattern'      : ANTENNA_PATTERN,
    'tx_agl_m'             : TX_AGL_M,
    'rx_agl_m'             : RX_AGL_M,
    'tx_conducted_dbm'     : TX_CONDUCTED_DBM,
    'tx_antenna_gain_dbi'  : TX_ANTENNA_GAIN_DBI,
    'tx_eirp_dbm'          : TX_CONDUCTED_DBM + TX_ANTENNA_GAIN_DBI,
    'rx_extra_gain_db'     : RX_EXTRA_GAIN_DB,
    'site_correction_db'   : SITE_CORRECTION_DB,
})

with open(_params_path, 'w') as _f:
    _json.dump(_meta, _f, indent=2)

print()
print(f'scene_parameters.json updated: {_params_path}')
print(json.dumps({k: v for k, v in _meta.items()
                  if k in ('frequency_hz','antenna_pattern','tx_eirp_dbm',
                           'tx_agl_m','rx_agl_m','rx_extra_gain_db')}, indent=2))

---

# Section B — Blender Scene Conversion

These cells are **standalone** and can be run independently of Section A. They convert Blender-exported XML or OSM-built scenes into the target Sionna format.

Set the input/output paths at the top of each cell before running.


## CELL B1 — Blender XML → Sionna 0.19

Reads a Blender-exported or Sionna 2.0 scene XML and writes a Sionna 0.19 compatible version:
- Strips `mat-` prefix from ITU material IDs
- Replaces `diffuse`/`twosided` BSDFs with `conductor` + ITU-R P.2040-2 `eta`/`k`
- Maps colour-named materials to the nearest ITU equivalent

**Inputs to set:** `BLENDER_XML_IN`, `SIONNA19_XML_OUT`


In [ ]:
# ============================================================
# CELL B1 — CONVERT BLENDER / SIONNA 2.0 XML → SIONNA 0.19
# ============================================================

BLENDER_XML_IN  = os.path.join(SCENE_DIR, 'scene_with_full.xml')
SIONNA19_XML_OUT = os.path.join(SCENE_DIR, 'scene_with_full_019.xml')

# ── ITU-R P.2040-2 EM parameters (relative_permittivity, conductivity_S/m) ──
# Sionna 0.19 radio-material parameters: (relative_permittivity, conductivity, scattering, xpd)
# Valid 0.19 ITU built-ins: concrete, brick, plywood, glass, ceiling_board, chipboard,
#                            floorboard, metal, very_dry_ground, medium_dry_ground, wet_ground
# NOT valid: itu_wood → itu_plywood  |  asphalt/vegetation/water → ground built-ins
ITU_EM = {
    'itu_concrete'          : (5.31,  0.092, 0.30, 0.10),
    'itu_brick'             : (3.75,  0.038, 0.30, 0.10),
    'itu_glass'             : (6.27,  0.000, 0.10, 0.05),
    'itu_plywood'           : (1.99,  0.000, 0.30, 0.10),  # itu_wood remapped to itu_plywood
    'itu_metal'             : (1.00,  1.0e7, 0.05, 0.05),
    'itu_wet_ground'        : (30.0,  0.150, 0.10, 0.05),  # ITU-R P.2040-2 Table 3
    'itu_medium_dry_ground' : (15.0,  0.035, 0.10, 0.05),  # ITU-R P.2040-2 Table 3 — roads
    'itu_very_dry_ground'   : ( 3.0,  0.001, 0.10, 0.05),  # ITU-R P.2040-2 Table 3 — vegetation
}

# ── Material name mapping: Blender/Sionna2 ID → Sionna 0.19 ITU name ────────
# Add custom entries here as needed.
MAT_MAP = {
    # Strip mat- from known ITU names
    'mat-itu_brick'     : 'itu_brick',
    'mat-itu_concrete'  : 'itu_concrete',
    'mat-itu_glass'     : 'itu_glass',
    'mat-itu_wood'      : 'itu_plywood',  # itu_wood not in Sionna 0.19 registry → itu_plywood
    'mat-itu_metal'     : 'itu_metal',
    'mat-asphalt'       : 'itu_medium_dry_ground',  # roads
    'mat-itu_asphalt'   : 'itu_medium_dry_ground',
    'mat-itu_wet_ground': 'itu_wet_ground',
    'mat-itu_water'     : 'itu_wet_ground',          # water → wet ground
    'mat-water'         : 'itu_wet_ground',
    'mat-itu_vegetation': 'itu_very_dry_ground',      # vegetation → very dry ground
    'mat-itu_very_dry_ground'   : 'itu_wet_ground',
    'mat-itu_medium_dry_ground' : 'itu_wet_ground',
    # Colour / custom → nearest ITU
    'mat-white'       : 'itu_concrete',
    'mat-grey'        : 'itu_concrete',
    'mat-darkgrey'    : 'itu_concrete',
    'mat-red'         : 'itu_brick',
    'mat-salmon'      : 'itu_brick',
    'mat-tan'         : 'itu_brick',
    'mat-brown'       : 'itu_brick',
    'mat-d5b9a3'      : 'itu_brick',
    'mat-85552e'      : 'itu_brick',
    'mat-ff9e6b'      : 'itu_brick',
    'mat-green'       : 'itu_very_dry_ground',
    'mat-lime'        : 'itu_very_dry_ground',
    'mat-forest'      : 'itu_very_dry_ground',
    'mat-vegetation'  : 'itu_very_dry_ground',
    'mat-areas_pedestrian' : 'itu_concrete',
    'mat-areas_service'    : 'itu_concrete',
    'mat-areas_railways'   : 'itu_concrete',
    'mat-areas_footway'    : 'itu_concrete',
    'mat-areas_road'       : 'itu_concrete',
    'mat-road'             : 'itu_concrete',
    'mat-pavement'         : 'itu_concrete',
}

def resolve_mat(mat_id):
    """Return Sionna 0.19 ITU name for a Blender material ID."""
    if mat_id in MAT_MAP:
        return MAT_MAP[mat_id]
    # Auto-strip mat- prefix if it looks like an ITU name
    stripped = mat_id.replace('mat-', '')
    if stripped in ITU_EM:
        return stripped
    # itu_wood → itu_plywood remapping for direct references
    if stripped == 'itu_wood':
        return 'itu_plywood'
    # Fallback: unknown material → itu_concrete
    print(f'  [WARN] unknown material "{mat_id}" → itu_concrete (add to MAT_MAP to override)')
    return 'itu_concrete'

# ── Parse input XML ───────────────────────────────────────────────────────────
import xml.etree.ElementTree as ET
assert os.path.exists(BLENDER_XML_IN), f'Input not found: {BLENDER_XML_IN}'

tree = ET.parse(BLENDER_XML_IN)
root = tree.getroot()

# Collect all material IDs used in shapes (only write materials actually used)
used_mat_ids = set()
for shape in root.findall('.//shape'):
    ref = shape.find('ref[@name="bsdf"]')
    if ref is not None:
        used_mat_ids.add(ref.get('id'))

print(f'Input   : {BLENDER_XML_IN}')
print(f'Shapes  : {len(root.findall(".//shape"))}')
print(f'Mat IDs : {sorted(used_mat_ids)}')
print()

# Resolve to Sionna 0.19 names
mat_resolved = {mid: resolve_mat(mid) for mid in used_mat_ids}
used_itu = sorted(set(mat_resolved.values()))
print(f'Resolved ITU materials: {used_itu}')
print()
for orig, itu in sorted(mat_resolved.items()):
    if orig != itu:
        print(f'  {orig:<35} → {itu}')

# ── Build output XML lines ────────────────────────────────────────────────────
out = []
out.append('<?xml version="1.0" encoding="utf-8"?>')
out.append('<scene version="2.1.0">')
out.append('')
out.append('  <!-- ── ITU-R P.2040-2 Materials (Sionna 0.19) ────────── -->')

# All materials written as radio-material — required by Sionna 0.19 scene loader
for itu_name in used_itu:
    eps, sigma, sc, xpd = ITU_EM.get(itu_name, (5.31, 0.092, 0.30, 0.10))
    out.append(f'  <bsdf type="radio-material" id="{itu_name}">')
    out.append(f'    <float name="relative_permittivity"  value="{eps}"/>')
    out.append(f'    <float name="conductivity"           value="{sigma}"/>')
    out.append(f'    <float name="scattering_coefficient" value="{sc}"/>')
    out.append(f'    <float name="xpd_coefficient"        value="{xpd}"/>')
    out.append(f'  </bsdf>')
    out.append('')

out.append('  <!-- ── Shapes ─────────────────────────────────────────── -->')

# Build PLY lookup from MESH_DIR + SCENE_DIR (terrain.ply lives in meshes_roads)
_ply_lookup = {}
for _search_dir in [MESH_DIR, os.path.join(SCENE_DIR, 'meshes'), os.path.join(SCENE_DIR, 'meshes_roads'), SCENE_DIR]:
    if not os.path.isdir(_search_dir):
        continue
    for _r, _d, _fs in os.walk(_search_dir):
        for _f in _fs:
            if _f.endswith('.ply'):
                stem = _f[:-4]
                if stem not in _ply_lookup:   # MESH_DIR takes priority
                    _ply_lookup[stem] = os.path.join(_r, _f)
print(f"PLY files found in MESH_DIR: {sorted(_ply_lookup.keys())}")

for shape in root.findall('.//shape'):
    stype = shape.get('type', 'ply')
    sid   = shape.get('id', '')
    ref   = shape.find('ref[@name="bsdf"]')
    if ref is None:
        continue

    raw_fn = shape.findtext('string[@name="filename"]') or ''

    # Resolve absolute PLY path — four strategies
    filename = ''
    # S1: already absolute and exists
    if raw_fn and os.path.isabs(raw_fn) and os.path.isfile(raw_fn):
        filename = raw_fn
    # S2: relative to XML directory
    if not filename and raw_fn:
        cand = os.path.normpath(os.path.join(_xml_dir, raw_fn))
        if os.path.isfile(cand):
            filename = cand
    # S3: basename from raw_fn in MESH_DIR lookup
    if not filename and raw_fn:
        stem = os.path.basename(raw_fn).replace('.ply', '')
        if stem in _ply_lookup:
            filename = _ply_lookup[stem]
    # S4: derive stem from shape id (id="mesh-bld_itu_brick" -> stem="bld_itu_brick")
    # Special case: "mesh-ground" is the terrain, mapped to terrain.ply
    if not filename and sid.startswith('mesh-'):
        stem = sid[len('mesh-'):]
        if stem == 'ground':
            stem = 'terrain'
        if stem in _ply_lookup:
            filename = _ply_lookup[stem]
            print(f"  [S4] {sid} -> {filename}")
    # S5: terrain special case — search known locations for terrain.ply
    if not filename and sid == 'mesh-ground':
        _merged = globals().get('MERGED_DIR', os.path.join(SCENE_DIR, 'meshes_roads'))
        for _tdir in [MESH_DIR, _merged, SCENE_DIR, os.path.join(SCENE_DIR, 'meshes')]:
            _tcand = os.path.join(_tdir, 'terrain.ply')
            if os.path.isfile(_tcand):
                filename = _tcand
                print(f"  [S5] terrain.ply -> {filename}")
                break
    if not filename:
        print(f"  [ERR] Cannot resolve PLY for shape {sid!r} (raw_fn={raw_fn!r}) — shape skipped")
        continue

    orig_mat = ref.get('id')
    itu_mat  = mat_resolved.get(orig_mat, 'itu_concrete')

    out.append(f'  <shape type="{stype}" id="{sid}">')
    out.append(f'    <string name="filename" value="{filename}"/>')
    out.append(f'    <ref id="{itu_mat}" name="bsdf"/>')
    out.append(f'    <boolean name="face_normals" value="true"/>')
    out.append(f'  </shape>')

out.append('')
out.append('</scene>')

with open(SIONNA19_XML_OUT, 'w') as f:
    f.write('\n'.join(out))

print(f'\nWrote: {SIONNA19_XML_OUT}')
print(f'  Shapes    : {len(root.findall(".//shape"))}')
print(f'  Materials : {len(used_itu)} ITU materials')

# ── Validate: re-read the written 019 XML and check all PLY paths ──────────
import xml.etree.ElementTree as _ET2
_tree2 = _ET2.parse(SIONNA19_XML_OUT)
_errors = []
print("PLY path validation:")
for _s in _tree2.getroot().findall(".//shape"):
    _elem = _s.find('string[@name="filename"]')
    _fn = (_elem.attrib.get("value", "") if _elem is not None else "")
    _ok  = os.path.isfile(_fn)
    _abs = os.path.isabs(_fn)
    _icon = "OK  " if _ok else ("REL " if not _abs else "MISS")
    print(f"  [{_icon}] {_fn}")
    if not _ok:
        _errors.append(_fn)
if _errors:
    print(f"FIX NEEDED: {len(_errors)} PLY path(s) invalid.")
    print(f"  xml_dir      : {os.path.dirname(os.path.abspath(BLENDER_XML_IN))}")
    print(f"  BLENDER_XML_IN: {BLENDER_XML_IN}")
else:
    print("All PLY paths valid — scene ready to load.")


## CELL B2 — Blender XML → Sionna 2.0

Copies the original Blender XML and makes the minimum changes needed for Sionna 2.0:
- Maps colour/custom material IDs to nearest `mat-itu_*` equivalent
- Adds missing `mat-itu_*` BSDF definitions (twosided/diffuse)
- Does **not** modify the original Blender file

**Inputs to set:** `BLENDER_XML_IN`, `SIONNA2_XML_OUT`


In [ ]:
# ============================================================
# CELL MERGE — MERGE PLYs BY MATERIAL
# Run this BEFORE Cell B2 if you have 100K+ individual PLY files
# ============================================================
import os, glob, logging
from collections import defaultdict
from pathlib import Path

try:
    import trimesh
except ImportError:
    os.system('pip install trimesh -q')
    import trimesh

# Suppress all trimesh/logging output
logging.getLogger('trimesh').setLevel(logging.ERROR)

MESHES_DIR = MESH_DIR  # auto-derived from SCENARIO_NAME in CELL 0

def get_material(filename):
    name = Path(filename).stem
    return name[4:] if name.startswith('bld_') else name

ply_files = glob.glob(os.path.join(MESHES_DIR, '*.ply'))
print(f'Found {len(ply_files):,} PLY files')

materials = defaultdict(list)
for ply_path in ply_files:
    materials[get_material(ply_path)].append(ply_path)

print(f'Materials: {len(materials)}')
for mat in sorted(materials.keys()):
    print(f'  {mat}: {len(materials[mat]):,} files')
print()

for material, ply_list in sorted(materials.items()):
    if len(ply_list) == 1:
        print(f'✓ {material}: 1 file (skip)')
        continue

    print(f'• {material}: merging {len(ply_list):,} files...', end=' ', flush=True)

    meshes = []
    for ply_path in ply_list:
        try:
            meshes.append(trimesh.load(ply_path, process=False))
        except Exception:
            continue

    if not meshes:
        print('FAILED'); continue

    combined = trimesh.util.concatenate(meshes)
    output_path = os.path.join(MESHES_DIR, f'bld_{material}.ply')
    combined.export(output_path)

    for p in ply_list:
        os.remove(p)

    size_mb = os.path.getsize(output_path) / 1024**2
    print(f'✓ {combined.vertices.shape[0]:,} verts, {size_mb:.1f} MB')

merged = glob.glob(os.path.join(MESHES_DIR, '*.ply'))
print(f'\n✓ Done — {len(merged)} merged files:')
for f in sorted(merged):
    print(f'  {Path(f).name}  {os.path.getsize(f)/1024**2:.1f} MB')


In [ ]:
# ============================================================
# CELL B2 — BUILD scene_sionna2.xml FROM meshes_merged/
# ============================================================
# Reads existing meshes_merged/*.ply (Blender-merged files +
# terrain.ply from CELL 3). Writes a clean Sionna 2.0 XML.
# Does NOT parse untitled.xml or touch meshes/.
# ============================================================
import os
from lxml import etree as _lxe

_merged_dir = globals().get('MERGED_DIR', os.path.join(SCENE_DIR, 'meshes_merged'))
_out_xml    = os.path.join(SCENE_DIR, 'scene_sionna2.xml')

assert os.path.isdir(_merged_dir), f'meshes_merged/ not found: {_merged_dir}'

_ply_files = sorted(f for f in os.listdir(_merged_dir) if f.endswith('.ply'))
print(f'meshes_merged/: {len(_ply_files)} PLY files')

# ── ITU visual colours ───────────────────────────────────────────────────────
_COLOURS = {
    'mat-itu_concrete'        : '0.539 0.539 0.539',
    'mat-itu_brick'           : '1.000 0.498 0.055',
    'mat-itu_glass'           : '0.596 0.875 0.541',
    'mat-itu_wood'            : '0.043 0.580 0.184',
    'mat-itu_metal'           : '0.220 0.220 0.254',
    'mat-itu_wet_ground'      : '0.910 0.569 0.055',
    'mat-itu_very_dry_ground' : '0.498 0.498 0.498',
    'mat-itu_medium_dry_ground': '0.780 0.780 0.780',
}

# ── Blender material name → mat-itu_* ───────────────────────────────────────
_MAT_MAP = {
    'mat-itu_brick'            : 'mat-itu_brick',
    'mat-itu_metal'            : 'mat-itu_metal',
    'mat-itu_concrete'         : 'mat-itu_concrete',
    'mat-itu_glass'            : 'mat-itu_glass',
    'mat-itu_wood'             : 'mat-itu_wood',
    'mat-itu_wet_ground'       : 'mat-itu_wet_ground',
    'mat-itu_very_dry_ground'  : 'mat-itu_very_dry_ground',
    'mat-itu_medium_dry_ground': 'mat-itu_medium_dry_ground',
    'terrain'                  : f'mat-{globals().get("TERRAIN_MATERIAL","itu_wet_ground")}',
}

def _resolve(stem):
    if stem in _MAT_MAP:
        return _MAT_MAP[stem]
    # strip common prefixes and try
    core = stem.replace('mat-itu_','').replace('mat-','')
    _valid = {'concrete','brick','glass','wood','metal',
              'wet_ground','very_dry_ground','medium_dry_ground'}
    if core in _valid:
        return f'mat-itu_{core}'
    return 'mat-itu_concrete'   # fallback

# ── Build XML ────────────────────────────────────────────────────────────────
_root = _lxe.Element('scene', version='2.1.0')

# Integrator
_integ = _lxe.SubElement(_root, 'integrator', type='path', id='elm__0', name='elm__0')
_lxe.SubElement(_integ, 'integer', name='max_depth', value='12')

# Collect needed materials
_used_mats = set()
for _f in _ply_files:
    _used_mats.add(_resolve(_f.replace('.ply','')))

# Write twosided BSDFs
for _mid in sorted(_used_mats):
    _colour = _COLOURS.get(_mid, '0.5 0.5 0.5')
    _b = _lxe.SubElement(_root, 'bsdf', type='twosided', id=_mid, name=_mid)
    _i = _lxe.SubElement(_b, 'bsdf', type='diffuse')
    _lxe.SubElement(_i, 'rgb', value=_colour, name='reflectance')

# Write shapes
for _f in _ply_files:
    _stem  = _f.replace('.ply','')
    _bsdid = _resolve(_stem)
    _sh = _lxe.SubElement(_root, 'shape', type='ply', id=f'mesh-{_stem}')
    _lxe.SubElement(_sh, 'string', name='filename', value=f'meshes_merged/{_f}')
    _lxe.SubElement(_sh, 'ref', id=_bsdid, name='bsdf')
    _lxe.SubElement(_sh, 'boolean', name='face_normals', value='true')

_tree = _lxe.ElementTree(_root)
_tree.write(_out_xml, pretty_print=True, xml_declaration=True, encoding='utf-8')

print(f'\nWrote: {_out_xml}')
print(f'  Shapes   : {len(_ply_files)}')
print(f'  Materials: {sorted(_used_mats)}')
has_terrain = any('terrain' in f for f in _ply_files)
print(f'  Terrain  : {"yes" if has_terrain else "NO — run CELL 3 first"}')


## CELL B3 — OSM Scene → Sionna 2.0 Export

Generates a Sionna 2.0 compatible scene XML from the OSM-built scene (produced by Section A). Uses `mat-` prefixed material IDs, `twosided/diffuse` BSDFs for rendering, and `face_normals=true` on all shapes.

**Requires:** Section A cells 0–5 to have been run first (SCENE_DIR, MESH_DIR must be set).


In [ ]:
# ============================================================
# CELL B3 — WRITE SCENE_SIONNA2.XML  (Sionna 2.0 / Mitsuba 2.1.0)
# ============================================================
# Generates a Sionna 2.0 compatible scene XML alongside the
# existing Sionna 0.19 scene.xml.
# Key differences vs 0.19:
#   - Material IDs prefixed with "mat-"  (mat-itu_brick etc.)
#   - BSDF type = twosided/diffuse + rgb  (visual colour for rendering)
#   - Shapes have id + face_normals=true
#   - Integrator, emitter, sensor blocks included
# ============================================================

# ── Reconstruct mat_plys and config if running standalone ────────────────────
import os
if 'TERRAIN_MATERIAL' not in dir():
    TERRAIN_MATERIAL = 'itu_wet_ground'      # sub-1GHz handled by monkeypatch in sim notebook
# Always reconstruct mat_plys from disk to ensure correct subdirectory paths
mat_plys = {}
_prefix_role = {'bld_': 'buildings', 'road_': 'roads',
                'water_': 'water',   'veg_': 'vegetation',
                'trees_': 'vegetation', 'rail_': 'railways',
                'barriers_': 'barriers', 'infra_': 'infrastructure'}
# Known ITU material names — sorted longest-first so 'itu_wet_ground' matches before 'itu_wet'
_KNOWN_ITU = sorted([
    'itu_brick', 'itu_concrete', 'itu_glass', 'itu_wood', 'itu_metal',
    'itu_wet_ground', 'itu_medium_dry_ground', 'itu_very_dry_ground',
    'itu_asphalt', 'itu_vegetation', 'itu_water',
], key=len, reverse=True)

def _ply_to_mat(fname, pfx):
    raw = fname[len(pfx):-4]          # e.g. 'itu_metal_pylons'
    for km in _KNOWN_ITU:
        if raw == km or raw.startswith(km + '_'):
            return km                  # e.g. 'itu_metal'
    return raw                         # fallback: use raw (e.g. 'itu_brick' already clean)

for _fname in os.listdir(MESH_DIR):
    if not _fname.endswith('.ply') or _fname == 'terrain.ply':
        continue
    for _pfx, _role in _prefix_role.items():
        if _fname.startswith(_pfx):
            _mat = _ply_to_mat(_fname, _pfx)
            mat_plys.setdefault(_mat, []).append((os.path.basename(MESH_DIR) + '/' + _fname, _role))
            break
print(f'mat_plys from disk: {list(mat_plys.keys())}')
if 'center_lat' not in dir():
    import json as _json
    _p = os.path.join(BASE_DIR, 'scene_parameters.json')
    if os.path.exists(_p):
        _meta = _json.load(open(_p))
        center_lat = _meta['scene_center_lat']
        center_lon = _meta['scene_center_lon']
        UTM_EPSG   = _meta.get('utm_epsg', 32630)
    else:
        center_lat = center_lon = 0.0; UTM_EPSG = 32630

# ── ITU visual colours (for Mitsuba rendering — not EM properties) ──────────
ITU_COLOURS = {
    'itu_concrete'          : '0.539 0.539 0.539',
    'itu_brick'             : '1.000 0.498 0.055',
    'itu_glass'             : '0.596 0.875 0.541',
    'itu_wood'              : '0.514 0.376 0.220',  # warm wood brown
    'itu_plywood'           : '0.514 0.376 0.220',
    'itu_metal'             : '0.220 0.220 0.254',
    'itu_wet_ground'        : '0.910 0.569 0.055',  # wet soil brown
    'itu_very_dry_ground'   : '0.498 0.498 0.498',  # dry pale grey
    'itu_medium_dry_ground' : '0.780 0.780 0.780',  # medium grey
    'asphalt'               : '0.200 0.200 0.200',  # dark road grey (mapped from itu_asphalt)
    'vegetation'            : '0.180 0.450 0.180',  # green
    'water'                 : '0.180 0.330 0.640',  # blue
}

used_mats2 = set(mat_plys.keys()) | {TERRAIN_MATERIAL}

# Sionna 2.0 ITU material types do NOT include asphalt/vegetation/water.
# Give those non-ITU ids (no "itu_" prefix) so Sionna loads them as plain
# RadioMaterials (default eps=1, sigma=0) instead of failing ITU validation.
# The sim notebook (CELL 4A/4B) then sets their eps/sigma/scattering by name.
_NON_ITU_MAP = {'itu_asphalt': 'asphalt',
                'itu_vegetation': 'vegetation',
                'itu_water': 'water'}
def _s2_matid(m):
    return _NON_ITU_MAP.get(m, m)

# Explicit radio-material properties for the non-ITU surfaces (Sionna has no
# ITU type for these). Written as <bsdf type="radio-material"> so Sionna loads
# them as real radio materials. The sim notebook CELL 4A/4B refine these.
# ── ITU-R P.2040-2 (2023) Table 3 — frequency-portable non-ITU materials ──
# These materials are not in Sionna 2.0 built-in registry so written as
# explicit radio-material BSDFs. Values computed from P.2040-2 formula.
_ITU_P2040_NON_ITU = {
    #                    a      b       c       d      s    rgb
    'itu_asphalt'   : (2.56, 0.000, 0.0050, 0.0000, 0.30, '0.200 0.200 0.200'),
    'itu_vegetation': (1.50, 0.000, 0.0020, 0.5000, 0.75, '0.180 0.450 0.180'),  # ITU-R P.833
    'itu_water'     : (80.0, 0.000, 0.0100, 0.0000, 0.02, '0.180 0.330 0.640'),  # ITU-R P.527
}
_f_ghz_b3 = float(globals().get('FREQUENCY_HZ', 915.95e6)) / 1e9
_NON_ITU_RM = {}
for _mn, (_a, _b, _c, _d, _s, _rgb) in _ITU_P2040_NON_ITU.items():
    _NON_ITU_RM[_mn] = dict(
        er    = round(_a * (_f_ghz_b3 ** _b), 6),
        sigma = round(_c * (_f_ghz_b3 ** _d), 8),
        scat  = _s,
        rgb   = _rgb,
    )
print(f'Non-ITU materials computed at {_f_ghz_b3*1e3:.2f} MHz (ITU-R P.2040-2):')
for _mn, _v in _NON_ITU_RM.items():
    print(f'  {_mn:<18}  er={_v["er"]:.4f}  sigma={_v["sigma"]:.6f}  S={_v["scat"]:.2f}')

lines2 = []
lines2.append('<?xml version="1.0" ?>')
lines2.append('<scene version="2.1.0">')
lines2.append('')

# ── Metadata defaults ────────────────────────────────────────────────────────
lines2.append('  <!-- ── Scene metadata ──────────────────────────────────── -->')
lines2.append(f'  <default name="scenegen_version"     value="1.0.0"/>')
lines2.append(f'  <default name="scenegen_min_lat"     value="{SCENE_SOUTH}"/>')
lines2.append(f'  <default name="scenegen_max_lat"     value="{SCENE_NORTH}"/>')
lines2.append(f'  <default name="scenegen_min_lon"     value="{SCENE_WEST}"/>')
lines2.append(f'  <default name="scenegen_max_lon"     value="{SCENE_EAST}"/>')
lines2.append(f'  <default name="scenegen_center_lat"  value="{center_lat}"/>')
lines2.append(f'  <default name="scenegen_center_lon"  value="{center_lon}"/>')
lines2.append(f'  <default name="scenegen_UTM_zone"    value="EPSG:{UTM_EPSG}"/>')
lines2.append(f'  <default name="scenegen_ground_material"  value="mat-{TERRAIN_MATERIAL}"/>')
lines2.append('')

# ── Integrator ───────────────────────────────────────────────────────────────
lines2.append('  <!-- ── Integrator ──────────────────────────────────────── -->')
lines2.append('  <integrator type="path">')
lines2.append('    <integer name="max_depth" value="12"/>')
lines2.append('  </integrator>')
lines2.append('')

# ── Materials — twosided diffuse (visual) ────────────────────────────────────
lines2.append('  <!-- ── ITU-R Materials (visual BSDF for rendering) ─────── -->')
for mat_name in sorted(used_mats2):
    if mat_name in _NON_ITU_RM:
        _rm = _NON_ITU_RM[mat_name]
        lines2.append(f'  <bsdf type="radio-material" id="mat-{_s2_matid(mat_name)}">')
        lines2.append(f'    <float name="relative_permittivity" value="{_rm["er"]}"/>')
        lines2.append(f'    <float name="conductivity"          value="{_rm["sigma"]}"/>')
        lines2.append(f'    <float name="scattering_coefficient" value="{_rm["scat"]}"/>')
        lines2.append(f'  </bsdf>')
        lines2.append('')
    else:
        colour = ITU_COLOURS.get(mat_name, '0.5 0.5 0.5')
        lines2.append(f'  <bsdf type="twosided" id="mat-{_s2_matid(mat_name)}">')
        lines2.append(f'    <bsdf type="diffuse">')
        lines2.append(f'      <rgb value="{colour}" name="reflectance"/>')
        lines2.append(f'    </bsdf>')
        lines2.append(f'  </bsdf>')
        lines2.append('')

# ── Emitter ──────────────────────────────────────────────────────────────────
lines2.append('  <!-- ── Environment ─────────────────────────────────────── -->')
lines2.append('  <emitter type="constant" id="World">')
lines2.append('    <rgb value="1.0 1.0 1.0" name="radiance"/>')
lines2.append('  </emitter>')
lines2.append('')

# ── Camera ───────────────────────────────────────────────────────────────────
lines2.append('  <!-- ── Camera (top-down overview) ─────────────────────── -->')
lines2.append('  <sensor type="perspective" id="Camera">')
lines2.append('    <string name="fov_axis" value="x"/>')
lines2.append('    <float  name="fov"      value="42.855"/>')
lines2.append('    <float  name="near_clip" value="0.1"/>')
lines2.append('    <float  name="far_clip"  value="10000.0"/>')
lines2.append('    <transform name="to_world">')
lines2.append('      <rotate z="1" angle="-90"/>')
lines2.append('      <translate value="0 0 500"/>')
lines2.append('    </transform>')
lines2.append('    <sampler type="independent">')
lines2.append('      <integer name="sample_count" value="4096"/>')
lines2.append('    </sampler>')
lines2.append('    <film type="hdrfilm">')
lines2.append('      <integer name="width"  value="1024"/>')
lines2.append('      <integer name="height" value="1024"/>')
lines2.append('    </film>')
lines2.append('  </sensor>')
lines2.append('')

# ── Terrain shape ─────────────────────────────────────────────────────────────
lines2.append('  <!-- ── Terrain ─────────────────────────────────────────── -->')
lines2.append('  <shape type="ply" id="mesh-ground">')
lines2.append(f'    <string name="filename" value="meshes/terrain.ply"/>')
lines2.append(f'    <ref id="mat-{TERRAIN_MATERIAL}" name="bsdf"/>')
lines2.append('    <boolean name="face_normals" value="true"/>')
lines2.append('  </shape>')
lines2.append('')

# ── Building shapes ───────────────────────────────────────────────────────────
lines2.append('  <!-- ── Buildings (one merged PLY per material) ──────────── -->')
for mat_name, ply_list in sorted(mat_plys.items()):
    for ply_path, role in ply_list:
        mesh_id = 'mesh-' + ply_path.split('/')[-1].replace('.ply', '')
        lines2.append(f'  <shape type="ply" id="{mesh_id}">')
        lines2.append(f'    <string name="filename" value="{ply_path}"/>')
        lines2.append(f'    <ref id="mat-{_s2_matid(mat_name)}" name="bsdf"/>')
        lines2.append('    <boolean name="face_normals" value="true"/>')
        lines2.append('  </shape>')

lines2.append('')
lines2.append('</scene>')

scene2_xml = os.path.join(SCENE_DIR, 'scene_with_full.xml')
with open(scene2_xml, 'w') as f:
    f.write('\n'.join(lines2))

print(f'Wrote: {scene2_xml}')
print(f'  Materials : {len(used_mats2)}')
total2 = 1 + sum(len(v) for v in mat_plys.values())
print(f'  Shapes    : {total2}  (1 terrain + {total2-1} feature PLYs: buildings + roads)')
print(f'  Format    : Sionna 2.0 / Mitsuba 2.1.0  (twosided diffuse + face_normals)')


In [ ]:
# ============================================================
# CELL PREVIEW — RENDER THE BUILT SCENE  (same as 900 flat nb)
# ============================================================
# Loads the freshly written scene_sionna2.xml and shows the
# interactive preview so you can visually confirm terrain,
# buildings, roads, water and the new standing vegetation canopy.
# Run after CELL B3.
# ============================================================
%matplotlib inline
no_preview = False   # set True to skip the widget

from sionna.rt import load_scene

_preview_xml = SIONNA19_XML_OUT if "SIONNA19_XML_OUT" in dir() else os.path.join(SCENE_DIR, "scene_with_full_019.xml")
print(f"Loading scene: {_preview_xml}")
scene = load_scene(_preview_xml)
print(f"Objects: {len(scene.objects)}  |  Materials: {len(scene.radio_materials)}")

if not no_preview:
    scene.preview()
